In [1]:
print('start')

start


In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10.0, -3.4)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10.0, -3.4)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [4]:
#Monomeric models
def clean_feature_names(df):
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)
    df.columns = [clean_name(col) for col in df.columns]
    return df

In [5]:
#Monomer composition
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp_caco2.csv')
df_mc_train = clean_feature_names(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_Caco2.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(1007, 385)
(1007,)
(252, 385)
(252,)
0.6947370760913676
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001880 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 338
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 40
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2314,0.3504,0.4810,0.6390,0.8032,0.7958,0.1902,0.3235,0.4361,0.6947,0.8338,0.8091
LGBMRegressor,0.2719,0.3999,0.5214,0.5758,0.7590,0.7382,0.2190,0.3565,0.4679,0.6486,0.8057,0.7915
XGBRegressor,0.2337,0.3593,0.4834,0.6353,0.8003,0.7911,0.1807,0.3204,0.4251,0.7100,0.8426,0.8256
DecisionTreeRegressor,0.3495,0.4331,0.5912,0.4546,0.7224,0.7043,0.2174,0.3535,0.4662,0.6511,0.8089,0.7824
RandomForestRegressor,0.2269,0.3571,0.4764,0.6459,0.8042,0.7955,0.1824,0.3167,0.4270,0.7073,0.8422,0.8275
GradientBoostingRegressor,0.2416,0.3793,0.4916,0.6230,0.7940,0.7745,0.2202,0.3614,0.4692,0.6466,0.8137,0.7798
AdaBoostRegressor,0.3893,0.5300,0.6239,0.3926,0.6445,0.6209,0.3745,0.5223,0.6119,0.3990,0.6635,0.6110
SVR,0.2792,0.3998,0.5284,0.5644,0.7521,0.7359,0.2696,0.3831,0.5192,0.5674,0.7542,0.7422
LinearRegression,0.3454,0.4310,0.5877,0.4611,0.7036,0.7233,0.3098,0.4103,0.5566,0.5028,0.7361,0.7405
KNeighborsRegressor,0.4156,0.4655,0.6447,0.3515,0.6240,0.6130,0.3688,0.4465,0.6073,0.4082,0.6560,0.6365


In [6]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.624300000000002, -7.6168, -6.9989999999999...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6355, -5.970749387369999, -6.5853, -5.843...","[-6.9707589015040075, -6.302000536705999, -6.4...","[0.19533808581587186, 0.1826368835490829, 0.12..."
1,LGBMRegressor,"[-6.5576691810467125, -6.76750588414383, -6.33...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.417646829627649, -6.203162551332383, -6.1...","[-6.630318931129807, -6.247585241916136, -6.16...","[0.17052303985983328, 0.0850752574709542, 0.08..."
2,XGBRegressor,"[-6.5595627, -7.070038, -7.0501184, -6.88162, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5710807, -6.1793747, -6.23096, -5.75674, ...","[-6.931924, -6.270083, -6.2176056, -5.7719316,...","[0.23225579, 0.07379878, 0.19781186, 0.0289832..."
3,DecisionTreeRegressor,"[-5.96, -7.0, -7.0, -6.89, -6.11, -7.15, -5.96...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -5.72, -6.77, -5.849999999999999, -7....","[-6.6370000000000005, -5.756, -6.58, -5.816, -...","[0.7069059343363868, 0.044542114902640345, 0.4..."
4,RandomForestRegressor,"[-6.405360314716668, -7.282599999999999, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.384400000000002, -6.024083333333338, -6.4...","[-6.646382922184003, -6.26722166666667, -6.429...","[0.2261066560905955, 0.1648215865042477, 0.072..."
5,GradientBoostingRegressor,"[-6.4003129359931155, -6.998434496154232, -6.7...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.363410127824608, -6.400930698865167, -5.9...","[-6.671047975187451, -6.398322447784479, -6.08...","[0.20237106919716685, 0.05323936514275996, 0.1..."
6,AdaBoostRegressor,"[-6.408263403263382, -6.390385778152708, -6.40...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.408263403263382, -6.0055429258521436, -6....","[-6.404022174410424, -6.081649953482442, -6.27...","[0.03897779182734475, 0.07546928036755012, 0.0..."
7,SVR,"[-6.472172731011601, -7.124544175778197, -7.08...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.295596962054628, -5.857971433203633, -5.5...","[-6.25371393811278, -5.852913707098395, -5.561...","[0.03490851424273175, 0.020133874886575098, 0...."
8,LinearRegression,"[-6.131967782103228, -7.190073043067374, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.918578221512914, -5.951252652491644, -5.3...","[-5.909099043104533, -6.09161491453398, -5.304...","[0.04340671448031737, 0.1346054310693922, 0.05..."
9,KNeighborsRegressor,"[-5.863333333333333, -7.296666666666667, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.72, -6.353333333333334, -5.85333333333333...","[-6.3839999999999995, -6.369333333333334, -5.9...","[0.24078021328819993, 0.12070901650940005, 0.1..."


In [7]:
result_df.to_csv('results/Monomeric/Monomer_comp_results_Caco2.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_prediction_data_Caco2.csv')

In [8]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [9]:
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp_caco2.csv')
df_mc_train = clean_feature_names(df_mc_train)
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_Caco2.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(1007, 213)
(1007,)
(252, 213)
(252,)
0.6966294730309766
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004326 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 338
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 40
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


-0.2527762471370312


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2350,0.3527,0.4847,0.6334,0.7999,0.7941,0.1890,0.3219,0.4348,0.6966,0.8350,0.8120
LGBMRegressor,0.2719,0.3999,0.5214,0.5758,0.7590,0.7382,0.2190,0.3565,0.4679,0.6486,0.8057,0.7915
XGBRegressor,0.2337,0.3593,0.4834,0.6353,0.8003,0.7911,0.1807,0.3204,0.4251,0.7100,0.8426,0.8256
DecisionTreeRegressor,0.3493,0.4335,0.5911,0.4549,0.7245,0.7046,0.2236,0.3505,0.4729,0.6411,0.8035,0.7795
RandomForestRegressor,0.2264,0.3565,0.4758,0.6468,0.8046,0.7957,0.1826,0.3175,0.4274,0.7069,0.8420,0.8263
GradientBoostingRegressor,0.2424,0.3804,0.4924,0.6217,0.7931,0.7731,0.2218,0.3627,0.4709,0.6441,0.8120,0.7772
AdaBoostRegressor,0.3928,0.5326,0.6268,0.3870,0.6392,0.6328,0.3854,0.5308,0.6208,0.3816,0.6422,0.6026
SVR,0.2792,0.3998,0.5284,0.5644,0.7521,0.7359,0.2696,0.3831,0.5192,0.5673,0.7542,0.7422
LinearRegression,0.3454,0.4310,0.5877,0.4611,0.7036,0.7233,0.3098,0.4103,0.5566,0.5028,0.7361,0.7405
KNeighborsRegressor,0.4156,0.4655,0.6447,0.3515,0.6240,0.6130,0.3688,0.4465,0.6073,0.4082,0.6560,0.6365


In [10]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.7435000000000045, -7.423499999999998, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.634200000000003, -5.9956499999999995, -6....","[-6.982151863912007, -6.276903596552002, -6.42...","[0.19910331700662334, 0.19593850182387082, 0.1..."
1,LGBMRegressor,"[-6.5576691810467125, -6.76750588414383, -6.33...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.417646829627649, -6.203162551332383, -6.1...","[-6.630318931129807, -6.247585241916136, -6.16...","[0.17052303985983328, 0.0850752574709542, 0.08..."
2,XGBRegressor,"[-6.5595627, -7.070038, -7.0501184, -6.88162, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5710807, -6.1793747, -6.23096, -5.75674, ...","[-6.931924, -6.270083, -6.2176056, -5.7719316,...","[0.23225579, 0.07379878, 0.19781186, 0.0289832..."
3,DecisionTreeRegressor,"[-5.96, -8.0, -6.89, -7.0, -6.24, -7.15, -5.96...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -5.72, -6.21, -5.85, -7.85, -7.0, -6....","[-6.634, -5.776999999999999, -6.346, -5.872, -...","[0.6884359084184964, 0.04728636167014757, 0.38..."
4,RandomForestRegressor,"[-6.406175406586667, -7.230949999999996, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.373544129000001, -5.995808333333339, -6.4...","[-6.637578414650669, -6.253813333333336, -6.42...","[0.2433108064915238, 0.1657281928506874, 0.070..."
5,GradientBoostingRegressor,"[-6.4003129359931155, -6.924245626677923, -6.7...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.363410127824608, -6.400930698865167, -5.9...","[-6.671047982512519, -6.398322451503861, -6.04...","[0.2023710758327277, 0.05323936210180362, 0.11..."
6,AdaBoostRegressor,"[-6.443200029054911, -6.39131123919308, -6.443...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.443200029054911, -6.020344827586207, -6.3...","[-6.395588105874461, -6.006763491439747, -6.31...","[0.06294096907050022, 0.030910788254346157, 0...."
7,SVR,"[-6.47217274006265, -7.124544335187731, -7.085...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.295596825434822, -5.857971461215588, -5.5...","[-6.2537072786137955, -5.852938483019007, -5.5...","[0.0349109082866144, 0.02008901863164398, 0.02..."
8,LinearRegression,"[-6.131967782103228, -7.190073043067372, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.918578221512915, -5.951252652491643, -5.3...","[-5.909099043104534, -6.091614914533975, -5.30...","[0.04340671448031744, 0.13460543106938813, 0.0..."
9,KNeighborsRegressor,"[-5.863333333333333, -7.296666666666667, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.72, -6.353333333333334, -5.85333333333333...","[-6.3839999999999995, -6.369333333333334, -5.9...","[0.24078021328819993, 0.12070901650940005, 0.1..."


In [11]:
const_col

['Ala_tBu_',
 'Me_Ala_indol_2_yl_',
 'Ala_5_Tet_',
 'Me_dAbu',
 '2Abz',
 'HOCOCH2_Bal',
 'Cys_EtO2H__NH2',
 'Cha',
 'dCha',
 'Asp_OMe_',
 'Asp_Ph_2_NH2__',
 'dAsp_pyrrol_1_yl_',
 'E',
 'Glu_NH2',
 'Glu_3R_Me_',
 'Glu_OMe_',
 'dGlu_OMe_',
 'Phe_4_F_',
 'dPhe_4_F_',
 'Phe_4_NO2_',
 'dPhe_3_4_diF_',
 'Me_Phe_a_b_dehydro_',
 'Bn_4_OH__Gly',
 'Et_Gly',
 'HOCOCH2_Gly_ol',
 'MeOEt_Gly',
 'NH2Bu_Gly',
 'PhPr_Gly',
 'cHexCH2_Gly',
 '2_pyridylmethyl_Gly',
 'd_N__O_Gly_allyl_',
 'GABA',
 'bHph',
 'dHyp',
 '_N__O_xiIle',
 'd_N__O_aIle',
 'Me_dK',
 'Lys_Cbz_',
 'Me_Lys_Me_',
 'Lys_Tfa_',
 'aMeLeu',
 'dLeu_3R_OH_',
 '_N__O_Leu',
 'd_N__O_Leu',
 'M',
 'meM',
 'Met_O2_',
 'meN',
 'dAsn_Me2_',
 'Nal',
 '1_Nal',
 'd1_Nal',
 'Me_dNle',
 'dNva',
 'Me_dNva',
 'Orn',
 'meQ',
 'dGln_Me2_',
 'R',
 'Arg_Me_Me_',
 'Ser_Ac_',
 'dSer_Me_',
 'Sta',
 'Sta_3R_4R_',
 'dT',
 'Tza',
 'Me_dV',
 '_N__O_Val',
 'd_N__O_Val',
 '_N__O_Val_3_OH_',
 'Trp_5_Br_',
 'Trp_6_Br_',
 'Trp_7_Br_',
 'dY',
 'Me_dY',
 'Me_Tyr_Me_',
 'dTy

In [12]:
result_df.to_csv('results/Monomeric/Monomer_comp_constRemoval_results_Caco2.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_constRemoval_prediction_data_Caco2.csv')

In [13]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [14]:
df_train = pd.read_csv('features/Monomeric/Train_mon_comp_caco2.csv')
df_mc_train = clean_feature_names(df_train)
df_mc_train = df_mc_train.drop(['ID','SMILES','Permeability'],axis=1)
df_mc, const_col = remove_low_variance_columns(df_mc_train)
X_train = df_mc
y_train = df_train['Permeability']
print(X_train.shape)
print(y_train.shape)

df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_Caco2.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(1007, 5)
(1007,)
(252, 5)
(252,)
0.4074055550556759
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001366 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 82
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 5
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.064125765030147


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.4380,0.5056,0.6618,0.3166,0.5916,0.5396,0.3693,0.4654,0.6077,0.4074,0.6496,0.5792
LGBMRegressor,0.4127,0.5048,0.6424,0.3560,0.5986,0.5335,0.3751,0.4784,0.6125,0.3980,0.6315,0.5515
XGBRegressor,0.4625,0.5177,0.6801,0.2784,0.5766,0.5333,0.3778,0.4758,0.6147,0.3937,0.6458,0.5839
DecisionTreeRegressor,0.5083,0.5281,0.7129,0.2069,0.5530,0.5160,0.3957,0.4784,0.6290,0.3650,0.6273,0.5607
RandomForestRegressor,0.4150,0.4947,0.6442,0.3525,0.6076,0.5523,0.3703,0.4684,0.6085,0.4057,0.6446,0.5745
GradientBoostingRegressor,0.4080,0.5070,0.6388,0.3633,0.6031,0.5417,0.3884,0.4891,0.6232,0.3767,0.6147,0.5521
AdaBoostRegressor,0.4589,0.5731,0.6774,0.2840,0.5378,0.4661,0.4545,0.5630,0.6742,0.2705,0.5259,0.4424
SVR,0.4481,0.5193,0.6694,0.3008,0.5569,0.4907,0.4381,0.5074,0.6619,0.2970,0.5551,0.4748
LinearRegression,0.5741,0.6326,0.7577,0.1042,0.3230,0.2424,0.5699,0.6275,0.7549,0.0853,0.2951,0.2154
KNeighborsRegressor,0.5624,0.5688,0.7499,0.1224,0.4694,0.3909,0.4204,0.4832,0.6483,0.3254,0.5946,0.5539


In [15]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-7.240000000000006, -6.128750000000005, -6.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.597780000000001, -6.237142857142873, -6.0...","[-6.645734666666667, -6.349500000000007, -5.91...","[0.11777817074389538, 0.16537613281886734, 0.0..."
1,LGBMRegressor,"[-6.475985125632606, -6.087083415265274, -6.17...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.385752795325076, -6.087407342222135, -6.1...","[-6.6068759986607875, -6.1350288145143566, -6....","[0.19374599380670626, 0.0558834294295135, 0.04..."
2,XGBRegressor,"[-7.2385406, -6.130092, -6.1286955, -6.053405,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.703607, -6.135389, -6.514294, -6.183023, ...","[-6.9024696, -6.231168, -6.204558, -6.1036077,...","[0.14186822, 0.10279509, 0.33365387, 0.0446562..."
3,DecisionTreeRegressor,"[-7.24, -6.12875, -6.120000000000001, -6.018, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.03, -6.237142857142857, -5.88, -6.1542857...","[-7.0569999999999995, -6.349499999999999, -5.9...","[0.13847743498490997, 0.16537613281886368, 0.1..."
4,RandomForestRegressor,"[-6.731572380952385, -6.157787071588545, -6.11...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.412997055860805, -6.222832295172152, -6.0...","[-6.586288244505496, -6.346211132388854, -5.98...","[0.16220920646418122, 0.1615669774206806, 0.04..."
5,GradientBoostingRegressor,"[-6.875658982177979, -6.2515871140193395, -5.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.895637202337529, -6.2436328891470145, -6....","[-6.967866959418764, -6.166187901484449, -6.28...","[0.10685875140012335, 0.053127374212715316, 0...."
6,AdaBoostRegressor,"[-6.350972893176028, -6.350972893176028, -6.25...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.350972893176028, -6.32162643553719, -6.32...","[-6.399112214112099, -6.376884081310595, -6.32...","[0.05578906315406391, 0.060918563729190696, 0...."
7,SVR,"[-6.504852213063235, -6.228218180411977, -6.07...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.540016558433213, -6.094436962836127, -6.1...","[-6.679494792600314, -6.033216624737014, -6.04...","[0.12550847246893887, 0.0852537019082089, 0.06..."
8,LinearRegression,"[-6.247849400619675, -6.083497938849115, -6.06...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.45255702822148, -6.147858282675403, -6.09...","[-6.452573612971596, -6.14436513251909, -6.082...","[0.029441311275874432, 0.012907662954100623, 0..."
9,KNeighborsRegressor,"[-6.3549999999999995, -6.283333333333334, -6.3...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.55, -6.133333333333333, -5.95, -5.8233333...","[-6.931333333333333, -6.286, -5.924, -5.807999...","[0.2811057531329528, 0.2622416188683, 0.066043..."


In [16]:
result_df.to_csv('results/Monomeric/Monomer_comp_LVR_results_Caco2.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_LVR_prediction_data_Caco2.csv')

In [17]:
#AA composition
df_aac_train = pd.read_csv('features/Monomeric/Train_aac_Caco2.csv')
X_train = df_aac_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_aac_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_aac_test = pd.read_csv('features/Monomeric/Test_aac_Caco2.csv')
X_test = df_aac_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_aac_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
aac_comp,prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
aac_comp

(1007, 21)
(1007,)
(252, 21)
(252,)
0.6328278585565359
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000544 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 178
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 13
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warn

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.1452102140493421


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2752,0.3941,0.5246,0.5706,0.7619,0.7435,0.2288,0.3523,0.4783,0.6328,0.7973,0.7570
LGBMRegressor,0.2869,0.4159,0.5356,0.5524,0.7435,0.7197,0.2600,0.3932,0.5099,0.5827,0.7634,0.7205
XGBRegressor,0.2847,0.4014,0.5336,0.5557,0.7574,0.7408,0.2487,0.3777,0.4987,0.6008,0.7801,0.7388
DecisionTreeRegressor,0.3455,0.4360,0.5878,0.4609,0.7210,0.7011,0.2547,0.3689,0.5047,0.5913,0.7772,0.7307
RandomForestRegressor,0.2625,0.3887,0.5123,0.5905,0.7702,0.7496,0.2341,0.3616,0.4838,0.6244,0.7905,0.7459
GradientBoostingRegressor,0.2911,0.4248,0.5395,0.5458,0.7395,0.7237,0.2487,0.4021,0.4987,0.6009,0.7797,0.7452
AdaBoostRegressor,0.4126,0.5483,0.6424,0.3562,0.6012,0.5924,0.3823,0.5285,0.6183,0.3864,0.6305,0.6139
SVR,0.3222,0.4423,0.5676,0.4973,0.7060,0.6890,0.2944,0.4227,0.5426,0.5275,0.7270,0.7024
LinearRegression,0.4554,0.5582,0.6748,0.2894,0.5387,0.5091,0.4660,0.5732,0.6826,0.2522,0.5029,0.4412
KNeighborsRegressor,0.3578,0.4537,0.5982,0.4416,0.6831,0.6659,0.3140,0.4216,0.5604,0.4961,0.7125,0.6854


In [18]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-7.240000000000006, -7.445000000000011, -6.91...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.240000000000006, -6.5137, -6.444583333333...","[-7.215000000000005, -6.6255352438628305, -6.7...","[0.01581138830083931, 0.0845998573652395, 0.20..."
1,LGBMRegressor,"[-6.704383489364697, -6.665711482220352, -6.56...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.704383489364697, -6.559869384199295, -6.1...","[-6.868390003198918, -6.728410493756985, -6.12...","[0.1394472935170537, 0.15069537498565147, 0.07..."
2,XGBRegressor,"[-7.2275796, -7.466514, -7.0293694, -5.9889054...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.2275796, -6.655986, -6.369739, -5.841352,...","[-7.207259, -6.722191, -6.586835, -5.7703457, ...","[0.016037496, 0.13797994, 0.3353082, 0.0449222..."
3,DecisionTreeRegressor,"[-7.24, -7.445, -7.0, -7.445, -5.92, -6.095, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -6.13, -6.21, -5.85, -5.92, -7.015000...","[-7.215000000000001, -6.800999999999999, -6.93...","[0.01581138830084184, 0.3426134848484513, 0.79..."
4,RandomForestRegressor,"[-6.509129693268279, -7.400115, -6.84678666666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.509129693268279, -6.565566428571424, -6.7...","[-6.806837188006537, -6.679561829748522, -6.77...","[0.2113426636629116, 0.07925358433874985, 0.11..."
5,GradientBoostingRegressor,"[-6.4132314212642845, -6.87640606650613, -7.13...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.4132314212642845, -6.77880317404218, -6.4...","[-6.618534345376179, -6.763064732414061, -6.62...","[0.18832005680893088, 0.08704204937536096, 0.1..."
6,AdaBoostRegressor,"[-6.593484676278435, -6.589927707868603, -6.53...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.593484676278435, -6.430034235919997, -6.2...","[-6.556978236155613, -6.429533850708988, -6.36...","[0.08807587145155367, 0.05718558821391062, 0.0..."
7,SVR,"[-6.532056614806742, -6.98960513364624, -7.075...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.532056614806742, -6.557506604279681, -6.1...","[-6.572328568697804, -6.661204996621135, -6.14...","[0.0835639223756577, 0.07406484669073116, 0.07..."
8,LinearRegression,"[-6.154831751050289, -7.017513688945867, -7.08...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.154831751050289, -5.573553823442962, -5.6...","[-6.135608168847521, -5.734091300609121, -5.66...","[0.014293264450051435, 0.11480166868913932, 0...."
9,KNeighborsRegressor,"[-7.4433333333333325, -7.3066666666666675, -7....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.4433333333333325, -6.6499999999999995, -6...","[-7.382666666666667, -6.730666666666667, -7.00...","[0.2023901622554262, 0.15333913032520063, 0.38..."


In [19]:
aac_comp.to_csv('results/Monomeric/AAC_comp_results_Caco2.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_prediction_data_Caco2.csv')

In [20]:
#Constant column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac_Caco2.csv')
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac_Caco2.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
aac_comp,prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
aac_comp

(1007, 18)
(1007,)
(252, 18)
(252,)
0.6321874712784235
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026318 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 178
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 13
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warn

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.027216044651674154


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2784,0.3934,0.5277,0.5656,0.7590,0.7418,0.2292,0.3528,0.4787,0.6322,0.7967,0.7556
LGBMRegressor,0.2869,0.4159,0.5356,0.5524,0.7435,0.7197,0.2600,0.3932,0.5099,0.5827,0.7634,0.7205
XGBRegressor,0.2847,0.4014,0.5336,0.5557,0.7574,0.7408,0.2487,0.3777,0.4987,0.6008,0.7801,0.7388
DecisionTreeRegressor,0.3558,0.4396,0.5965,0.4448,0.7112,0.6923,0.2587,0.3707,0.5086,0.5849,0.7746,0.7275
RandomForestRegressor,0.2631,0.3892,0.5129,0.5895,0.7696,0.7480,0.2342,0.3615,0.4839,0.6242,0.7905,0.7458
GradientBoostingRegressor,0.2908,0.4246,0.5393,0.5462,0.7398,0.7240,0.2490,0.4024,0.4990,0.6005,0.7794,0.7448
AdaBoostRegressor,0.4035,0.5447,0.6352,0.3704,0.6134,0.6025,0.3800,0.5266,0.6164,0.3902,0.6331,0.6163
SVR,0.3222,0.4423,0.5676,0.4973,0.7060,0.6890,0.2944,0.4227,0.5426,0.5275,0.7270,0.7024
LinearRegression,0.4554,0.5582,0.6748,0.2894,0.5387,0.5091,0.4660,0.5732,0.6826,0.2522,0.5029,0.4412
KNeighborsRegressor,0.3580,0.4540,0.5983,0.4414,0.6830,0.6659,0.3157,0.4226,0.5618,0.4934,0.7110,0.6834


In [21]:
aac_comp.to_csv('results/Monomeric/AAC_comp_results_const_rem_Caco2.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_prediction_data_const_rem_Caco2.csv')

In [22]:
#LVR column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac_Caco2.csv')
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
X_train, const_col = remove_low_variance_columns(X_train)

y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac_Caco2.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_mc = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_mc, X_train,y_train, X_test,  y_test)
result_df

(1007, 5)
(1007,)
(252, 5)
(252,)
0.47860832361440253
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001244 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 95
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 5
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.050740523488790545


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3684,0.4613,0.6069,0.4252,0.6711,0.6419,0.3249,0.4217,0.5700,0.4786,0.7055,0.6407
LGBMRegressor,0.3596,0.4683,0.5997,0.4389,0.6637,0.6254,0.3167,0.4371,0.5628,0.4917,0.7018,0.6319
XGBRegressor,0.3966,0.4736,0.6298,0.3811,0.6475,0.6216,0.3307,0.4346,0.5751,0.4693,0.7001,0.6374
DecisionTreeRegressor,0.4159,0.4825,0.6449,0.3511,0.6394,0.6127,0.3466,0.4371,0.5888,0.4437,0.6882,0.6178
RandomForestRegressor,0.3506,0.4539,0.5921,0.4529,0.6804,0.6442,0.3154,0.4219,0.5616,0.4939,0.7081,0.6407
GradientBoostingRegressor,0.3512,0.4701,0.5926,0.4520,0.6723,0.6302,0.3190,0.4523,0.5648,0.4880,0.6988,0.6361
AdaBoostRegressor,0.4681,0.5881,0.6842,0.2696,0.5325,0.4868,0.4310,0.5569,0.6565,0.3083,0.5666,0.5153
SVR,0.4115,0.5080,0.6415,0.3580,0.6038,0.5622,0.4061,0.4936,0.6373,0.3482,0.5985,0.5364
LinearRegression,0.5539,0.6214,0.7442,0.1357,0.3685,0.3328,0.5990,0.6364,0.7739,0.0387,0.2447,0.1935
KNeighborsRegressor,0.4567,0.5161,0.6758,0.2874,0.5998,0.5639,0.4156,0.4823,0.6446,0.3331,0.6184,0.5570


In [23]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-7.240000000000006, -6.2888888888888905, -6.2...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.240000000000006, -6.5837690476190485, -6....","[-7.215000000000005, -6.681444976190479, -6.10...","[0.01581138830083931, 0.09241996263331237, 0.0..."
1,LGBMRegressor,"[-6.734398408991375, -6.252326570497141, -6.25...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.734398408991375, -6.591221529303887, -6.1...","[-6.84859637819783, -6.508633313420499, -6.123...","[0.12503354494723248, 0.15132717401605036, 0.1..."
2,XGBRegressor,"[-7.227395, -6.255674, -6.255674, -6.1056747, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.227395, -6.1409283, -6.035343, -5.841507,...","[-7.2080255, -6.4419746, -5.92645, -5.761138, ...","[0.016136112, 0.48889858, 0.13624847, 0.049382..."
3,DecisionTreeRegressor,"[-7.24, -6.288888888888889, -6.288888888888889...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -5.89, -6.21, -5.85, -5.7, -6.2888888...","[-7.215000000000001, -6.672, -6.072, -5.765, -...","[0.01581138830084184, 0.6558170476588728, 0.17..."
4,RandomForestRegressor,"[-6.952304761904762, -6.288510802253301, -6.28...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.952304761904762, -6.407430705923889, -6.1...","[-7.048347142857146, -6.557763846606724, -6.13...","[0.09446605053591989, 0.14221276002137329, 0.0..."
5,GradientBoostingRegressor,"[-6.660027396897072, -6.0120906168439685, -6.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.660027396897072, -6.80443958227034, -6.19...","[-6.809215792960032, -6.822100280732499, -6.26...","[0.09534016085336046, 0.024818062803272763, 0...."
6,AdaBoostRegressor,"[-6.530402814544042, -6.360356471861508, -6.36...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.530402814544042, -6.66888806359669, -6.42...","[-6.4741267725942775, -6.610262295677119, -6.3...","[0.055853440736776803, 0.04762913154662724, 0...."
7,SVR,"[-5.880101841208261, -5.956852375942953, -5.95...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.880101841208261, -6.350905490146464, -6.3...","[-5.9508724114365545, -6.323901006879562, -6.2...","[0.08484503801162649, 0.045423682441562425, 0...."
8,LinearRegression,"[-6.015955736748274, -6.0844573135405, -6.0844...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.015955736748274, -6.770530138653449, -5.6...","[-6.032155976221347, -6.764550345352158, -5.66...","[0.015631605786701606, 0.028306999929333287, 0..."
9,KNeighborsRegressor,"[-6.058333333333334, -6.8566666666666665, -6.8...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.058333333333334, -6.196666666666666, -6.0...","[-6.422666666666666, -6.190666666666667, -5.91...","[0.4485477801874747, 0.14775956445824057, 0.05..."


In [24]:
result_df.to_csv('results/Monomeric/AAC_comp_LVR_results_Caco2.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_LVR_prediction_data_Caco2.csv')

In [25]:
#Atomic models
df_train = pd.read_csv('features/Atomic/Train_all_atomic_desc_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Atomic/Test_all_atomic_desc_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_degree = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_degree, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 23)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 23)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 497
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 17
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.3385391597875297


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2581,0.3883,0.5080,0.5973,0.7735,0.7385,0.1864,0.3321,0.4318,0.7008,0.8382,0.8052
DecisionTreeRegressor,0.3857,0.4593,0.6210,0.3982,0.6905,0.6599,0.2231,0.3500,0.4723,0.6419,0.8081,0.7771
RandomForestRegressor,0.2568,0.3819,0.5067,0.5993,0.7759,0.7474,0.1835,0.3267,0.4284,0.7055,0.8403,0.8096
GradientBoostingRegressor,0.2635,0.3986,0.5134,0.5888,0.7673,0.7311,0.1986,0.3493,0.4456,0.6813,0.8283,0.7916
AdaBoostRegressor,0.3281,0.4828,0.5728,0.4880,0.7114,0.6750,0.2941,0.4586,0.5423,0.5281,0.7482,0.6859
XGBRegressor,0.2778,0.3953,0.5271,0.5665,0.7637,0.7375,0.1867,0.3248,0.4321,0.7004,0.8380,0.8139
ExtraTreesRegressor,0.2721,0.3869,0.5216,0.5755,0.7672,0.7421,0.1689,0.3174,0.4110,0.7289,0.8539,0.8288
LinearRegression,0.3720,0.4989,0.6099,0.4196,0.6482,0.5908,0.3841,0.5129,0.6197,0.3836,0.6246,0.5278
KNeighborsRegressor,0.2794,0.3972,0.5286,0.5640,0.7577,0.7269,0.2094,0.3592,0.4576,0.6640,0.8152,0.7749
SVR,0.2884,0.4117,0.5370,0.5501,0.7452,0.7030,0.2252,0.3726,0.4746,0.6385,0.7996,0.7510


In [26]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.754422652015954, -7.111583378893853, -7.11...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.754422652015954, -5.966024805535367, -6.2...","[-6.931948080586923, -6.175923965470291, -6.47...","[0.13802386819722842, 0.15394930791906114, 0.1..."
1,DecisionTreeRegressor,"[-6.77, -7.05, -7.05, -7.22, -5.62, -5.54, -6....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.77, -6.0, -6.21, -5.85, -5.7207897275, -7...","[-7.116000000000001, -6.133, -6.32200000000000...","[0.17408044117591184, 0.506217344625804, 0.223..."
2,RandomForestRegressor,"[-6.848987885455004, -7.038399999999997, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.864987885455003, -6.054101750563331, -6.5...","[-6.977049539063008, -6.135700905142667, -6.60...","[0.07195729544386753, 0.10244054370127868, 0.1..."
3,GradientBoostingRegressor,"[-7.432323362182204, -7.083117659605887, -7.08...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.432323362182204, -6.545255935478724, -6.6...","[-7.2528132358002235, -6.679278781638989, -6.6...","[0.09095984342929965, 0.20649988932344793, 0.1..."
4,AdaBoostRegressor,"[-6.561295520813597, -6.634331797235033, -6.63...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.561295520813597, -6.338013235482812, -6.4...","[-6.770485839140052, -6.232131058147876, -6.68...","[0.16974967670856989, 0.06390011927927522, 0.2..."
5,XGBRegressor,"[-7.391063, -7.0727706, -7.0727706, -6.6725855...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.391063, -6.123893, -6.429983, -5.848609, ...","[-7.235428, -6.1012964, -6.435276, -5.8802347,...","[0.0792569, 0.09017505, 0.16200295, 0.05680899..."
6,ExtraTreesRegressor,"[-7.073520010585006, -7.030399999999995, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.091920010585007, -5.900505388240001, -6.5...","[-7.174584002117013, -6.014656261651001, -6.72...","[0.04142202160688046, 0.10389875669966449, 0.0..."
7,LinearRegression,"[-6.7296527968597415, -7.10277252935611, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.728485203304215, -6.3174285011340725, -5....","[-6.711627814398803, -6.332693608340135, -5.89...","[0.03002425777314033, 0.08399681950738272, 0.0..."
8,KNeighborsRegressor,"[-6.87, -7.3500000000000005, -7.35000000000000...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.87, -5.595, -6.863333333333333, -6.013749...","[-6.981333333333334, -5.564666666666666, -7.19...","[0.10504179062534011, 0.09847391081454557, 0.2..."
9,SVR,"[-6.736264840933409, -7.101254240551137, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.738272362301044, -6.152196579390029, -6.2...","[-6.808653603960916, -6.347974662665533, -6.42...","[0.10472019480354049, 0.10923587967659769, 0.1..."


In [27]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_Caco2.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_Caco2.csv')

In [28]:
#Atomic + monomeric_composition based features
df1 = pd.read_csv('features/Monomeric/Train_mon_comp_caco2.csv')
df2 = pd.read_csv('features/Atomic/Train_all_atomic_desc_Caco2.csv')
df_train = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_train

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Degree_F,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,0.071429,0.000000,0.071429,0.000000,0.0,0.0,0.0,...,0,87,14,0,24,0,0,154,1,1
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,0.071429,0.071429,0.071429,0.000000,0.0,0.0,0.0,...,0,86,14,0,24,0,0,155,1,1
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,85,15,0,12,0,0,155,1,1
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,0.071429,0.000000,0.142857,0.071429,0.0,0.0,0.0,...,0,85,14,0,18,0,0,149,1,1
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,85,15,0,12,0,0,148,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,22,4,0,6,0,0,40,1,1
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,0.250000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,22,4,0,6,0,0,41,1,1
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,0.333333,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,21,3,0,6,0,0,35,1,1
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,0.333333,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,20,3,0,6,0,0,35,1,1


In [29]:
df1 = pd.read_csv('features/Monomeric/Test_mon_comp_Caco2.csv')
df2 = pd.read_csv('features/Atomic/Test_all_atomic_desc_Caco2.csv')
df_test = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_test

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Degree_F,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,0.071429,0.000000,0.142857,0.000000,0.0,0.0,0.0,...,0,87,14,0,24,0,0,155,1,1
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,0.000000,0.000000,0.000000,0.083333,0.0,0.0,0.0,...,0,95,12,0,6,0,0,141,1,1
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,81,12,0,24,0,0,140,1,1
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,1,89,12,0,12,0,0,148,1,1
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,0.000000,0.090909,0.000000,0.000000,0.0,0.0,0.0,...,1,90,12,0,12,0,0,143,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,0.333333,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,23,3,0,6,0,0,35,1,1
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,0.333333,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,23,3,0,6,0,0,34,1,1
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,23,3,0,6,0,0,35,1,1
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,0.333333,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0,23,3,0,6,0,0,35,1,1


In [30]:
import re
def clean_feature_names(df):
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)
    df.columns = [clean_name(col) for col in df.columns]
    return df

In [31]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [32]:
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = clean_feature_names(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = clean_feature_names(X_test)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 408)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 408)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 835
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 57
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2114,0.3509,0.4598,0.6701,0.8186,0.8013,0.1601,0.3082,0.4001,0.7431,0.8629,0.8489
DecisionTreeRegressor,0.3238,0.4127,0.5690,0.4947,0.7489,0.7232,0.1874,0.3178,0.4329,0.6993,0.8382,0.8076
RandomForestRegressor,0.1918,0.3283,0.4380,0.7007,0.8372,0.8200,0.1449,0.2869,0.3807,0.7674,0.8779,0.8588
GradientBoostingRegressor,0.2039,0.3469,0.4516,0.6818,0.8276,0.8051,0.1746,0.3255,0.4179,0.7198,0.8528,0.8251
AdaBoostRegressor,0.3055,0.4598,0.5527,0.5233,0.7402,0.7134,0.2652,0.4319,0.5150,0.5744,0.7876,0.7335
XGBRegressor,0.2068,0.3366,0.4547,0.6773,0.8252,0.8087,0.1381,0.2789,0.3716,0.7784,0.8827,0.8661
ExtraTreesRegressor,0.1988,0.3227,0.4459,0.6898,0.8335,0.8192,0.1553,0.2871,0.3940,0.7508,0.8669,0.8351
LinearRegression,0.5065,0.4544,0.7117,0.2098,0.6355,0.7288,0.4041,0.4293,0.6357,0.3515,0.7016,0.7161
KNeighborsRegressor,0.3441,0.4316,0.5866,0.4630,0.6916,0.6777,0.2778,0.3986,0.5271,0.5541,0.7460,0.7117
SVR,0.2608,0.3843,0.5107,0.5931,0.7707,0.7623,0.2056,0.3489,0.4534,0.6701,0.8207,0.8009


In [33]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_and_mono_comp_Caco2.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_and_mono_comp_Caco2.csv')

In [34]:
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = clean_feature_names(X_train)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = clean_feature_names(X_test)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 230)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 230)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004647 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 835
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 57
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2114,0.3509,0.4598,0.6701,0.8186,0.8013,0.1601,0.3082,0.4001,0.7431,0.8629,0.8489
DecisionTreeRegressor,0.3249,0.4159,0.5700,0.4931,0.7466,0.7220,0.1836,0.3163,0.4285,0.7053,0.8407,0.8107
RandomForestRegressor,0.1927,0.3303,0.4390,0.6993,0.8364,0.8187,0.1455,0.2875,0.3814,0.7665,0.8773,0.8586
GradientBoostingRegressor,0.2029,0.3461,0.4504,0.6835,0.8286,0.8066,0.1734,0.3243,0.4164,0.7217,0.8541,0.8270
AdaBoostRegressor,0.3079,0.4644,0.5549,0.5196,0.7367,0.6946,0.2667,0.4348,0.5164,0.5720,0.7842,0.7265
XGBRegressor,0.2068,0.3366,0.4547,0.6773,0.8252,0.8087,0.1381,0.2789,0.3716,0.7784,0.8827,0.8661
ExtraTreesRegressor,0.1950,0.3192,0.4416,0.6958,0.8369,0.8219,0.1558,0.2873,0.3947,0.7500,0.8664,0.8364
LinearRegression,0.5065,0.4544,0.7117,0.2098,0.6355,0.7288,0.4041,0.4293,0.6357,0.3515,0.7016,0.7161
KNeighborsRegressor,0.3441,0.4316,0.5866,0.4630,0.6916,0.6777,0.2778,0.3986,0.5271,0.5541,0.7460,0.7117
SVR,0.2608,0.3843,0.5107,0.5931,0.7707,0.7622,0.2056,0.3489,0.4534,0.6701,0.8207,0.8009


In [35]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.896612355941233, -6.997146140291811, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.839770795376238, -6.246373339275974, -6.3...","[-6.9753048555422215, -6.281398385489146, -6.5...","[0.11480424961158599, 0.08867066452491894, 0.1..."
1,DecisionTreeRegressor,"[-7.24, -6.89, -7.0, -6.89, -6.16, -6.72, -6.8...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -6.07, -6.24, -5.85, -5.8, -7.0, -7.3...","[-6.890000000000001, -6.072000000000001, -6.3,...","[0.6252999280345396, 0.13891004283348266, 0.15..."
2,RandomForestRegressor,"[-6.879400000000005, -7.074399999999997, -6.99...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.857100000000006, -6.184900000000003, -6.5...","[-6.9378930147293385, -6.2927396907140025, -6....","[0.07550219101229438, 0.1055717705022892, 0.16..."
3,GradientBoostingRegressor,"[-6.589556041522321, -6.8253182090041795, -6.8...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.7174227767671955, -6.327527114444261, -6....","[-6.780084433167547, -6.307985443765072, -6.59...","[0.07754442510464928, 0.057280047912514044, 0...."
4,AdaBoostRegressor,"[-6.352909090909091, -6.453772170000001, -6.51...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.352909090909091, -6.352909090909091, -6.2...","[-6.548083169180205, -6.389679128709482, -6.35...","[0.21645789905270932, 0.06551977586362429, 0.1..."
5,XGBRegressor,"[-6.733893, -7.1260386, -7.005443, -6.90017, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.568446, -6.1547966, -6.5000176, -5.752067...","[-6.663398, -6.2179637, -6.516675, -5.8226385,...","[0.24760047, 0.22388083, 0.11908818, 0.0368654..."
6,ExtraTreesRegressor,"[-7.093800000000005, -7.257099999999997, -7.11...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.121800000000007, -6.177009053099999, -6.5...","[-7.16407598389601, -6.357585929965999, -6.677...","[0.03066364700025517, 0.19189252457513897, 0.1..."
7,LinearRegression,"[-6.427723434181164, -7.16874637496758, -7.030...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.226908069439451, -6.180463403833782, -5.3...","[-6.243226850635642, -6.499604961395714, -5.40...","[0.049436898912673916, 0.2994804241933193, 0.0..."
8,KNeighborsRegressor,"[-6.363333333333333, -7.3500000000000005, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.353333333333334, -6.14000000000000...","[-6.077999999999999, -6.369333333333334, -6.41...","[0.24302354709872184, 0.12070901650940005, 0.1..."
9,SVR,"[-6.544265430392661, -7.162905412896653, -7.11...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.3703085713698115, -5.892671105427937, -5....","[-6.40397771815686, -5.891769686373762, -5.678...","[0.03695639226291499, 0.017914077629174956, 0...."


In [36]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_and_mono_comp_const_rem_Caco2.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_and_mono_comp_const_rem_Caco2.csv')

In [37]:
const_col

['Ala_tBu_',
 'Me_Ala_indol_2_yl_',
 'Ala_5_Tet_',
 'Me_dAbu',
 '2Abz',
 'HOCOCH2_Bal',
 'Cys_EtO2H__NH2',
 'Cha',
 'dCha',
 'Asp_OMe_',
 'Asp_Ph_2_NH2__',
 'dAsp_pyrrol_1_yl_',
 'E',
 'Glu_NH2',
 'Glu_3R_Me_',
 'Glu_OMe_',
 'dGlu_OMe_',
 'Phe_4_F_',
 'dPhe_4_F_',
 'Phe_4_NO2_',
 'dPhe_3_4_diF_',
 'Me_Phe_a_b_dehydro_',
 'Bn_4_OH__Gly',
 'Et_Gly',
 'HOCOCH2_Gly_ol',
 'MeOEt_Gly',
 'NH2Bu_Gly',
 'PhPr_Gly',
 'cHexCH2_Gly',
 '2_pyridylmethyl_Gly',
 'd_N__O_Gly_allyl_',
 'GABA',
 'bHph',
 'dHyp',
 '_N__O_xiIle',
 'd_N__O_aIle',
 'Me_dK',
 'Lys_Cbz_',
 'Me_Lys_Me_',
 'Lys_Tfa_',
 'aMeLeu',
 'dLeu_3R_OH_',
 '_N__O_Leu',
 'd_N__O_Leu',
 'M',
 'meM',
 'Met_O2_',
 'meN',
 'dAsn_Me2_',
 'Nal',
 '1_Nal',
 'd1_Nal',
 'Me_dNle',
 'dNva',
 'Me_dNva',
 'Orn',
 'meQ',
 'dGln_Me2_',
 'R',
 'Arg_Me_Me_',
 'Ser_Ac_',
 'dSer_Me_',
 'Sta',
 'Sta_3R_4R_',
 'dT',
 'Tza',
 'Me_dV',
 '_N__O_Val',
 'd_N__O_Val',
 '_N__O_Val_3_OH_',
 'Trp_5_Br_',
 'Trp_6_Br_',
 'Trp_7_Br_',
 'dY',
 'Me_dY',
 'Me_Tyr_Me_',
 'dTy

In [39]:
#Fingerprints models
#All fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 20188)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 20188)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 44.000109 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12109
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2587
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1736,0.3195,0.4167,0.7291,0.8542,0.8438,0.1440,0.2911,0.3795,0.7689,0.8780,0.8595
DecisionTreeRegressor,0.3010,0.4031,0.5487,0.5303,0.7602,0.7514,0.1682,0.3113,0.4102,0.7300,0.8553,0.8409
RandomForestRegressor,0.1737,0.3176,0.4168,0.7290,0.8556,0.8456,0.1430,0.2919,0.3782,0.7705,0.8807,0.8634
GradientBoostingRegressor,0.1893,0.3384,0.4350,0.7047,0.8416,0.8330,0.1548,0.3083,0.3934,0.7516,0.8720,0.8501
AdaBoostRegressor,0.2778,0.4393,0.5270,0.5666,0.7626,0.7347,0.2417,0.4073,0.4916,0.6122,0.8003,0.7546
XGBRegressor,0.1843,0.3240,0.4293,0.7125,0.8446,0.8408,0.1339,0.2795,0.3660,0.7851,0.8867,0.8700
ExtraTreesRegressor,0.1909,0.3259,0.4369,0.7022,0.8386,0.8353,0.1544,0.2959,0.3929,0.7522,0.8674,0.8427
LinearRegression,2.0984,0.9848,1.4486,-2.2742,0.3179,0.4258,0.7950,0.6533,0.8916,-0.2758,0.5259,0.5745
KNeighborsRegressor,0.2502,0.3727,0.5002,0.6096,0.7854,0.7743,0.1844,0.3258,0.4294,0.7040,0.8414,0.8146
SVR,0.2247,0.3605,0.4740,0.6494,0.8076,0.7989,0.1990,0.3413,0.4460,0.6807,0.8289,0.8103


In [40]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.54202394719274, -7.089175043312861, -7.067...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.3130797770068945, -6.1865063870755606, -6...","[-6.726450986435317, -6.195841255207989, -6.57...","[0.22671678010193877, 0.10416220121854193, 0.1..."
1,DecisionTreeRegressor,"[-6.16, -7.0, -7.0, -6.89, -5.89, -6.44, -5.46...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.64, -5.835, -6.82, -5.85, -5.89, -7.0, -6...","[-6.890000000000001, -5.936, -6.491, -5.82, -5...","[0.6252999280345396, 0.5468582997450072, 0.369..."
2,RandomForestRegressor,"[-6.448551546379999, -7.250072161429994, -7.06...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.151217600416663, -6.305140000000002, -6.2...","[-6.598742931519334, -6.253808236726, -6.41164...","[0.26362335066588594, 0.053163349964435835, 0...."
3,GradientBoostingRegressor,"[-6.756820505702874, -7.195328391064949, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.746825321632111, -6.120191269951889, -6.4...","[-6.809585041686184, -6.160470176860888, -6.50...","[0.11185889678532618, 0.07146811249980477, 0.1..."
4,AdaBoostRegressor,"[-6.6291463414634135, -6.840813970434778, -6.2...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.21144292944479, -6.211043165467638, -6.18...","[-6.50334784603525, -6.139070271721728, -6.178...","[0.2631904337041914, 0.05701258239837439, 0.03..."
5,XGBRegressor,"[-6.8884892, -7.4920135, -7.1608243, -6.220619...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6289363, -5.821832, -6.405744, -5.841382,...","[-6.892578, -5.9083986, -6.536628, -5.851502, ...","[0.24762942, 0.1301441, 0.21530905, 0.01108467..."
6,ExtraTreesRegressor,"[-6.336099999999999, -7.350899999999997, -6.94...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.162549999999998, -6.032149999999996, -6.2...","[-6.5459458107940005, -6.120609999999999, -6.2...","[0.2662612021071345, 0.09837911566994394, 0.17..."
7,LinearRegression,"[-5.565557685559003, -6.203603026942818, -7.72...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.490142134940051, -9.275223097793475, -6.4...","[-6.043990583304499, -8.216538415907308, -6.30...","[0.4885916427093304, 1.535514634776705, 0.2514..."
8,KNeighborsRegressor,"[-6.646666666666666, -7.343333333333334, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.646666666666666, -5.706666666666666, -6.8...","[-6.764, -5.955, -6.668000000000001, -5.810666...","[0.2010682581723044, 0.1319132711712937, 0.262..."
9,SVR,"[-6.527406145924302, -7.141792038599275, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.376372651085278, -6.135957809753267, -5.9...","[-6.483122257574964, -6.2347235745852405, -5.9...","[0.22814746764112895, 0.05201627830200602, 0.0..."


In [41]:
result_df.to_csv('results/Fingerprints/Results_All_fingerprints_fp_Caco2.csv')
prediction_df.to_csv('results/Fingerprints/Prediction_data_All_fingerprints_fp_Caco2.csv')

In [42]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [43]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [44]:
#All fingerprints constant removal
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 5503)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 5503)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.091417 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12109
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2587
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1736,0.3195,0.4167,0.7291,0.8542,0.8438,0.1440,0.2911,0.3795,0.7689,0.8780,0.8595
DecisionTreeRegressor,0.2886,0.3922,0.5372,0.5497,0.7695,0.7548,0.1720,0.3143,0.4147,0.7240,0.8518,0.8392
RandomForestRegressor,0.1738,0.3177,0.4169,0.7288,0.8555,0.8457,0.1425,0.2913,0.3775,0.7713,0.8812,0.8641
GradientBoostingRegressor,0.1896,0.3387,0.4355,0.7041,0.8412,0.8327,0.1549,0.3084,0.3936,0.7514,0.8718,0.8502
AdaBoostRegressor,0.2815,0.4406,0.5306,0.5607,0.7614,0.7365,0.2430,0.4086,0.4929,0.6100,0.8021,0.7575
XGBRegressor,0.1843,0.3240,0.4293,0.7125,0.8446,0.8408,0.1339,0.2795,0.3660,0.7851,0.8867,0.8700
ExtraTreesRegressor,0.1908,0.3262,0.4368,0.7023,0.8386,0.8355,0.1531,0.2951,0.3912,0.7544,0.8686,0.8456
LinearRegression,2.0984,0.9848,1.4486,-2.2742,0.3179,0.4258,0.7950,0.6533,0.8916,-0.2758,0.5259,0.5745
KNeighborsRegressor,0.2502,0.3727,0.5002,0.6096,0.7854,0.7743,0.1844,0.3258,0.4294,0.7040,0.8414,0.8146
SVR,0.2247,0.3605,0.4740,0.6494,0.8076,0.7989,0.1989,0.3413,0.4460,0.6807,0.8289,0.8103


In [45]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.54202394719274, -7.089175043312861, -7.067...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.3130797770068945, -6.1865063870755606, -6...","[-6.726450986435317, -6.195841255207989, -6.57...","[0.22671678010193877, 0.10416220121854193, 0.1..."
1,DecisionTreeRegressor,"[-6.16, -7.0, -6.89, -6.89, -5.835, -6.44, -5....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.64, -5.82, -6.82, -5.85, -5.85, -7.0, -5....","[-6.654000000000001, -6.384, -6.67400000000000...","[0.7048290572897803, 0.7493357058088181, 0.236..."
2,RandomForestRegressor,"[-6.454761846340002, -7.291572161429996, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.16434517967333, -6.337133333333332, -6.24...","[-6.631803887201334, -6.273940263218664, -6.40...","[0.26856000631128724, 0.06487407293911299, 0.0..."
3,GradientBoostingRegressor,"[-6.756820505702872, -7.251059741233977, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.74682532163211, -6.12019126995189, -6.428...","[-6.789723648063192, -6.160470176860889, -6.50...","[0.09189681139779317, 0.07146811249980363, 0.1..."
4,AdaBoostRegressor,"[-6.548634729287383, -6.988956521739118, -6.38...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.280956831695096, -6.21752237802439, -6.21...","[-6.5229629286963675, -6.2021109456794665, -6....","[0.2480691787276146, 0.02223413700978757, 0.09..."
5,XGBRegressor,"[-6.8884892, -7.4920135, -7.1608243, -6.220619...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6289363, -5.821832, -6.405744, -5.841382,...","[-6.892578, -5.9083986, -6.536628, -5.851502, ...","[0.24762942, 0.1301441, 0.21530905, 0.01108467..."
6,ExtraTreesRegressor,"[-6.35283228266, -7.299199999999998, -6.932299...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.076183829039999, -6.097699999999998, -6.2...","[-6.537426696586001, -6.14637, -6.315150000000...","[0.2726077148296817, 0.09909386257483552, 0.18..."
7,LinearRegression,"[-5.565557685557012, -6.2036030269428055, -7.7...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.4901421349400215, -9.275223097813232, -6....","[-6.043990583304451, -8.216538415912428, -6.30...","[0.4885916427094256, 1.5355146347715927, 0.251..."
8,KNeighborsRegressor,"[-6.646666666666666, -7.343333333333334, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.646666666666666, -5.706666666666666, -6.8...","[-6.764, -5.955, -6.668000000000001, -5.810666...","[0.2010682581723044, 0.1319132711712937, 0.262..."
9,SVR,"[-6.527405895742985, -7.141791958913238, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.376372401215437, -6.135957807721879, -5.9...","[-6.483056060919954, -6.234753053021122, -5.98...","[0.22818060023406608, 0.05202636574403531, 0.0..."


In [46]:
result_df.to_csv('results/Fingerprints/Results_All_const_rem_fingerprints_Caco2.csv')
prediction_df.to_csv('results/Fingerprints/Prediction_data_All_const_rem_fingerprints_Caco2.csv')

In [47]:
#Morgan fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/morgan_fp_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/morgan_fp_test_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_fp

X_train shape:  (1007, 2048)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 2048)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009632 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 780
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 260
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2213,0.3647,0.4704,0.6547,0.8095,0.8042,0.1953,0.3443,0.4420,0.6865,0.8306,0.8043
DecisionTreeRegressor,0.4118,0.4650,0.6417,0.3575,0.6733,0.6648,0.1980,0.3409,0.4450,0.6822,0.8271,0.7950
RandomForestRegressor,0.2131,0.3509,0.4616,0.6675,0.8177,0.8141,0.1724,0.3171,0.4152,0.7234,0.8544,0.8280
GradientBoostingRegressor,0.2361,0.3824,0.4859,0.6316,0.7981,0.7935,0.2188,0.3745,0.4678,0.6488,0.8100,0.7639
AdaBoostRegressor,0.3573,0.5135,0.5978,0.4425,0.6833,0.6458,0.3354,0.4984,0.5791,0.4617,0.7041,0.6450
XGBRegressor,0.2391,0.3698,0.4890,0.6269,0.7968,0.8000,0.1845,0.3229,0.4295,0.7039,0.8403,0.8077
ExtraTreesRegressor,0.3685,0.4390,0.6071,0.4250,0.7010,0.6895,0.1926,0.3328,0.4389,0.6909,0.8318,0.8033
LinearRegression,0.3753,0.4457,0.6126,0.4143,0.7155,0.7406,0.3395,0.4381,0.5826,0.4552,0.7317,0.7261
KNeighborsRegressor,0.2956,0.4110,0.5437,0.5387,0.7453,0.7289,0.2211,0.3555,0.4702,0.6452,0.8072,0.7871
SVR,0.2498,0.3796,0.4998,0.6102,0.7828,0.7834,0.2330,0.3649,0.4827,0.6260,0.7956,0.7837


In [48]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.187139019706735, -7.107650951621631, -6.92...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.908710784306917, -5.8957299500075395, -5....","[-5.952691869304015, -5.878571023739924, -5.94...","[0.15830597237039098, 0.12278973610559706, 0.1..."
1,DecisionTreeRegressor,"[-7.48, -8.0, -7.0, -8.0, -5.835, -6.68, -6.7,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.64, -5.89, -5.92, -5.85, -5.96, -6.89, -6...","[-5.99, -5.846, -5.927999999999999, -5.816, -5...","[0.6017973080697523, 0.2617326880616939, 0.070..."
2,RandomForestRegressor,"[-6.717006666666663, -7.405699999999997, -6.97...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.860846335313329, -6.076951666666668, -5.9...","[-5.950268944312664, -6.107383857142857, -5.95...","[0.16158441587560837, 0.030183174075764645, 0...."
3,GradientBoostingRegressor,"[-6.044452465414234, -7.311749828164496, -6.78...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.128329057871889, -6.2454317683582214, -5....","[-6.143048725580941, -6.247793853262576, -5.96...","[0.0801019928043006, 0.07777573734930662, 0.04..."
4,AdaBoostRegressor,"[-6.453746145861258, -6.625024088700903, -6.46...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.304351086505014, -6.3120426768463345, -6....","[-6.308531428477974, -6.226540683429008, -6.32...","[0.023725144684871868, 0.0747556634335139, 0.0..."
5,XGBRegressor,"[-6.187684, -7.6495337, -6.9216437, -7.043452,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.692211, -5.8782115, -5.9105783, -5.826021...","[-5.835387, -6.0612864, -5.87897, -5.8047285, ...","[0.2838243, 0.102390006, 0.0519276, 0.04738293..."
6,ExtraTreesRegressor,"[-7.450000000000005, -7.663199999999997, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.647199999999993, -5.750499999999997, -5.9...","[-5.975240000000001, -5.975139999999997, -5.96...","[0.6079152625160917, 0.43292728996911106, 0.09..."
7,LinearRegression,"[-6.545804566267349, -7.195267456753081, -7.50...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.9389382366218655, -5.911222288979818, -5....","[-6.116234738340955, -5.85394809868737, -5.807...","[0.18610899058099303, 0.21772158928874674, 0.0..."
8,KNeighborsRegressor,"[-6.646666666666666, -7.343333333333334, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.646666666666666, -5.921666666666667, -6.2...","[-6.764, -5.998, -6.1899999999999995, -5.84200...","[0.2010682581723044, 0.05865530192953126, 0.13..."
9,SVR,"[-6.439344689387928, -7.202284649277825, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.452686535341325, -5.973071989640252, -5.8...","[-6.690939312612227, -6.052808564018333, -5.82...","[0.2622704769540775, 0.053882118339350187, 0.0..."


In [49]:
df_morgan_fp.to_csv('results/Fingerprints/Results_Morgan_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Morgan_fp_Caco2.csv')

In [50]:
#Morgan count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/count_morgan_fp_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/count_morgan_fp_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_count_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_count_fp

X_train shape:  (1007, 2048)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 2048)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009957 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1388
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 272
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1842,0.3286,0.4292,0.7126,0.8447,0.8360,0.1588,0.3042,0.3985,0.7451,0.8647,0.8498
DecisionTreeRegressor,0.3362,0.4220,0.5798,0.4754,0.7346,0.7150,0.1738,0.3203,0.4169,0.7210,0.8494,0.8300
RandomForestRegressor,0.1781,0.3161,0.4220,0.7221,0.8515,0.8442,0.1499,0.2953,0.3871,0.7595,0.8742,0.8579
GradientBoostingRegressor,0.2011,0.3453,0.4484,0.6863,0.8327,0.8234,0.1735,0.3264,0.4166,0.7215,0.8537,0.8341
AdaBoostRegressor,0.3153,0.4705,0.5615,0.5080,0.7275,0.7233,0.2855,0.4485,0.5343,0.5418,0.7568,0.7368
XGBRegressor,0.1956,0.3329,0.4422,0.6949,0.8346,0.8299,0.1442,0.2844,0.3797,0.7686,0.8779,0.8627
ExtraTreesRegressor,0.1843,0.3207,0.4293,0.7124,0.8443,0.8386,0.1549,0.2967,0.3935,0.7515,0.8675,0.8521
LinearRegression,0.2632,0.3879,0.5131,0.5893,0.7809,0.7840,0.2719,0.3856,0.5215,0.5636,0.7834,0.7709
KNeighborsRegressor,0.2515,0.3733,0.5015,0.6076,0.7815,0.7606,0.2044,0.3406,0.4521,0.6719,0.8227,0.8115
SVR,0.2336,0.3658,0.4833,0.6355,0.7985,0.7930,0.2061,0.3395,0.4540,0.6692,0.8209,0.8087


In [51]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.5414879799116505, -7.029786118847843, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.370152247402247, -6.170990910418278, -6.4...","[-6.540442410258438, -6.265462678549791, -6.56...","[0.14764247747864048, 0.1772126874382535, 0.09..."
1,DecisionTreeRegressor,"[-7.47, -8.0, -7.0, -7.22, -6.11, -6.7, -5.34,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -7.66, -6.7, -5.85, -5.96, -7.0, -6.3...","[-6.836774392800001, -6.314, -6.58799999999999...","[0.4684101639965073, 0.6990164518807837, 0.330..."
2,RandomForestRegressor,"[-6.627600000000002, -7.097800000000001, -6.96...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.540170000000002, -6.256425, -6.5066774392...","[-6.513828652235334, -6.248860333333333, -6.53...","[0.12325960377769357, 0.12006388879999734, 0.0..."
3,GradientBoostingRegressor,"[-6.130765798904613, -7.153848336824983, -6.93...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.461481015103466, -6.4785446053629965, -6....","[-6.536584903090831, -6.357122553759629, -6.45...","[0.07934298324122134, 0.14232872641444944, 0.0..."
4,AdaBoostRegressor,"[-6.306255742844302, -6.289341953896916, -6.22...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.231037994747612, -6.231037994747612, -6.3...","[-6.2879544427786005, -6.240987633013953, -6.2...","[0.10287206712375913, 0.05086545391879181, 0.0..."
5,XGBRegressor,"[-7.1061993, -7.5125227, -7.2517953, -6.380294...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5637994, -6.0844193, -6.5793133, -5.81090...","[-6.671776, -5.9783325, -6.4604597, -5.8041773...","[0.1089868, 0.2025249, 0.19037998, 0.04512759,..."
6,ExtraTreesRegressor,"[-6.486849999999997, -7.5540999999999965, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.273849999999999, -6.197999999999997, -6.2...","[-6.44608, -6.122869999999997, -6.407697743928...","[0.13056454955308758, 0.10198688445089385, 0.1..."
7,LinearRegression,"[-6.63637690950577, -7.306814804353056, -7.152...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.993413822545603, -5.972256847687078, -5.5...","[-6.1528214376795685, -6.211704929902202, -5.5...","[0.20877906068773508, 0.1403621903578366, 0.03..."
8,KNeighborsRegressor,"[-6.646666666666666, -7.343333333333334, -6.98...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.646666666666666, -5.618333333333332, -6.2...","[-6.764, -5.853666666666667, -6.22466666666666...","[0.2010682581723044, 0.13010423171869243, 0.10..."
9,SVR,"[-6.5244907706063096, -7.245460188237773, -7.1...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5185880162467855, -6.069677968486385, -5....","[-6.755631749291304, -6.120203798889731, -5.82...","[0.22831278713933678, 0.028318161212210584, 0...."


In [52]:
df_morgan_count_fp.to_csv('results/Fingerprints/Results_Count_Morgan_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Count_Morgan_fp_Caco2.csv')

In [53]:
#AtomPairs2d fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2D_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2D_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2D_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2D_fp

X_train shape:  (1007, 780)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 780)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004272 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 279
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 93
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4299,0.5513,0.6557,0.3291,0.5737,0.4597,0.3904,0.5127,0.6248,0.3735,0.6118,0.4998
DecisionTreeRegressor,0.4240,0.5405,0.6511,0.3385,0.5846,0.4711,0.3882,0.5109,0.6231,0.3769,0.6156,0.4933
RandomForestRegressor,0.4161,0.5352,0.6451,0.3507,0.5932,0.4789,0.3836,0.5076,0.6193,0.3845,0.6206,0.4974
GradientBoostingRegressor,0.4142,0.5377,0.6436,0.3536,0.5947,0.4793,0.3829,0.5099,0.6188,0.3855,0.6211,0.5067
AdaBoostRegressor,0.4462,0.5739,0.6680,0.3038,0.5531,0.4220,0.4188,0.5480,0.6471,0.3280,0.5776,0.4239
XGBRegressor,0.4171,0.5356,0.6459,0.3491,0.5925,0.4786,0.3880,0.5121,0.6229,0.3774,0.6164,0.5003
ExtraTreesRegressor,0.4225,0.5402,0.6500,0.3407,0.5862,0.4709,0.3884,0.5112,0.6232,0.3767,0.6153,0.4937
LinearRegression,0.4239,0.5443,0.6511,0.3385,0.5837,0.4716,0.4017,0.5268,0.6338,0.3553,0.6036,0.5059
KNeighborsRegressor,0.4937,0.5694,0.7027,0.2296,0.5368,0.4458,0.4289,0.5363,0.6549,0.3116,0.5904,0.5065
SVR,0.4244,0.5412,0.6514,0.3378,0.5824,0.4788,0.3956,0.5146,0.6289,0.3652,0.6048,0.4999


In [54]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.468659891249207, -6.646573858887905, -6.64...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.468659891249207, -5.887418963204108, -6.4...","[-6.451096353158418, -6.23046579318943, -6.451...","[0.01039576325170493, 0.2645081283598121, 0.01..."
1,DecisionTreeRegressor,"[-6.475860759493668, -7.2700000000000005, -7.2...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.475860759493668, -5.755357142857142, -6.4...","[-6.465300477821283, -5.812567765567765, -6.46...","[0.009320426057594576, 0.03829173865021229, 0...."
2,RandomForestRegressor,"[-6.476363710576915, -7.2362705411255375, -7.2...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.476363710576916, -5.757301023088537, -6.4...","[-6.464153881752523, -5.809970101052282, -6.46...","[0.012553405453399918, 0.03727087607137162, 0...."
3,GradientBoostingRegressor,"[-6.461306820405278, -7.1891346353965355, -7.1...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.461306820405278, -5.821074498645228, -6.4...","[-6.4540267362528025, -5.870983835392373, -6.4...","[0.008840489869515656, 0.035958925212100305, 0..."
4,AdaBoostRegressor,"[-6.464510546822876, -6.495219605826151, -6.49...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.464510546822876, -5.845277777777777, -6.4...","[-6.454458864315503, -6.195863398978763, -6.45...","[0.012393496724563675, 0.2497437118447272, 0.0..."
5,XGBRegressor,"[-6.475813, -7.2693768, -7.2693768, -7.0374303...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.475813, -5.755599, -6.475813, -5.850096, ...","[-6.465239, -5.812947, -6.465239, -5.874279, -...","[0.009329559, 0.038354944, 0.009329559, 0.0473..."
6,ExtraTreesRegressor,"[-6.47586075949368, -7.26999999999999, -7.2699...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.47586075949368, -5.755357142857139, -6.47...","[-6.465300477821292, -5.812567765567765, -6.46...","[0.009320426057594844, 0.03829173865021755, 0...."
7,LinearRegression,"[-6.462195250957275, -7.113512911383492, -7.11...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.462195250957275, -5.867790962031551, -6.4...","[-6.4555323390887365, -5.897486233742362, -6.4...","[0.008223973953173633, 0.02309663778049059, 0...."
8,KNeighborsRegressor,"[-6.28, -7.36, -7.36, -7.036666666666666, -5.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -5.9316666666666675, -6.28, -5.87, -5...","[-6.544, -6.006666666666668, -6.544, -5.960666...","[0.22580620993330663, 0.04082482904638626, 0.2..."
9,SVR,"[-6.508316097535734, -7.099941375389619, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.508316097535734, -5.859917834318121, -6.5...","[-6.4876126166240144, -5.876257425852801, -6.4...","[0.019982898961570163, 0.01928620708937343, 0...."


In [55]:
df_AtomPairs2D_fp.to_csv('results/Fingerprints/Results_AtomPairs2D_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_AtomPairs2D_fp_Caco2.csv')

In [56]:
#AtomPairs2d Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2DCount_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2DCount_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2DCount_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2DCount_fp

X_train shape:  (1007, 780)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 780)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2301
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 132
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2208,0.3652,0.4699,0.6555,0.8096,0.7915,0.1558,0.3124,0.3948,0.7499,0.8683,0.8477
DecisionTreeRegressor,0.4058,0.4652,0.6371,0.3667,0.6789,0.6518,0.1978,0.3414,0.4447,0.6826,0.8273,0.8002
RandomForestRegressor,0.2273,0.3687,0.4768,0.6453,0.8037,0.7806,0.1621,0.3156,0.4026,0.7398,0.8644,0.8452
GradientBoostingRegressor,0.2365,0.3830,0.4863,0.6310,0.7954,0.7755,0.1790,0.3363,0.4231,0.7128,0.8511,0.8196
AdaBoostRegressor,0.3320,0.4881,0.5762,0.4820,0.7022,0.6753,0.2870,0.4561,0.5357,0.5394,0.7551,0.7147
XGBRegressor,0.2343,0.3678,0.4841,0.6344,0.7996,0.7830,0.1555,0.2989,0.3943,0.7504,0.8664,0.8406
ExtraTreesRegressor,0.2170,0.3529,0.4658,0.6614,0.8134,0.7997,0.1468,0.2952,0.3832,0.7643,0.8753,0.8586
LinearRegression,0.3026,0.4202,0.5501,0.5279,0.7374,0.7401,0.3589,0.4361,0.5991,0.4240,0.6883,0.6844
KNeighborsRegressor,0.2935,0.4055,0.5418,0.5420,0.7454,0.7199,0.2122,0.3574,0.4606,0.6595,0.8139,0.7771
SVR,0.2895,0.4223,0.5380,0.5483,0.7434,0.7028,0.2529,0.3950,0.5029,0.5941,0.7765,0.7122


In [57]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.703475111117595, -7.276579243765134, -7.14...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.975310771394594, -6.212566729324526, -6.5...","[-6.884834558753643, -6.3939524730050845, -6.6...","[0.13873439530992926, 0.10024124023766673, 0.0..."
1,DecisionTreeRegressor,"[-7.24, -7.0, -7.05, -6.89, -5.89, -7.24, -7.2...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -6.66, -6.82, -5.85, -6.82, -7.0, -6....","[-6.589999999999999, -6.433, -6.566, -5.736, -...","[0.7758865896508332, 0.3873706235635325, 0.291..."
2,RandomForestRegressor,"[-6.4510000000000005, -7.300133333333331, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5592999999999995, -6.3609093195600055, -6...","[-6.658406666666669, -6.421239607840002, -6.58...","[0.14716758308510589, 0.05452873208445444, 0.0..."
3,GradientBoostingRegressor,"[-6.626325458919022, -6.89960507024043, -6.859...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.046718888165866, -6.214757364617255, -6.5...","[-6.933604497657072, -6.336786966413021, -6.68...","[0.15333653191093213, 0.19708389431803403, 0.1..."
4,AdaBoostRegressor,"[-6.59140350877193, -6.7428358208955235, -6.74...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6408602150537615, -6.364042553191492, -6....","[-6.716525224888834, -6.519181384814738, -6.41...","[0.13280807413311435, 0.08739092464982226, 0.1..."
5,XGBRegressor,"[-6.568456, -7.132397, -7.1177907, -6.8571453,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.86004, -6.4497824, -6.542727, -5.85496, -...","[-6.8726363, -6.3546576, -6.734441, -5.845447,...","[0.17798942, 0.20832224, 0.19090977, 0.0105844..."
6,ExtraTreesRegressor,"[-6.522599999999999, -7.139099999999997, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.834600000000003, -6.3252500000000005, -6....","[-6.739360000000003, -6.423714119982, -6.63543...","[0.3826757875800396, 0.079346646478975, 0.0887..."
7,LinearRegression,"[-6.053943002001618, -7.277113665209492, -6.88...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.579057365770716, -6.013970093135591, -5.4...","[-5.696366633793763, -6.165467727421313, -5.46...","[0.109934740686308, 0.15186890247023502, 0.070..."
8,KNeighborsRegressor,"[-6.28, -7.09, -7.3500000000000005, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -5.918333333333333, -6.04666666666666...","[-6.462000000000001, -6.0040000000000004, -6.3...","[0.23225082226860794, 0.045772383716725394, 0...."
9,SVR,"[-6.067517714321913, -6.952309805679351, -6.95...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.882722598560687, -6.1202465262347605, -5....","[-5.954597345085347, -6.283664263811469, -5.52...","[0.09444704946573003, 0.09521716572185665, 0.0..."


In [58]:
df_AtomPairs2DCount_fp.to_csv('results/Fingerprints/Results_AtomPairs2D_Count_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_df_AtomPairs2D_Count_fp_Caco2.csv')

In [59]:
#EState fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/EState_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/EState_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_estate_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_estate_fp

X_train shape:  (1007, 79)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 79)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.079346 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 30
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 10
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4476,0.5414,0.6690,0.3016,0.5502,0.4638,0.4439,0.5454,0.6662,0.2876,0.5403,0.5191
DecisionTreeRegressor,0.4378,0.5347,0.6617,0.3169,0.5683,0.4854,0.3860,0.5048,0.6213,0.3806,0.6174,0.5535
RandomForestRegressor,0.4316,0.5316,0.6570,0.3265,0.5739,0.4867,0.3933,0.5136,0.6271,0.3689,0.6076,0.5376
GradientBoostingRegressor,0.4308,0.5369,0.6564,0.3278,0.5734,0.4897,0.4241,0.5435,0.6513,0.3193,0.5653,0.5228
AdaBoostRegressor,0.4903,0.5916,0.7002,0.2349,0.4869,0.4457,0.4979,0.5974,0.7056,0.2010,0.4575,0.4837
XGBRegressor,0.4325,0.5325,0.6576,0.3252,0.5744,0.4891,0.3941,0.5122,0.6278,0.3675,0.6073,0.5482
ExtraTreesRegressor,0.4378,0.5348,0.6617,0.3169,0.5683,0.4852,0.3855,0.5041,0.6209,0.3813,0.6179,0.5546
LinearRegression,0.5058,0.5933,0.7112,0.2107,0.4598,0.3893,0.5297,0.6156,0.7278,0.1499,0.3996,0.3390
KNeighborsRegressor,0.5300,0.5828,0.7280,0.1731,0.4705,0.4072,0.4842,0.5699,0.6958,0.2230,0.5071,0.4681
SVR,0.4472,0.5163,0.6687,0.3023,0.5549,0.5014,0.4005,0.5075,0.6329,0.3573,0.5991,0.5411


In [60]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.562376063651145, -6.385984740698623, -6.38...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.562376063651145, -6.1895178668280755, -6....","[-6.551674959193825, -6.196002412729412, -6.19...","[0.017306300525522487, 0.022838905723536678, 0..."
1,DecisionTreeRegressor,"[-6.5876691729323325, -7.2700000000000005, -7....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5876691729323325, -6.189531250000003, -6....","[-6.57395907697529, -6.19446358349091, -6.1944...","[0.014999513466367633, 0.02437769905083331, 0...."
2,RandomForestRegressor,"[-6.588513815128117, -7.235436733821731, -7.23...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.58851381512812, -6.194443171642051, -6.19...","[-6.572895390968218, -6.195953846753588, -6.19...","[0.017394052280790457, 0.02537588225160562, 0...."
3,GradientBoostingRegressor,"[-6.530739445352666, -7.179650556735497, -7.17...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.530739445352666, -6.169051153508325, -6.1...","[-6.511289425219395, -6.175144698329147, -6.17...","[0.014066155665352222, 0.019892173280669306, 0..."
4,AdaBoostRegressor,"[-6.421179245283013, -6.470597402597407, -6.47...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.421179245283013, -6.267238372093021, -6.2...","[-6.391671855207873, -6.291657841316703, -6.29...","[0.026565857328181685, 0.03413290924923251, 0...."
5,XGBRegressor,"[-6.5871205, -7.266745, -7.266745, -7.4786177,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5871205, -6.1892548, -6.1892548, -5.85573...","[-6.5732727, -6.194278, -6.194278, -5.8713884,...","[0.015007186, 0.024242908, 0.024242908, 0.0346..."
6,ExtraTreesRegressor,"[-6.587669172932343, -7.26999999999999, -7.269...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.587669172932343, -6.189531249999998, -6.1...","[-6.573959076975292, -6.194463583490913, -6.19...","[0.014999513466369869, 0.0243776990508343, 0.0..."
7,LinearRegression,"[-6.467157445913245, -7.448339096228841, -7.44...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.467157445913245, -6.11933349575399, -6.11...","[-6.446417281670631, -6.143437366719235, -6.14...","[0.0114460032823387, 0.02195362352998485, 0.02..."
8,KNeighborsRegressor,"[-6.28, -7.36, -7.36, -7.036666666666666, -6.3...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.366666666666666, -6.36666666666666...","[-6.544, -6.380000000000001, -6.38000000000000...","[0.22580620993330663, 0.18086213288334022, 0.1..."
9,SVR,"[-6.719724755825709, -7.099858462210147, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.719724755825709, -5.9898309564981576, -5....","[-6.701834413479078, -5.981727146828557, -5.98...","[0.0146506798309607, 0.02759958732556664, 0.02..."


In [61]:
df_estate_fp.to_csv('results/Fingerprints/Results_EState_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_EState_fp_Caco2.csv')

In [62]:
#Extended fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Extended_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Extended_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_extended_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_extended_fp

X_train shape:  (1007, 1024)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1024)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017696 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1353
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 451
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2799,0.4089,0.5290,0.5633,0.7507,0.7333,0.2578,0.4093,0.5077,0.5863,0.7684,0.7312
DecisionTreeRegressor,0.3996,0.4583,0.6321,0.3765,0.6719,0.6631,0.3088,0.4139,0.5557,0.5044,0.7228,0.6802
RandomForestRegressor,0.2845,0.3981,0.5334,0.5561,0.7479,0.7312,0.2567,0.3956,0.5067,0.5880,0.7677,0.7295
GradientBoostingRegressor,0.3039,0.4374,0.5512,0.5258,0.7267,0.7035,0.2717,0.4265,0.5213,0.5639,0.7550,0.7070
AdaBoostRegressor,0.4098,0.5505,0.6402,0.3605,0.6107,0.5789,0.3924,0.5380,0.6264,0.3703,0.6225,0.5823
XGBRegressor,0.2829,0.3950,0.5319,0.5586,0.7555,0.7483,0.2542,0.3866,0.5042,0.5921,0.7713,0.7392
ExtraTreesRegressor,0.3877,0.4532,0.6226,0.3951,0.6770,0.6661,0.3018,0.4109,0.5493,0.5157,0.7270,0.6845
LinearRegression,0.5572,0.5240,0.7465,0.1305,0.5784,0.6453,0.3648,0.4749,0.6040,0.4145,0.6741,0.6489
KNeighborsRegressor,0.3454,0.4408,0.5877,0.4611,0.7013,0.6822,0.3251,0.4363,0.5702,0.4783,0.7134,0.6644
SVR,0.3061,0.4279,0.5532,0.5224,0.7231,0.7022,0.2840,0.4252,0.5329,0.5443,0.7391,0.6967


In [63]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.991084248105287, -7.215073781308516, -7.21...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.832829954765613, -6.2043992597030675, -6....","[-6.79395277042151, -6.33466211894083, -6.3120...","[0.11032687838942429, 0.09062387568666776, 0.0..."
1,DecisionTreeRegressor,"[-5.74, -7.343333333333334, -7.343333333333334...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.866666666666667, -6.9319999999999995, -5....","[-6.254333333333333, -6.5024, -6.2260000000000...","[0.544935877981172, 0.45232625393624876, 0.816..."
2,RandomForestRegressor,"[-6.669995000000004, -7.328231666666666, -7.32...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.842550000000001, -6.440117947330444, -6.8...","[-6.704126984126984, -6.536046957431457, -6.60...","[0.1542703274261865, 0.08917272669189312, 0.20..."
3,GradientBoostingRegressor,"[-6.9652564038652045, -7.084890774677078, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.68871855668707, -6.534738170948296, -6.23...","[-6.714254670625664, -6.609214162488096, -6.29...","[0.07566873594673192, 0.1444480583804083, 0.07..."
4,AdaBoostRegressor,"[-6.757835309138117, -6.757835309138117, -6.75...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.519742052211105, -6.441977939585068, -6.4...","[-6.556199874355519, -6.426703197552024, -6.35...","[0.1236751465512148, 0.06947847161976735, 0.04..."
5,XGBRegressor,"[-6.869757, -7.3350344, -7.3350344, -6.283718,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.2667146, -6.4167666, -6.743267, -5.838397...","[-7.129899, -6.3715043, -6.612886, -5.721328, ...","[0.15118496, 0.08329285, 0.1883414, 0.23126613..."
6,ExtraTreesRegressor,"[-5.7400000000000055, -7.3433333333333515, -7....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.866666666666673, -6.600720000000003, -5.7...","[-6.252933333333331, -6.478324000000002, -6.22...","[0.543848402079511, 0.35308404764871193, 0.529..."
7,LinearRegression,"[-6.559641621852421, -7.188901137123713, -7.18...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.958319247583283, -7.447539399315966, -6.3...","[-6.939430924138838, -7.736688311181543, -6.42...","[0.06509470117361979, 0.21291953480069084, 0.1..."
8,KNeighborsRegressor,"[-6.246666666666666, -7.343333333333334, -7.34...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.233333333333333, -5.706666666666666, -5.8...","[-6.040666666666667, -5.8740000000000006, -5.8...","[0.2485281115331264, 0.10335268635975492, 0.13..."
9,SVR,"[-6.765747668717584, -7.099864611951071, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5876480553139265, -6.257023329867319, -5....","[-6.567125375625929, -6.415430461752456, -5.88...","[0.02964191114537755, 0.08640704936142748, 0.0..."


In [64]:
df_extended_fp.to_csv('results/Fingerprints/Results_Extended_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Extended_fp_Caco2.csv')

In [65]:
#Fingerprinter fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Fingerprinter_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Fingerprinter_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_fingerprinter_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_fingerprinter_fp

X_train shape:  (1007, 1024)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1024)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015868 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1287
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 429
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2896,0.4202,0.5382,0.5481,0.7404,0.7265,0.2799,0.4220,0.5291,0.5508,0.7426,0.6973
DecisionTreeRegressor,0.4330,0.4836,0.6580,0.3243,0.6396,0.6284,0.3233,0.4299,0.5686,0.4812,0.7072,0.6607
RandomForestRegressor,0.3128,0.4227,0.5593,0.5119,0.7188,0.7041,0.2882,0.4172,0.5369,0.5374,0.7343,0.6872
GradientBoostingRegressor,0.3161,0.4459,0.5623,0.5067,0.7123,0.6912,0.2995,0.4466,0.5472,0.5194,0.7219,0.6654
AdaBoostRegressor,0.3993,0.5428,0.6319,0.3770,0.6251,0.6039,0.3972,0.5466,0.6303,0.3625,0.6115,0.5486
XGBRegressor,0.3218,0.4186,0.5672,0.4979,0.7186,0.7199,0.2648,0.3946,0.5146,0.5750,0.7621,0.7177
ExtraTreesRegressor,0.4191,0.4749,0.6474,0.3460,0.6491,0.6375,0.3056,0.4198,0.5528,0.5095,0.7223,0.6790
LinearRegression,0.5853,0.5304,0.7650,0.0867,0.5568,0.6355,0.4101,0.4996,0.6404,0.3418,0.6557,0.6409
KNeighborsRegressor,0.3932,0.4578,0.6271,0.3864,0.6716,0.6605,0.3877,0.4687,0.6226,0.3778,0.6636,0.6214
SVR,0.3059,0.4282,0.5530,0.5227,0.7236,0.7056,0.2976,0.4349,0.5455,0.5224,0.7242,0.6700


In [66]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.72783300254763, -7.219809565530734, -7.219...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.476342765764085, -6.4256921182469195, -5....","[-6.500856894314884, -6.494596390613668, -5.77...","[0.036205483171115466, 0.08034048925542495, 0...."
1,DecisionTreeRegressor,"[-7.77, -7.5, -7.5, -5.96, -6.055, -6.39, -6.6...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.542857142857143, -7.06, -5.85249999999999...","[-6.504126984126984, -7.087142857142856, -5.85...","[0.05569488198266256, 0.23350392209265566, 0.0..."
2,RandomForestRegressor,"[-6.4090466666666694, -7.4346, -7.4346, -5.869...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.560161499056501, -6.432827272727273, -5.8...","[-6.4985467501124585, -6.631176160173162, -5.8...","[0.04928033770881082, 0.10506119057016318, 0.0..."
3,GradientBoostingRegressor,"[-6.643612042714251, -7.049524079763018, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.522472779238392, -6.492185087685265, -5.9...","[-6.463450241100185, -6.439985446831614, -5.92...","[0.03628319296137664, 0.17927083277723413, 0.0..."
4,AdaBoostRegressor,"[-6.579021591597479, -6.715156527670256, -6.71...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.579021591597479, -6.545323660714273, -6.2...","[-6.569819985179616, -6.488360277020159, -6.30...","[0.05840597163241562, 0.06529261499025331, 0.0..."
5,XGBRegressor,"[-5.629067, -7.4805236, -7.4805236, -5.9950614...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.556187, -5.848614, -5.7633605, -5.786849,...","[-6.5486436, -6.3171415, -5.7367654, -5.775786...","[0.022041213, 0.32658377, 0.05267713, 0.080818..."
6,ExtraTreesRegressor,"[-7.769999999999989, -7.5, -7.5, -5.9791999999...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.542857142857132, -6.055071931039992, -5.8...","[-6.504126984126975, -6.744854386207995, -5.85...","[0.05569488198266363, 0.48251287533536097, 0.0..."
7,LinearRegression,"[-6.436923842960775, -7.200373112948846, -7.20...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.461704759245489, -6.782966298390155, -5.7...","[-6.392865335595297, -7.2748237285308885, -5.7...","[0.0366395187703881, 0.27932627699587154, 0.04..."
8,KNeighborsRegressor,"[-6.213333333333334, -7.343333333333334, -7.34...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.093333333333334, -5.775000000000001, -5.8...","[-6.265333333333333, -5.974666666666667, -5.77...","[0.31465395454548295, 0.14283012442914247, 0.0..."
9,SVR,"[-6.718951806594065, -7.0982577911960725, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.513964499049653, -6.320675382650534, -5.7...","[-6.466100109621484, -6.486472753313441, -5.76...","[0.029663489049051788, 0.09177955361117862, 0...."


In [67]:
df_fingerprinter_fp.to_csv('results/Fingerprints/Results_Fingerprinter_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Fingerprinter_fp_Caco2.csv')

In [68]:
#GraphOnly fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Graphonly_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Graphonly_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_graph_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_graph_fp

X_train shape:  (1007, 1024)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1024)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008260 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 202
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3409,0.4644,0.5838,0.4681,0.6848,0.6573,0.3086,0.4482,0.5555,0.5047,0.7107,0.6608
DecisionTreeRegressor,0.4517,0.5019,0.6721,0.2952,0.6176,0.6008,0.3111,0.4271,0.5578,0.5007,0.7163,0.6580
RandomForestRegressor,0.3443,0.4521,0.5868,0.4627,0.6861,0.6544,0.2891,0.4177,0.5377,0.5360,0.7328,0.6817
GradientBoostingRegressor,0.3457,0.4709,0.5880,0.4606,0.6788,0.6467,0.3155,0.4692,0.5617,0.4937,0.7041,0.6448
AdaBoostRegressor,0.4213,0.5570,0.6491,0.3426,0.5911,0.5429,0.4009,0.5503,0.6332,0.3566,0.6091,0.5651
XGBRegressor,0.3603,0.4618,0.6003,0.4378,0.6799,0.6564,0.2909,0.4182,0.5393,0.5332,0.7352,0.6857
ExtraTreesRegressor,0.4278,0.4909,0.6541,0.3324,0.6320,0.6141,0.3113,0.4272,0.5580,0.5004,0.7155,0.6593
LinearRegression,0.4325,0.5066,0.6576,0.3252,0.6107,0.6269,0.3735,0.4800,0.6111,0.4006,0.6643,0.6539
KNeighborsRegressor,0.4203,0.4884,0.6483,0.3442,0.6325,0.6038,0.4087,0.4842,0.6393,0.3441,0.6397,0.5990
SVR,0.3524,0.4629,0.5937,0.4501,0.6731,0.6387,0.3344,0.4617,0.5783,0.4633,0.6830,0.6163


In [69]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.850068625693584, -7.137041417907131, -7.13...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.474481354713469, -6.274369021272716, -5.9...","[-6.4723627648309545, -6.391141454066542, -5.9...","[0.02175253274541283, 0.14230499000653762, 0.0..."
1,DecisionTreeRegressor,"[-6.943636363636363, -7.343333333333334, -7.34...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.515789473684211, -7.06, -6.10269230769230...","[-6.479512709505843, -6.029999999999999, -5.99...","[0.029167923742578136, 0.5799310303820618, 0.0..."
2,RandomForestRegressor,"[-6.781241260491319, -7.327331666666667, -7.32...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.50381600933777, -6.629134642857146, -6.14...","[-6.469618288083808, -6.579672174603178, -5.99...","[0.027175533004664666, 0.08726751665955255, 0...."
3,GradientBoostingRegressor,"[-6.918391194157251, -7.127257084408424, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.461051841751047, -6.0141151650436, -5.967...","[-6.4852705427715405, -6.295415943091806, -5.9...","[0.018679087857426953, 0.16692206580614552, 0...."
4,AdaBoostRegressor,"[-6.640567376073861, -6.693491656059168, -6.69...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.537271668824857, -6.354796225778048, -6.4...","[-6.4863812318180765, -6.368953709396588, -6.3...","[0.057140864987918, 0.06367827547169182, 0.057..."
5,XGBRegressor,"[-6.8175488, -7.3406186, -7.3406186, -6.0361, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.4841266, -6.7296457, -5.986416, -5.838810...","[-6.4686394, -6.5751677, -5.9307117, -5.939645...","[0.025495762, 0.28401208, 0.05120725, 0.187236..."
6,ExtraTreesRegressor,"[-6.943636363636374, -7.3433333333333515, -7.3...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.515789473684209, -7.0505999999999975, -6....","[-6.479512709505839, -6.5061457142857195, -5.9...","[0.029167923742579874, 0.4489472687003083, 0.0..."
7,LinearRegression,"[-6.823381336108451, -7.120357346626479, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.44645564040744, -6.7492140265863085, -5.9...","[-6.4410962887961585, -6.954341484828257, -5.9...","[0.02105277413720497, 0.2100939421570873, 0.03..."
8,KNeighborsRegressor,"[-7.103333333333334, -7.343333333333334, -7.34...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -5.706666666666666, -6.21, -5.87, -5....","[-6.292, -5.941000000000001, -6.40833333333333...","[0.19938349424608287, 0.11845580516706603, 0.2..."
9,SVR,"[-6.950755191396464, -7.100033997551495, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.539044019556263, -6.145098981960024, -5.8...","[-6.559955146567018, -6.270258595582398, -5.86...","[0.02566431072367601, 0.08521476165865213, 0.0..."


In [70]:
df_graph_fp.to_csv('results/Fingerprints/Results_Graphonly_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Graphonly_fp_Caco2.csv')

In [71]:
#KlekotaRoth fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRoth_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRoth_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRoth_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRoth_fp

X_train shape:  (1007, 4860)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 4860)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009006 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 576
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 192
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2455,0.3892,0.4954,0.6170,0.7858,0.7700,0.2372,0.3729,0.4871,0.6193,0.7873,0.7338
DecisionTreeRegressor,0.3988,0.4656,0.6315,0.3777,0.6825,0.6666,0.2217,0.3441,0.4708,0.6442,0.8084,0.7825
RandomForestRegressor,0.2405,0.3779,0.4904,0.6247,0.7905,0.7791,0.1949,0.3439,0.4415,0.6872,0.8301,0.7908
GradientBoostingRegressor,0.2622,0.4093,0.5121,0.5909,0.7707,0.7575,0.2411,0.3908,0.4910,0.6131,0.7850,0.7333
AdaBoostRegressor,0.3724,0.5277,0.6103,0.4189,0.6616,0.6192,0.3591,0.5140,0.5992,0.4237,0.6665,0.5752
XGBRegressor,0.2437,0.3778,0.4937,0.6197,0.7908,0.7779,0.1868,0.3310,0.4322,0.7002,0.8370,0.7945
ExtraTreesRegressor,0.3656,0.4487,0.6047,0.4295,0.7019,0.6820,0.2186,0.3440,0.4676,0.6491,0.8103,0.7848
LinearRegression,0.4310,0.4773,0.6565,0.3275,0.6514,0.6951,0.3243,0.4182,0.5695,0.4795,0.7238,0.7126
KNeighborsRegressor,0.2813,0.3956,0.5304,0.5611,0.7549,0.7409,0.2468,0.3757,0.4968,0.6039,0.7824,0.7478
SVR,0.2649,0.4000,0.5147,0.5866,0.7668,0.7606,0.2565,0.3856,0.5065,0.5883,0.7691,0.7470


In [72]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.419532057026409, -7.106278208854648, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.856263171707779, -6.058664956082965, -5.8...","[-5.959514448086172, -6.088154192240038, -5.86...","[0.09102007849154034, 0.05424376041742775, 0.1..."
1,DecisionTreeRegressor,"[-6.59, -7.0, -7.05, -6.89, -6.28, -6.0, -7.15...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.76, -6.64, -6.3625, -5.85, -6.27, -6.89, ...","[-5.784000000000001, -6.5040000000000004, -6.0...","[0.16119553343687923, 0.28210636292008706, 0.2..."
2,RandomForestRegressor,"[-6.4711333333333325, -7.208901666666663, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.013593333333329, -6.199099999999996, -5.9...","[-6.036479333333329, -6.2500015145795205, -5.9...","[0.052510110620082595, 0.04982099178667896, 0...."
3,GradientBoostingRegressor,"[-6.513955674310614, -6.904692965666, -6.99993...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.211300488599819, -6.11966917190846, -6.01...","[-6.215998916685288, -6.089180402172512, -5.99...","[0.03448790735497056, 0.04259396951492425, 0.0..."
4,AdaBoostRegressor,"[-6.3997454655166965, -6.657543553, -6.6575435...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.336687369041451, -6.307261716919664, -6.3...","[-6.287280413457683, -6.257620881984437, -6.30...","[0.03654384333866176, 0.04345579110195821, 0.0..."
5,XGBRegressor,"[-6.1214395, -7.1127567, -7.115187, -6.262302,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.792931, -6.090113, -6.089814, -5.919704, ...","[-5.9898896, -6.080966, -5.9319544, -5.8534584...","[0.11065774, 0.059059255, 0.095012866, 0.04629..."
6,ExtraTreesRegressor,"[-6.370499999999997, -7.123599999999997, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.7599999999999945, -6.63999999999999, -6.3...","[-5.827279999999996, -6.424219999999994, -6.00...","[0.13544555216027174, 0.2575622518926194, 0.26..."
7,LinearRegression,"[-6.697263184787081, -6.786582064847822, -7.17...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.210954990195391, -5.412576079842955, -6.0...","[-6.218022434230563, -5.4802835256184625, -6.0...","[0.029583132768257192, 0.12895400699261161, 0...."
8,KNeighborsRegressor,"[-6.146666666666666, -7.343333333333334, -7.35...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.053333333333334, -5.618333333333332, -5.9...","[-6.362666666666668, -5.870333333333335, -6.04...","[0.15475284955193713, 0.1535664894000865, 0.08..."
9,SVR,"[-6.662841418130758, -7.1054363230340005, -7.1...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.237479851610802, -5.906789445191533, -5.8...","[-6.317057433162506, -6.00990070141217, -5.871...","[0.04447081736659046, 0.06547909069018806, 0.0..."


In [73]:
df_KlekotaRoth_fp.to_csv('results/Fingerprints/Results_KlekotaRoth_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_KlekotaRoth_fp_Caco2.csv')

In [74]:
#KlekotaRoth Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRothCount_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRothCount_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRothCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRothCount_fp

X_train shape:  (1007, 4860)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 4860)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007799 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2356
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 257
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2008,0.3440,0.4481,0.6866,0.8287,0.8134,0.1896,0.3273,0.4354,0.6958,0.8342,0.8071
DecisionTreeRegressor,0.3383,0.4271,0.5817,0.4721,0.7311,0.7074,0.2057,0.3411,0.4535,0.6699,0.8226,0.7983
RandomForestRegressor,0.1960,0.3395,0.4427,0.6942,0.8341,0.8218,0.1767,0.3184,0.4204,0.7164,0.8469,0.8232
GradientBoostingRegressor,0.2128,0.3650,0.4613,0.6680,0.8196,0.8045,0.1955,0.3416,0.4421,0.6863,0.8305,0.7986
AdaBoostRegressor,0.3143,0.4743,0.5606,0.5096,0.7247,0.6749,0.2862,0.4505,0.5350,0.5406,0.7510,0.6880
XGBRegressor,0.2044,0.3420,0.4521,0.6810,0.8269,0.8185,0.1650,0.3084,0.4062,0.7352,0.8578,0.8351
ExtraTreesRegressor,0.2025,0.3381,0.4500,0.6840,0.8277,0.8225,0.1650,0.3018,0.4062,0.7352,0.8576,0.8360
LinearRegression,0.3036,0.4070,0.5510,0.5263,0.7559,0.7879,0.3104,0.4024,0.5572,0.5018,0.7454,0.7359
KNeighborsRegressor,0.2408,0.3643,0.4907,0.6243,0.7934,0.7815,0.2015,0.3341,0.4489,0.6766,0.8247,0.7922
SVR,0.2269,0.3618,0.4763,0.6460,0.8045,0.7912,0.2009,0.3333,0.4482,0.6776,0.8239,0.7925


In [75]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.809143735385342, -7.323023831028456, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.239615769497015, -6.446145870835306, -5.8...","[-6.374005638800446, -6.483910157255909, -5.89...","[0.19061572432352603, 0.06747225975353463, 0.0..."
1,DecisionTreeRegressor,"[-5.64, -8.0, -6.89, -8.0, -6.11, -5.96, -6.89...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -6.515000000000001, -6.82, -5.85, -6....","[-6.048, -6.265000000000001, -6.65600000000000...","[0.57052256747652, 0.6606966020799562, 0.56740..."
2,RandomForestRegressor,"[-6.427774999999998, -7.387699999999998, -6.99...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.1444499999999955, -6.421366666666668, -6....","[-6.345095333333332, -6.410336821304665, -6.04...","[0.15207551528179256, 0.06404520231320103, 0.0..."
3,GradientBoostingRegressor,"[-6.842735762070924, -7.359761226322191, -7.14...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.573917385238288, -6.29397966971387, -6.15...","[-6.607360761894698, -6.261139437245423, -6.16...","[0.16633472538694613, 0.12621393593118768, 0.0..."
4,AdaBoostRegressor,"[-6.336060606060606, -7.009741298682347, -6.26...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.267846405915011, -6.220875933933989, -6.2...","[-6.399568984318607, -6.34709396031609, -6.248...","[0.148522361652122, 0.21546836076137663, 0.048..."
5,XGBRegressor,"[-6.675882, -7.7024045, -7.066247, -6.3847404,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.16422, -6.1245956, -6.034928, -5.844535, ...","[-6.431615, -6.4112306, -6.0114927, -5.875106,...","[0.37156498, 0.24861959, 0.14009765, 0.0595162..."
6,ExtraTreesRegressor,"[-6.495649999999999, -7.800799999999998, -6.95...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.071699999999996, -6.15685, -6.21839999999...","[-6.441942113296003, -6.26175, -6.239343820025...","[0.23252035261896623, 0.05364169087566113, 0.0..."
7,LinearRegression,"[-6.432799340523531, -7.261183174516079, -6.96...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.999295581660787, -5.817644352028209, -5.4...","[-6.002824773387756, -5.908086531085987, -5.47...","[0.047666531844845234, 0.11297790190918887, 0...."
8,KNeighborsRegressor,"[-6.0633333333333335, -7.343333333333334, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -5.838333333333334, -5.92833333333333...","[-6.544, -5.989333333333333, -6.03466666666666...","[0.22580620993330663, 0.08546344247688559, 0.1..."
9,SVR,"[-6.324431087269995, -7.167538159733818, -7.14...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.073848711797817, -6.141773240679906, -5.7...","[-6.132804957806524, -6.139463589092181, -5.78...","[0.0683522599089858, 0.01877319246088452, 0.00..."


In [76]:
df_KlekotaRothCount_fp.to_csv('results/Fingerprints/Results_KlekotaRoth_Count_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_KlekotaRoth_Count_fp_Caco2.csv')

In [77]:
#MACCS fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/MACCS_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/MACCS_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_MACCS_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_MACCS_fp

X_train shape:  (1007, 166)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 166)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017741 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 153
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 51
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3255,0.4516,0.5705,0.4921,0.7030,0.6688,0.2768,0.4265,0.5261,0.5557,0.7456,0.6987
DecisionTreeRegressor,0.3606,0.4641,0.6005,0.4373,0.6827,0.6567,0.2604,0.3908,0.5103,0.5821,0.7695,0.7298
RandomForestRegressor,0.3100,0.4365,0.5568,0.5163,0.7215,0.6880,0.2559,0.3948,0.5058,0.5894,0.7704,0.7255
GradientBoostingRegressor,0.3263,0.4648,0.5712,0.4908,0.7011,0.6665,0.2837,0.4453,0.5326,0.5448,0.7410,0.6945
AdaBoostRegressor,0.4005,0.5420,0.6328,0.3751,0.6161,0.5606,0.3737,0.5297,0.6113,0.4003,0.6418,0.5870
XGBRegressor,0.3171,0.4368,0.5632,0.5051,0.7181,0.6922,0.2581,0.4049,0.5080,0.5859,0.7683,0.7258
ExtraTreesRegressor,0.3461,0.4552,0.5883,0.4600,0.6947,0.6664,0.2605,0.3937,0.5104,0.5820,0.7686,0.7256
LinearRegression,0.3423,0.4786,0.5851,0.4659,0.6858,0.6505,0.3436,0.4600,0.5862,0.4486,0.6823,0.6661
KNeighborsRegressor,0.3754,0.4659,0.6127,0.4142,0.6775,0.6420,0.3802,0.4675,0.6166,0.3899,0.6736,0.6117
SVR,0.3106,0.4344,0.5573,0.5153,0.7191,0.6891,0.2516,0.3992,0.5016,0.5962,0.7727,0.7230


In [78]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.750000135208131, -7.0651066882980835, -6.6...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.221113954002175, -6.202038585288027, -6.1...","[-6.159182159756362, -6.285502558936039, -6.24...","[0.05751490830961122, 0.12733485053315838, 0.0..."
1,DecisionTreeRegressor,"[-6.7345945945945935, -7.515000000000001, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.218958333333332, -6.445, -6.1640909090909...","[-6.156535186533526, -6.352878571428571, -6.31...","[0.05863638121807372, 0.27235258472825874, 0.0..."
2,RandomForestRegressor,"[-6.74600451556483, -7.365404999999995, -7.025...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.218763416736359, -6.297468907856848, -6.1...","[-6.14798064685578, -6.328466361324949, -6.316...","[0.05370309072750065, 0.11067235675359473, 0.0..."
3,GradientBoostingRegressor,"[-6.641712518585876, -7.350387323228987, -6.92...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.43527239403956, -6.138453562986402, -6.12...","[-6.402073282402138, -6.1668080075636995, -6.1...","[0.04573341504954011, 0.03781958873614043, 0.0..."
4,AdaBoostRegressor,"[-6.585639810426545, -6.585639810426545, -6.39...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.392568493150688, -6.087679008294644, -6.3...","[-6.4166099175677545, -6.181285913222319, -6.3...","[0.026947995723450687, 0.048286982939993686, 0..."
5,XGBRegressor,"[-6.7252574, -7.513754, -7.0258512, -6.4258237...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.2383847, -6.3243136, -6.1426373, -5.84262...","[-6.188508, -6.3834662, -6.258603, -5.8432326,...","[0.048622247, 0.16072996, 0.07814822, 0.022589..."
6,ExtraTreesRegressor,"[-6.73459459459458, -7.51499999999999, -7.0249...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.218958333333335, -6.445000000000006, -6.1...","[-6.156535186533529, -6.352878571428571, -6.31...","[0.05863638121807515, 0.272352584728261, 0.099..."
7,LinearRegression,"[-6.593296387002117, -7.29035222310595, -7.131...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.434829308169338, -6.166747596544746, -6.1...","[-6.413524776276423, -6.22647337599704, -6.178...","[0.026892412306069154, 0.041460433695353675, 0..."
8,KNeighborsRegressor,"[-6.553333333333334, -7.36, -7.350000000000000...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.023333333333333, -5.673333333333333, -6.8...","[-6.046666666666667, -5.764, -7.19066666666666...","[0.09028350409189455, 0.19149006356582735, 0.2..."
9,SVR,"[-6.684715321838262, -7.130254320561682, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.410042087113681, -6.097293786725045, -6.0...","[-6.427803458539604, -6.133112238059027, -6.03...","[0.03587200732542298, 0.0404993507648744, 0.02..."


In [79]:
df_MACCS_fp.to_csv('results/Fingerprints/Results_MACCS_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_MACCS_fp_Caco2.csv')

In [80]:
#PubChem fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/PubChem_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/PubChem_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_PubChem_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_PubChem_fp

X_train shape:  (1007, 881)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 881)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.136068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 591
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 197
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3280,0.4529,0.5727,0.4881,0.6990,0.6468,0.2996,0.4352,0.5473,0.5192,0.7206,0.6771
DecisionTreeRegressor,0.3398,0.4490,0.5829,0.4698,0.6999,0.6612,0.3063,0.4286,0.5534,0.5085,0.7192,0.6830
RandomForestRegressor,0.3111,0.4322,0.5577,0.5146,0.7202,0.6739,0.2937,0.4216,0.5419,0.5287,0.7282,0.6867
GradientBoostingRegressor,0.3249,0.4573,0.5700,0.4931,0.7030,0.6644,0.3051,0.4465,0.5523,0.5104,0.7152,0.6682
AdaBoostRegressor,0.4117,0.5473,0.6416,0.3576,0.6018,0.5445,0.3887,0.5373,0.6234,0.3763,0.6213,0.5595
XGBRegressor,0.3222,0.4411,0.5676,0.4972,0.7128,0.6729,0.2999,0.4261,0.5477,0.5186,0.7249,0.6878
ExtraTreesRegressor,0.3267,0.4428,0.5716,0.4902,0.7113,0.6718,0.3034,0.4238,0.5508,0.5131,0.7217,0.6845
LinearRegression,0.3580,0.4831,0.5983,0.4414,0.6728,0.6479,0.3832,0.4869,0.6190,0.3851,0.6435,0.6259
KNeighborsRegressor,0.3999,0.4794,0.6324,0.3760,0.6574,0.6081,0.4103,0.4774,0.6406,0.3415,0.6429,0.5914
SVR,0.3290,0.4501,0.5736,0.4867,0.6978,0.6630,0.2874,0.4220,0.5361,0.5387,0.7341,0.6754


In [81]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.931966388313026, -6.925460021757564, -6.92...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.931966388313026, -5.929133589811104, -6.5...","[-6.944966644053361, -5.953537839711154, -6.70...","[0.04220539305975013, 0.0904423675576597, 0.11..."
1,DecisionTreeRegressor,"[-6.924285714285714, -7.025, -7.025, -5.96, -5...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.924285714285714, -5.96, -7.335, -5.85, -5...","[-6.905107142857142, -6.518761904761905, -7.35...","[0.0447954598140779, 0.4168252324133188, 0.109..."
2,RandomForestRegressor,"[-6.880636362803862, -7.017704285714279, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.8806363628038625, -5.983072696933238, -7....","[-6.892530448280479, -6.111896139805429, -7.30...","[0.05046192520163794, 0.10113707224294811, 0.1..."
3,GradientBoostingRegressor,"[-6.810058774896798, -6.897282176278333, -6.89...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.810058774896798, -5.981625867813132, -7.0...","[-6.822111669594174, -6.269241897182832, -7.13...","[0.03569190821613137, 0.1653116374628984, 0.10..."
4,AdaBoostRegressor,"[-6.587262291015223, -6.519716337427498, -6.51...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.587262291015223, -6.2943137254902, -6.350...","[-6.537664609685531, -6.2589874690009655, -6.4...","[0.045698361418806656, 0.05532132884202234, 0...."
5,XGBRegressor,"[-6.920826, -7.0274277, -7.0274277, -6.6187825...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.920826, -6.121579, -7.3332286, -5.8426685...","[-6.9082613, -6.2927475, -7.346595, -5.7700715...","[0.04809092, 0.10603608, 0.102035366, 0.108277..."
6,ExtraTreesRegressor,"[-6.9242857142857055, -7.024999999999988, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.9242857142857055, -5.959999999999997, -7....","[-6.905107142857146, -6.28597714285714, -7.358...","[0.04479545981407668, 0.39196923396551386, 0.1..."
7,LinearRegression,"[-7.075190708861321, -7.095010188658147, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.075190708861321, -6.15838598076042, -6.67...","[-7.103777572730604, -6.565865723611762, -6.73...","[0.05107798988840661, 0.27513116450751196, 0.0..."
8,KNeighborsRegressor,"[-6.739999999999999, -7.3500000000000005, -7.3...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.739999999999999, -5.918333333333333, -6.9...","[-6.877333333333333, -6.022666666666668, -7.21...","[0.10771154895264384, 0.05460565701260105, 0.1..."
9,SVR,"[-6.985440257175489, -7.023590131764155, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.985440257175489, -5.979725958989712, -6.7...","[-7.021658632428784, -6.092860287310588, -6.81...","[0.06607710233632581, 0.09285908787278215, 0.0..."


In [82]:
df_PubChem_fp.to_csv('results/Fingerprints/Results_PubChem_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_PubChem_fp_Caco2.csv')

In [83]:
#Substructure fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Substructure_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Substructure_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_Substructure_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_Substructure_fp

X_train shape:  (1007, 307)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 307)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000955 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 45
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 15
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.2708590355076985


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3939,0.5069,0.6276,0.3854,0.6211,0.5658,0.3767,0.5093,0.6138,0.3954,0.6288,0.5676
DecisionTreeRegressor,0.3735,0.4890,0.6111,0.4172,0.6500,0.5899,0.3685,0.4909,0.6070,0.4086,0.6435,0.6022
RandomForestRegressor,0.3670,0.4838,0.6058,0.4274,0.6555,0.5949,0.3623,0.4863,0.6020,0.4185,0.6488,0.6057
GradientBoostingRegressor,0.3839,0.5055,0.6196,0.4009,0.6340,0.5761,0.3896,0.5202,0.6242,0.3747,0.6126,0.5798
AdaBoostRegressor,0.4585,0.5774,0.6772,0.2845,0.5357,0.5173,0.4426,0.5719,0.6652,0.2898,0.5414,0.5423
XGBRegressor,0.3654,0.4846,0.6045,0.4298,0.6582,0.6019,0.3652,0.4913,0.6043,0.4139,0.6466,0.6038
ExtraTreesRegressor,0.3714,0.4870,0.6094,0.4205,0.6521,0.5920,0.3676,0.4914,0.6063,0.4100,0.6444,0.6052
LinearRegression,0.4327,0.5457,0.6578,0.3249,0.5724,0.5345,0.4849,0.5785,0.6964,0.2218,0.4902,0.4976
KNeighborsRegressor,0.4135,0.5162,0.6430,0.3548,0.6046,0.5823,0.3814,0.5100,0.6176,0.3879,0.6309,0.5979
SVR,0.3816,0.4801,0.6177,0.4046,0.6386,0.5808,0.3653,0.4877,0.6044,0.4138,0.6442,0.5946


In [84]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.692874716131753, -6.3485320708423965, -6.3...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.692874716131753, -5.837236050792211, -6.1...","[-6.694120704203551, -5.941471176347003, -6.14...","[0.013651798646830545, 0.08603893012310962, 0...."
1,DecisionTreeRegressor,"[-6.381428571428572, -7.2700000000000005, -7.2...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.381428571428572, -5.840625, -6.1775217391...","[-6.399738095238095, -5.910858333333333, -6.18...","[0.13933865802897322, 0.08254867269797604, 0.0..."
2,RandomForestRegressor,"[-6.391281672077922, -7.232022460317458, -7.23...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.39128167207792, -5.8072215732622645, -6.1...","[-6.388763075042003, -5.846997060086242, -6.18...","[0.13900573481088618, 0.049730099506466875, 0...."
3,GradientBoostingRegressor,"[-6.497104718349936, -7.106172820000674, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.497104718349936, -5.853724074836629, -6.1...","[-6.495855394497658, -5.835609444439538, -6.18...","[0.09574124014212634, 0.06800073005997398, 0.0..."
4,AdaBoostRegressor,"[-6.554461084840983, -6.585377327016184, -6.58...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.554461084840983, -6.316133589656567, -6.4...","[-6.514615877025652, -6.360083862602369, -6.40...","[0.02730392884990231, 0.030388733564473015, 0...."
5,XGBRegressor,"[-6.3856072, -7.2665005, -7.2665005, -6.37794,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.3856072, -5.6421, -6.1781406, -5.8821273,...","[-6.410409, -5.6972756, -6.183349, -6.0559244,...","[0.13717711, 0.11471994, 0.021171514, 0.375496..."
6,ExtraTreesRegressor,"[-6.381428571428571, -7.26999999999999, -7.269...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.381428571428573, -5.840624999999999, -6.1...","[-6.3997380952381, -5.885474333333334, -6.1820...","[0.13933865802897213, 0.03479702652331052, 0.0..."
7,LinearRegression,"[-6.42146619511297, -7.297785085018243, -7.297...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.42146619511297, -5.683040034374925, -6.17...","[-6.406062500466845, -5.789324152454872, -6.19...","[0.09149756413587784, 0.06692230521153594, 0.0..."
8,KNeighborsRegressor,"[-6.28, -7.36, -7.36, -7.036666666666666, -5.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -5.918333333333334, -6.41333333333333...","[-6.544, -6.040000000000001, -6.34133333333333...","[0.22580620993330663, 0.07486283753935895, 0.0..."
9,SVR,"[-6.060337535383792, -7.100440512100118, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.060337535383792, -5.7971883342469575, -6....","[-6.097858587172648, -5.832925964670624, -6.00...","[0.18225689376031903, 0.05939843248979665, 0.0..."


In [85]:
df_Substructure_fp.to_csv('results/Fingerprints/Results_Substructure_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Substructure_fp_Caco2.csv')

In [86]:
#Substructure Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/SubstructureCount_train_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/SubstructureCount_test_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_SubstructureCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_SubstructureCount_fp

X_train shape:  (1007, 307)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 307)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001097 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 26
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.4331533770712983


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2221,0.3631,0.4712,0.6535,0.8085,0.7935,0.1806,0.3252,0.4250,0.7102,0.8437,0.8254
DecisionTreeRegressor,0.3528,0.4380,0.5940,0.4495,0.7207,0.7051,0.2095,0.3432,0.4577,0.6637,0.8200,0.7976
RandomForestRegressor,0.2165,0.3549,0.4653,0.6622,0.8140,0.7982,0.1605,0.3059,0.4006,0.7425,0.8631,0.8488
GradientBoostingRegressor,0.2336,0.3833,0.4833,0.6355,0.7994,0.7855,0.2050,0.3594,0.4527,0.6710,0.8241,0.7883
AdaBoostRegressor,0.3452,0.4997,0.5875,0.4614,0.6877,0.6599,0.3224,0.4790,0.5678,0.4827,0.7090,0.6595
XGBRegressor,0.2148,0.3498,0.4635,0.6648,0.8192,0.7984,0.1588,0.2984,0.3985,0.7451,0.8638,0.8499
ExtraTreesRegressor,0.2168,0.3464,0.4656,0.6617,0.8164,0.7980,0.1696,0.3089,0.4118,0.7278,0.8541,0.8386
LinearRegression,0.2840,0.4323,0.5330,0.5568,0.7471,0.7299,0.3151,0.4490,0.5614,0.4942,0.7072,0.6606
KNeighborsRegressor,0.2623,0.3877,0.5122,0.5906,0.7752,0.7464,0.2159,0.3449,0.4646,0.6535,0.8125,0.8002
SVR,0.2912,0.4241,0.5397,0.5456,0.7402,0.7215,0.2440,0.3858,0.4940,0.6084,0.7872,0.7589


In [87]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.970845858047859, -7.084584746819027, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.8710667878522305, -5.67693681981292, -6.5...","[-6.9497191779702545, -6.156380394216466, -6.5...","[0.07027390482979086, 0.24382149243407014, 0.0..."
1,DecisionTreeRegressor,"[-7.24, -6.89, -6.96, -5.68, -5.89, -6.74, -6....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -5.835, -5.96, -5.85, -5.88, -6.92, -...","[-7.210000000000001, -6.24, -6.366000000000000...","[0.024494897427831695, 0.4273055113147969, 0.4..."
2,RandomForestRegressor,"[-6.794333333333335, -7.162699999999993, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.7177999999999995, -6.213900000000001, -6....","[-6.869410000000004, -6.396530000000001, -6.41...","[0.164850284804124, 0.12626524700011443, 0.130..."
3,GradientBoostingRegressor,"[-6.9191799194761225, -7.1604000352775445, -7....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.7483278988449324, -5.89044750553843, -6.5...","[-6.933931398305299, -6.203932773170977, -6.74...","[0.11438673817101042, 0.1829692108183291, 0.16..."
4,AdaBoostRegressor,"[-6.566410256410254, -7.0290625, -6.3239502532...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.566410256410254, -6.178222695683143, -6.3...","[-6.661085786103227, -6.221846033994851, -6.42...","[0.20518380950046566, 0.035636831583667886, 0...."
5,XGBRegressor,"[-7.2925024, -7.272503, -7.1747284, -6.4104953...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.2296557, -6.0982637, -6.3125625, -5.86047...","[-7.0888815, -6.18637, -6.4072213, -5.9842386,...","[0.21773534, 0.1271046, 0.12962312, 0.25158098..."
6,ExtraTreesRegressor,"[-7.114450000000005, -7.395499999999996, -6.99...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.192000000000006, -6.225550000000001, -6.1...","[-7.145820000000008, -6.464250000000002, -6.26...","[0.14304778781931565, 0.1434549790003818, 0.08..."
7,LinearRegression,"[-6.263837575239716, -7.247735720203182, -7.26...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.079493124270648, -5.506095017857442, -5.4...","[-6.105010106876998, -5.675006013659806, -5.48...","[0.14336751800045114, 0.09285430068493403, 0.0..."
8,KNeighborsRegressor,"[-6.646666666666666, -7.3500000000000005, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.646666666666666, -5.838333333333334, -5.8...","[-6.764, -5.976, -5.778666666666667, -6.186666...","[0.2010682581723044, 0.0705344675397147, 0.099..."
9,SVR,"[-6.358448003244602, -7.1059943124014575, -7.1...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.332325065800245, -6.204897986005287, -5.8...","[-6.552064336282919, -6.194638284698682, -5.79...","[0.27976953830012535, 0.015163683211721308, 0...."


In [88]:
df_SubstructureCount_fp.to_csv('results/Fingerprints/Results_Substructure_Count_fp_Caco2.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Substructure_Count_fp_Caco2.csv')

In [89]:
#Descriptors models
#2d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2drdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2drdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 217)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 217)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18108
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 146
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1830,0.3251,0.4278,0.7145,0.8455,0.8327,0.1336,0.2767,0.3655,0.7856,0.8870,0.8732
DecisionTreeRegressor,0.3377,0.4220,0.5811,0.4731,0.7296,0.7142,0.1876,0.3213,0.4332,0.6989,0.8389,0.8055
RandomForestRegressor,0.1988,0.3370,0.4458,0.6899,0.8311,0.8211,0.1442,0.2838,0.3797,0.7686,0.8782,0.8650
GradientBoostingRegressor,0.2027,0.3489,0.4502,0.6837,0.8275,0.8131,0.1464,0.2946,0.3826,0.7651,0.8765,0.8660
AdaBoostRegressor,0.2744,0.4360,0.5238,0.5719,0.7636,0.7329,0.2488,0.4089,0.4988,0.6006,0.7848,0.7332
XGBRegressor,0.2015,0.3343,0.4488,0.6857,0.8295,0.8176,0.1374,0.2766,0.3707,0.7795,0.8832,0.8690
ExtraTreesRegressor,0.1876,0.3240,0.4331,0.7073,0.8411,0.8317,0.1347,0.2795,0.3670,0.7838,0.8858,0.8765
LinearRegression,0.3199,0.4064,0.5656,0.5008,0.7340,0.7761,0.3663,0.4189,0.6052,0.4122,0.7027,0.7388
KNeighborsRegressor,0.2333,0.3573,0.4830,0.6360,0.8020,0.7806,0.1766,0.3177,0.4202,0.7166,0.8489,0.8295
SVR,0.2185,0.3591,0.4674,0.6591,0.8128,0.8010,0.1669,0.3123,0.4085,0.7322,0.8566,0.8365


In [90]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.7907689956964825, -7.072598161458679, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.84992459573381, -6.657905033744636, -6.61...","[-6.965424115744827, -6.954985702383442, -6.71...","[0.06511400574161183, 0.1831269089984422, 0.15..."
1,DecisionTreeRegressor,"[-7.24, -6.89, -8.0, -5.89, -5.68, -6.89, -5.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.64, -5.92, -6.82, -5.85, -5.85, -7.0, -6....","[-6.656000000000001, -6.792, -6.79000000000000...","[0.6286048043087169, 0.7492769848327119, 0.307..."
2,RandomForestRegressor,"[-6.691038719640001, -6.948149019599994, -6.93...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.7444887196400005, -6.58025818754, -6.6491...","[-6.773724716444002, -6.907326247138002, -6.75...","[0.12199218635450354, 0.1715125877867142, 0.08..."
3,GradientBoostingRegressor,"[-6.871235769753097, -6.758525734608978, -6.70...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.74705838157235, -6.6951264963498, -6.4929...","[-6.908992730642066, -7.053049701184662, -6.64...","[0.15433253757856819, 0.23100329480921167, 0.1..."
4,AdaBoostRegressor,"[-6.645274763863355, -7.11582417582417, -6.987...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.416674613612201, -6.831323270004748, -6.1...","[-6.6039544754179405, -6.679877621881685, -6.2...","[0.1617013417381991, 0.23909046780078624, 0.10..."
5,XGBRegressor,"[-6.56189, -6.928924, -6.935517, -6.5209293, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.833811, -6.4923477, -6.860438, -5.8499956...","[-6.8989425, -6.703523, -6.78486, -5.8582125, ...","[0.101790056, 0.2340725, 0.2557534, 0.01717799..."
6,ExtraTreesRegressor,"[-6.662935092750002, -7.121549999999997, -6.98...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6443, -6.234539700039999, -6.495989700040...","[-6.881140000000005, -6.472511760033998, -6.56...","[0.13279323175523913, 0.13477371532618196, 0.1..."
7,LinearRegression,"[-10.0, -7.121647640136901, -6.870258653649417...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -5.9235262413683625, -5.5279269868789...","[-7.199231996017202, -6.2056016855199925, -5.5...","[1.4005849238684682, 0.19863980318996638, 0.06..."
8,KNeighborsRegressor,"[-6.363333333333333, -7.3500000000000005, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.105, -6.316666666666667, -5.703333...","[-6.544, -6.328000000000001, -6.58933333333333...","[0.22580620993330663, 0.11151681487560522, 0.1..."
9,SVR,"[-6.334551871785265, -7.135390751220025, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.334456241120637, -6.417121338612503, -6.3...","[-6.899613072130135, -6.5080558359230665, -6.4...","[0.2825842806351981, 0.06583924459825034, 0.06..."


In [91]:
result_df.to_csv('results/Descriptors/Results_2d_RDKit_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_RDKit_desc_Caco2.csv')

In [100]:
#2d Mordred descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df , prediction_df= train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/2937008087.py:2: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
/tmp/ipykernel_1796054/2937008087.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 1429)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1429)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.021634 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 260021
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1170
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1932,0.3320,0.4395,0.6986,0.8360,0.8280,0.1291,0.2722,0.3593,0.7928,0.8920,0.8800
DecisionTreeRegressor,0.3846,0.4575,0.6201,0.4000,0.6952,0.6739,0.1650,0.3101,0.4063,0.7351,0.8574,0.8395
RandomForestRegressor,0.1993,0.3407,0.4465,0.6890,0.8318,0.8210,0.1441,0.2877,0.3796,0.7687,0.8810,0.8738
GradientBoostingRegressor,0.2090,0.3492,0.4572,0.6739,0.8211,0.8041,0.1476,0.2924,0.3842,0.7632,0.8766,0.8628
AdaBoostRegressor,0.2597,0.4168,0.5096,0.5948,0.7813,0.7532,0.2186,0.3754,0.4675,0.6492,0.8230,0.7874
XGBRegressor,0.2067,0.3405,0.4547,0.6774,0.8237,0.8126,0.1392,0.2806,0.3731,0.7766,0.8820,0.8750
ExtraTreesRegressor,0.1868,0.3213,0.4322,0.7085,0.8421,0.8356,0.1337,0.2757,0.3657,0.7854,0.8879,0.8782
LinearRegression,4.4331,1.6396,2.1055,-5.9172,0.2210,0.2526,1.6595,0.9922,1.2882,-1.6632,0.2103,0.3003
KNeighborsRegressor,0.2322,0.3604,0.4819,0.6377,0.8026,0.7878,0.1869,0.3124,0.4323,0.7000,0.8386,0.8088
SVR,0.2165,0.3498,0.4652,0.6623,0.8147,0.8036,0.1754,0.3192,0.4188,0.7185,0.8511,0.8382


In [101]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.6596891511849, -7.354533053242414, -7.1338...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.49661527890939, -6.813701147753388, -6.35...","[-6.700389696424854, -6.738942937572096, -6.57...","[0.19996621981070536, 0.1277283680615865, 0.14..."
1,DecisionTreeRegressor,"[-6.59, -6.66, -7.11, -7.22, -6.11, -6.06, -6....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.77, -7.6, -6.0, -5.85, -5.89, -6.66, -6.5...","[-6.182, -6.703999999999999, -6.72898039199999...","[0.49077082227858654, 0.978930028142972, 0.780..."
2,RandomForestRegressor,"[-6.534207282660001, -7.172127194429997, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.516424999999999, -6.457449999999999, -6.2...","[-6.699442281398001, -6.54424934033, -6.499592...","[0.17047406269560167, 0.10796232695122668, 0.1..."
3,GradientBoostingRegressor,"[-6.842190978527327, -7.202788121922744, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.432439778115576, -6.561760104994166, -6.4...","[-6.712612855789447, -6.559299760551118, -6.68...","[0.15112378821714129, 0.14736952753619023, 0.1..."
4,AdaBoostRegressor,"[-6.718705618223808, -7.2547040252075465, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.844082183678161, -6.416027397260271, -6.6...","[-6.604640194270667, -6.487956997160005, -6.40...","[0.25318961268748874, 0.12189820709692097, 0.1..."
5,XGBRegressor,"[-6.5181174, -7.6727934, -6.9720054, -7.021444...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.910313, -6.668077, -6.36401, -5.8499665, ...","[-6.630128, -6.520177, -6.496497, -5.8814535, ...","[0.45372728, 0.24646764, 0.12799953, 0.0507422..."
6,ExtraTreesRegressor,"[-6.723500000000001, -7.233199999999995, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.695399999999999, -6.376600000000002, -6.3...","[-6.869789607840003, -6.420507141468001, -6.43...","[0.18496866406682952, 0.04449737839791844, 0.1..."
7,LinearRegression,"[-7.244550242519364, -10.0, -10.0, -3.4, -5.70...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.443306911419211, -7.795852339473936, -3.4...","[-4.854276925721605, -5.760165286210843, -5.99...","[1.3747102823712136, 2.671314290284856, 1.6716..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -5.96166666666666...","[-6.544, -6.476000000000001, -6.05466666666666...","[0.22580620993330663, 0.13166286914354805, 0.0..."
9,SVR,"[-6.328296033227039, -7.157193184483718, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.161049704173652, -6.2953334308259405, -6....","[-6.421250798423857, -6.335298774882591, -6.25...","[0.2654536606007062, 0.05773624034636972, 0.09..."


In [102]:
result_df.to_csv('results/Descriptors/Results_2d_Mordred_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_Mordred_desc_Caco2.csv')

In [103]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [104]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [105]:
#2d RDKit descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 167)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 167)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004276 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18108
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 146
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1830,0.3251,0.4278,0.7145,0.8455,0.8327,0.1336,0.2767,0.3655,0.7856,0.8870,0.8732
DecisionTreeRegressor,0.3494,0.4333,0.5911,0.4548,0.7219,0.7045,0.1763,0.3106,0.4199,0.7170,0.8479,0.8137
RandomForestRegressor,0.1988,0.3374,0.4459,0.6898,0.8310,0.8202,0.1436,0.2829,0.3790,0.7695,0.8788,0.8651
GradientBoostingRegressor,0.2030,0.3487,0.4506,0.6832,0.8272,0.8132,0.1465,0.2944,0.3828,0.7649,0.8764,0.8670
AdaBoostRegressor,0.2782,0.4363,0.5275,0.5658,0.7586,0.7386,0.2470,0.4094,0.4970,0.6036,0.7892,0.7419
XGBRegressor,0.2015,0.3343,0.4488,0.6857,0.8295,0.8176,0.1374,0.2766,0.3707,0.7795,0.8832,0.8690
ExtraTreesRegressor,0.1859,0.3223,0.4311,0.7100,0.8427,0.8317,0.1350,0.2790,0.3674,0.7833,0.8857,0.8776
LinearRegression,0.3199,0.4064,0.5656,0.5008,0.7340,0.7761,0.3663,0.4189,0.6052,0.4122,0.7027,0.7388
KNeighborsRegressor,0.2340,0.3577,0.4837,0.6349,0.8013,0.7799,0.1768,0.3181,0.4205,0.7162,0.8486,0.8295
SVR,0.2185,0.3591,0.4674,0.6591,0.8128,0.8010,0.1669,0.3124,0.4086,0.7321,0.8565,0.8364


In [106]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.7907689956964825, -7.072598161458679, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.84992459573381, -6.657905033744636, -6.61...","[-6.965424115744827, -6.954985702383442, -6.71...","[0.06511400574161183, 0.1831269089984422, 0.15..."
1,DecisionTreeRegressor,"[-7.14, -7.05, -8.0, -5.8, -5.51, -6.89, -5.96...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -5.88, -6.89, -5.85, -5.96, -7.0, -6....","[-6.81, -6.802, -6.884, -5.764, -5.752, -7.266...","[0.4712536472007406, 0.8834115688624415, 0.138..."
2,RandomForestRegressor,"[-6.664989700040001, -6.990799999999994, -6.90...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.687400000000002, -6.567316666666666, -6.6...","[-6.763894487104002, -6.889281386539335, -6.74...","[0.13520286152205566, 0.1694822065847717, 0.08..."
3,GradientBoostingRegressor,"[-6.970184127557609, -6.758525734608978, -6.70...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.846006739376861, -6.879480017381737, -6.4...","[-6.91667343368731, -7.065124530097696, -6.643...","[0.14532275917767917, 0.18051800244869687, 0.1..."
4,AdaBoostRegressor,"[-6.6925, -7.20269544313924, -7.10799085263888...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.689439252336444, -6.673333333333333, -6.2...","[-6.695569414503032, -6.568210382152254, -6.26...","[0.08344849862750757, 0.26406709036188775, 0.0..."
5,XGBRegressor,"[-6.56189, -6.928924, -6.935517, -6.5209293, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.833811, -6.4923477, -6.860438, -5.8499956...","[-6.8989425, -6.703523, -6.78486, -5.8582125, ...","[0.101790056, 0.2340725, 0.2557534, 0.01717799..."
6,ExtraTreesRegressor,"[-6.734487739240002, -7.235899999999995, -7.08...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.788200000000002, -6.344949999999999, -6.4...","[-6.9029800000000066, -6.50406, -6.55646945498...","[0.09730610258355053, 0.0979796427835909, 0.08..."
7,LinearRegression,"[-10.0, -7.121647640135989, -6.870258653647624...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -5.923526241363061, -5.52792698687302...","[-7.199231996014855, -6.205601685507059, -5.56...","[1.4005849238691763, 0.1986398031903446, 0.062..."
8,KNeighborsRegressor,"[-6.363333333333333, -7.3500000000000005, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.105, -6.316666666666667, -5.703333...","[-6.544, -6.328000000000001, -6.58933333333333...","[0.22580620993330663, 0.11151681487560522, 0.1..."
9,SVR,"[-6.334551921065816, -7.135390370250723, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.334456291420712, -6.417122596640381, -6.3...","[-6.899776182695237, -6.5080032848414735, -6.4...","[0.28266640612229793, 0.06572998613440789, 0.0..."


In [107]:
result_df.to_csv('results/Descriptors/Results_2d_rdkit_const_rem_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_rdkit_const_rem_Caco2.csv')

In [108]:
#2d Mordred descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/1742967021.py:2: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')


X_train shape:  (1007, 1216)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1216)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_1796054/1742967021.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019522 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 260021
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1170
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1932,0.3320,0.4395,0.6986,0.8360,0.8280,0.1291,0.2722,0.3593,0.7928,0.8920,0.8800
DecisionTreeRegressor,0.3970,0.4582,0.6301,0.3805,0.6874,0.6667,0.1697,0.3139,0.4120,0.7276,0.8530,0.8385
RandomForestRegressor,0.2004,0.3417,0.4477,0.6873,0.8307,0.8195,0.1443,0.2888,0.3799,0.7684,0.8808,0.8738
GradientBoostingRegressor,0.2081,0.3492,0.4562,0.6753,0.8221,0.8052,0.1473,0.2917,0.3839,0.7635,0.8769,0.8644
AdaBoostRegressor,0.2547,0.4130,0.5047,0.6026,0.7849,0.7530,0.2125,0.3715,0.4610,0.6590,0.8296,0.7846
XGBRegressor,0.2067,0.3405,0.4547,0.6774,0.8237,0.8126,0.1392,0.2806,0.3731,0.7766,0.8820,0.8750
ExtraTreesRegressor,0.1857,0.3197,0.4309,0.7102,0.8430,0.8368,0.1346,0.2782,0.3669,0.7839,0.8869,0.8749
LinearRegression,4.4331,1.6396,2.1055,-5.9172,0.2210,0.2526,1.6595,0.9922,1.2882,-1.6632,0.2103,0.3003
KNeighborsRegressor,0.2328,0.3609,0.4825,0.6367,0.8020,0.7872,0.1866,0.3121,0.4320,0.7005,0.8388,0.8090
SVR,0.2164,0.3498,0.4652,0.6623,0.8147,0.8035,0.1754,0.3192,0.4188,0.7185,0.8512,0.8382


In [109]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.6596891511849, -7.354533053242414, -7.1338...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.49661527890939, -6.813701147753388, -6.35...","[-6.700389696424854, -6.738942937572096, -6.57...","[0.19996621981070536, 0.1277283680615865, 0.14..."
1,DecisionTreeRegressor,"[-5.96, -7.03, -7.08, -6.8, -6.11, -6.27, -6.6...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.3, -7.29, -5.96, -5.85, -5.89, -6.66, -6....","[-6.306, -6.831999999999999, -6.81220599920000...","[0.45093680266751357, 0.7360815172248248, 0.81..."
2,RandomForestRegressor,"[-6.5176586096866655, -7.144245314149998, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.4875879950099975, -6.430150000000001, -6....","[-6.679761140458001, -6.5774094549900015, -6.5...","[0.15866542781606108, 0.0944521311824051, 0.11..."
3,GradientBoostingRegressor,"[-6.7845848037227645, -7.202788121922744, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.585572630237401, -6.495974737122654, -6.4...","[-6.744838689778137, -6.501653483169761, -6.71...","[0.11487417760823011, 0.20845072085629274, 0.1..."
4,AdaBoostRegressor,"[-6.470169491525423, -7.200072992700735, -6.79...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.572099727999992, -6.264819977969299, -6.5...","[-6.663982117787124, -6.426408971518233, -6.40...","[0.14152605711603009, 0.20544420755135828, 0.2..."
5,XGBRegressor,"[-6.5181174, -7.6727934, -6.9720054, -7.021444...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.910313, -6.668077, -6.36401, -5.8499665, ...","[-6.630128, -6.520177, -6.496497, -5.8814535, ...","[0.45372728, 0.24646764, 0.12799953, 0.0507422..."
6,ExtraTreesRegressor,"[-6.700338719640001, -7.266899999999996, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.707955614110001, -6.387639700040001, -6.3...","[-6.870480070992005, -6.430016669619998, -6.50...","[0.11080424920783316, 0.028369967721663756, 0...."
7,LinearRegression,"[-7.244550242423563, -10.0, -10.0, -3.4, -5.70...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.443306911345437, -7.795852339646692, -3.4...","[-4.854276925644086, -5.760165286109875, -5.99...","[1.3747102823382709, 2.671314290390083, 1.6716..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -5.96166666666666...","[-6.544, -6.476000000000001, -6.05466666666666...","[0.22580620993330663, 0.13166286914354805, 0.0..."
9,SVR,"[-6.328147427655397, -7.157188135999723, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.160898250261715, -6.295156332115623, -6.0...","[-6.421213444406993, -6.335159041399239, -6.25...","[0.2655516605099862, 0.057745975283668906, 0.0..."


In [110]:
result_df.to_csv('results/Descriptors/Results_2d_Mordred_const_rem_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_df_2d_Mordred_const_rem_Caco2.csv')

In [111]:
#2d RDKit descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_LVR_rdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_LVR_rdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 152)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 152)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004010 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15399
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 135
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1974,0.3369,0.4442,0.6921,0.8319,0.8157,0.1339,0.2782,0.3660,0.7850,0.8869,0.8692
DecisionTreeRegressor,0.3676,0.4484,0.6063,0.4265,0.7088,0.6876,0.1794,0.3240,0.4236,0.7121,0.8457,0.8235
RandomForestRegressor,0.2040,0.3425,0.4517,0.6816,0.8259,0.8116,0.1467,0.2875,0.3831,0.7645,0.8757,0.8608
GradientBoostingRegressor,0.2073,0.3510,0.4553,0.6765,0.8229,0.8037,0.1562,0.3027,0.3952,0.7494,0.8676,0.8503
AdaBoostRegressor,0.2831,0.4412,0.5321,0.5582,0.7533,0.7297,0.2546,0.4183,0.5045,0.5915,0.7788,0.7288
XGBRegressor,0.2141,0.3472,0.4627,0.6659,0.8176,0.8017,0.1451,0.2885,0.3809,0.7671,0.8761,0.8605
ExtraTreesRegressor,0.1961,0.3324,0.4428,0.6941,0.8333,0.8177,0.1393,0.2820,0.3732,0.7765,0.8814,0.8719
LinearRegression,0.3204,0.4086,0.5661,0.5000,0.7346,0.7772,0.2409,0.3757,0.4908,0.6134,0.7890,0.7746
KNeighborsRegressor,0.2399,0.3620,0.4898,0.6257,0.7958,0.7732,0.1760,0.3149,0.4195,0.7176,0.8498,0.8287
SVR,0.2186,0.3586,0.4675,0.6590,0.8128,0.8004,0.1667,0.3102,0.4083,0.7325,0.8569,0.8382


In [112]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.7331380576542745, -7.0656267966705935, -7....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.756857761431517, -6.751488557100132, -6.6...","[-6.96619692026309, -6.921171504149899, -6.717...","[0.10773819061658559, 0.11104435310060434, 0.1..."
1,DecisionTreeRegressor,"[-7.24, -7.03, -8.04, -6.54, -6.11, -6.74, -5....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.64, -6.54, -6.82, -5.85, -5.85, -7.0, -5....","[-6.609999999999999, -6.962000000000001, -6.79...","[0.6098852351057535, 0.7012674240259562, 0.307..."
2,RandomForestRegressor,"[-6.635410299960001, -6.936439700039992, -6.97...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.633460299960002, -6.548489700039998, -6.5...","[-6.7468109121300035, -6.913396189294002, -6.7...","[0.11393676259647319, 0.18418354506687146, 0.1..."
3,GradientBoostingRegressor,"[-6.52389617591049, -6.767324538927256, -6.756...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.591196246829025, -7.047793179172765, -6.6...","[-6.887507910777892, -7.0664517093962, -6.6239...","[0.18308544384146208, 0.22351391618335759, 0.1..."
4,AdaBoostRegressor,"[-6.9488410122021165, -7.298248074583848, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.7665454879107125, -6.706780071223102, -6....","[-6.786955783897722, -6.562826258122937, -6.31...","[0.130696710489932, 0.21041580461360293, 0.182..."
5,XGBRegressor,"[-6.5961, -7.247772, -7.2307525, -6.6548653, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6627836, -6.3699, -6.6279893, -5.852619, ...","[-6.828593, -6.605147, -6.6740923, -5.855068, ...","[0.15943657, 0.27176362, 0.1647111, 0.00812149..."
6,ExtraTreesRegressor,"[-6.773800000000001, -7.227999999999995, -6.99...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.696110299960002, -6.241400000000001, -6.4...","[-6.880002059992006, -6.409157416578, -6.53702...","[0.12141808082784569, 0.12838309172879536, 0.1..."
7,LinearRegression,"[-10.0, -7.101711663227222, -6.815543396131391...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -6.110100472002072, -5.52486722628110...","[-7.348795481985587, -6.298093389903551, -5.53...","[1.3257371469141865, 0.1620417822516641, 0.041..."
8,KNeighborsRegressor,"[-6.363333333333333, -7.3500000000000005, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.105, -6.316666666666667, -5.703333...","[-6.544, -6.328000000000001, -6.58933333333333...","[0.22580620993330663, 0.11151681487560522, 0.1..."
9,SVR,"[-6.387090877405147, -7.134456790445833, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.38704380130956, -6.458249168307375, -6.37...","[-6.908807341155294, -6.52286116187244, -6.417...","[0.26088748483372726, 0.058692981973270615, 0...."


In [113]:
result_df.to_csv('results/Descriptors/Results_2d_rdkit_LVR_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_rdkit_LVR_Caco2.csv')

In [116]:
#2d Mordred descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101,),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
results_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
results_df

/tmp/ipykernel_1796054/2065252241.py:2: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
/tmp/ipykernel_1796054/2065252241.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 843)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 843)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014651 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 170679
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 809
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1928,0.3296,0.4391,0.6992,0.8363,0.8248,0.1322,0.2769,0.3636,0.7878,0.8895,0.8754
DecisionTreeRegressor,0.3803,0.4487,0.6167,0.4066,0.7013,0.6715,0.1844,0.3152,0.4294,0.7040,0.8406,0.8341
RandomForestRegressor,0.1993,0.3419,0.4464,0.6890,0.8314,0.8189,0.1430,0.2857,0.3782,0.7705,0.8818,0.8711
GradientBoostingRegressor,0.2103,0.3492,0.4586,0.6718,0.8197,0.8054,0.1459,0.2911,0.3819,0.7659,0.8775,0.8622
AdaBoostRegressor,0.2631,0.4189,0.5130,0.5894,0.7753,0.7524,0.2272,0.3884,0.4767,0.6353,0.8106,0.7556
XGBRegressor,0.2075,0.3404,0.4555,0.6762,0.8231,0.8187,0.1334,0.2702,0.3652,0.7859,0.8880,0.8813
ExtraTreesRegressor,0.1842,0.3175,0.4291,0.7126,0.8443,0.8364,0.1351,0.2768,0.3676,0.7832,0.8865,0.8762
LinearRegression,2.8989,1.2209,1.7026,-3.5233,0.3075,0.3808,1.1249,0.8289,1.0606,-0.8052,0.3837,0.4403
KNeighborsRegressor,0.2250,0.3565,0.4743,0.6490,0.8086,0.7933,0.1758,0.3073,0.4193,0.7178,0.8487,0.8228
SVR,0.2146,0.3503,0.4632,0.6652,0.8165,0.8069,0.1696,0.3129,0.4118,0.7278,0.8572,0.8420


In [117]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.815903062240063, -7.316151603285766, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.494782640568548, -6.637911265516238, -6.7...","[-6.689169275629672, -6.7037607228479645, -6.7...","[0.185214171014419, 0.0893342237282947, 0.0306..."
1,DecisionTreeRegressor,"[-5.96, -6.958607315, -7.03, -5.64, -5.6, -6.7...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -7.24, -6.82, -5.85, -5.89, -7.03, -5...","[-6.132, -6.575999999999999, -6.708, -5.872, -...","[0.41997142759954514, 0.5180579118206767, 0.41..."
2,RandomForestRegressor,"[-6.626109999999999, -7.107208187539999, -6.98...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.61457379957, -6.436149999999997, -6.43080...","[-6.700525944779334, -6.575970549637999, -6.55...","[0.19918985263209085, 0.1010058074994568, 0.07..."
3,GradientBoostingRegressor,"[-6.848904228176594, -7.165966386260953, -7.03...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.752516307534712, -6.779207284248975, -6.5...","[-6.8943354677226765, -6.749694826159984, -6.7...","[0.10850765061247127, 0.1794585074454129, 0.19..."
4,AdaBoostRegressor,"[-6.846182222096433, -7.265067567567566, -6.85...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.805706521739135, -6.639305555555561, -6.8...","[-6.674055008305244, -6.4342923576491415, -6.5...","[0.13433241426280537, 0.21307670714124005, 0.2..."
5,XGBRegressor,"[-7.1306734, -7.505147, -7.1657186, -6.478043,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.35097, -6.155054, -6.3079314, -5.8497014,...","[-6.900431, -6.6041327, -6.4641905, -5.876465,...","[0.3054, 0.24738714, 0.16381346, 0.053692956, ..."
6,ExtraTreesRegressor,"[-6.846800000000003, -7.195686073149995, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.854549019600004, -6.31615, -6.21003970003...","[-6.835288337452004, -6.382273596552001, -6.42...","[0.09793645924796762, 0.05481416422088993, 0.1..."
7,LinearRegression,"[-5.239553090748629, -5.186505717927318, -3.4,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.819331778893499, -9.36631449859469, -6.80...","[-4.867680967312589, -5.2557141357785895, -7.4...","[1.2420690777943113, 2.42279074339584, 2.03007..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31166666666666...","[-6.544, -6.476000000000001, -6.24933333333333...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.464019866338379, -7.177931288163172, -7.08...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.339315701026328, -6.351327056323548, -6.1...","[-6.643650728190556, -6.3594841994334095, -6.3...","[0.30150246467329644, 0.024677066723286407, 0...."


In [118]:
results_df.to_csv('results/Descriptors/Results_2d_Mordred_LVR_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_Mordred_LVR_Caco2.csv')

In [119]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_Caco2.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-0.9878,0.975749,203.2963,128.382338,0,0,120,54,66,...,106.404267,1.970449,37.175727,17.865461,19.310266,10383.0,92.0,5.550,272.0,1013
1,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,1003
2,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,1005
3,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,1002
4,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,1007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,0,-9.6097,92.346334,344.0469,231.595816,0,0,214,102,112,...,204.450386,2.004416,78.927096,30.584156,38.662023,60444.0,180.0,7.726,532.0,8354
1003,0,-3.4642,12.000682,366.2199,208.635370,0,0,185,95,90,...,189.206654,1.991649,73.459617,30.422275,37.991158,50819.0,160.0,2.411,480.0,8492
1004,0,-2.6870,7.219969,379.0408,217.916128,0,0,194,98,96,...,195.069444,1.990505,73.605382,30.474598,38.084600,54340.0,164.0,3.846,494.0,8493
1005,0,-2.9426,8.658895,374.5890,214.822542,0,0,191,97,94,...,193.061774,1.990328,73.567692,30.461069,38.060439,53039.0,164.0,3.066,490.0,8491


In [120]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
df 


/tmp/ipykernel_1796054/3321453248.py:1: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.071798,2.429231,4.858462,...,11.569599,162.262107,1663.015310,6.547304,94190,208,600.0,698.0,50.000000,26.805556
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,148.508426,2.439658,4.878720,...,11.569485,161.240399,1648.999660,6.569720,90388,207,596.0,694.0,49.750000,26.555556
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,130.882818,2.423927,4.847855,...,11.437501,150.677817,1585.815201,6.894849,69754,181,540.0,618.0,48.506944,24.125000
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,140.033288,2.434167,4.866917,...,11.543988,155.034166,1572.968359,6.526840,79884,202,566.0,663.0,49.750000,25.222222
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,132.454775,2.413793,4.827585,...,11.409840,150.603018,1565.869824,6.720471,69806,179,536.0,612.0,47.284722,24.347222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,37.874256,2.316794,4.629331,...,9.934744,65.834243,430.258006,6.619354,2632,45,148.0,164.0,11.750000,7.000000
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,37.883498,2.320955,4.638096,...,9.935228,65.834728,430.258006,6.619354,2620,45,148.0,164.0,11.750000,7.000000
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,35.896500,2.304816,4.609632,...,9.859065,63.474076,402.263091,6.385128,2286,42,138.0,153.0,10.638889,6.583333
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,34.487329,2.307278,4.602399,...,9.845805,62.359842,388.247441,6.470791,2069,41,134.0,149.0,10.388889,6.333333


In [121]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1013,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-6.420000,0,-0.9878,0.975749,203.2963,128.382338,0,0,...,6.287494,106.404267,1.970449,37.175727,17.865461,19.310266,10383.0,92.0,5.550,272.0
1,1003,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](...,-5.100000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
2,1005,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@H]2C...,-6.320000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
3,1002,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@H](C...,-6.840000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
4,1007,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](...,-6.640000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,8354,CC[C@H](C)[C@@H]1NC(=O)[C@H](C2CCCC2)N(C)C(=O)...,-6.207608,0,-9.6097,92.346334,344.0469,231.595816,0,0,...,6.788861,204.450386,2.004416,78.927096,30.584156,38.662023,60444.0,180.0,7.726,532.0
1003,8492,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)C[C@@H...,-7.000000,0,-3.4642,12.000682,366.2199,208.635370,0,0,...,7.289826,189.206654,1.991649,73.459617,30.422275,37.991158,50819.0,160.0,2.411,480.0
1004,8493,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)C[C@@H...,-7.096910,0,-2.6870,7.219969,379.0408,217.916128,0,0,...,7.168375,195.069444,1.990505,73.605382,30.474598,38.084600,54340.0,164.0,3.846,494.0
1005,8491,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)C[C@@H...,-6.958607,0,-2.9426,8.658895,374.5890,214.822542,0,0,...,7.207587,193.061774,1.990328,73.567692,30.461069,38.060439,53039.0,164.0,3.066,490.0


In [122]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,0,-3.5051,12.285726,462.8621,276.940262,0,0,...,6.547304,237.600778,1.980006,82.612830,37.983553,44.629277,94190.0,208.0,8.466,600.0
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,0,-3.7596,14.134592,457.1255,273.846676,0,0,...,6.569720,235.601472,1.979844,82.357901,37.975600,44.382301,90388.0,207.0,8.692,596.0
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,0,-4.7036,22.123853,403.5966,250.593953,0,0,...,6.894849,214.100758,1.964227,89.673466,40.258888,41.162462,69754.0,181.0,6.958,540.0
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,0,-5.1318,26.335371,429.6680,260.619504,0,0,...,6.526840,222.500266,1.969029,82.337179,37.880967,44.456212,79884.0,202.0,6.376,566.0
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,0,-6.2030,38.477209,402.2859,252.174332,0,0,...,6.720471,214.421483,1.967170,87.553124,40.278853,41.547594,69806.0,179.0,7.107,536.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,0,-1.3838,1.914902,115.0756,70.758962,0,0,...,6.619354,61.528299,1.984784,22.437229,10.154168,12.283061,2632.0,45.0,2.298,148.0
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,0,-1.3901,1.932378,114.0384,70.758962,0,0,...,6.619354,61.528106,1.984778,22.197058,10.153913,12.043145,2620.0,45.0,3.111,148.0
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,0,-1.1989,1.437361,110.0599,68.196962,0,0,...,6.385128,57.707840,1.989926,19.684197,7.618398,12.065799,2286.0,42.0,3.395,138.0
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,0,-0.9109,0.829739,107.1483,65.103376,0,0,...,6.470791,55.707630,1.989558,19.686908,7.618921,12.067987,2069.0,41.0,3.037,134.0


In [123]:
df_ordered.to_csv('features/Descriptors/Train_2d_padel_curated_Caco2.csv', index=False)

In [124]:
#2d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_Caco2.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-4.2390,17.969121,111.3714,69.282962,0,0,66,32,34,...,61.494697,1.921709,33.025324,14.973326,18.051998,2732.0,56.0,0.188,156.0,1020
1,0,-2.5853,6.683776,183.7187,117.767994,0,0,109,51,58,...,101.771380,1.995517,37.261449,17.936517,19.324931,9053.0,85.0,4.485,262.0,1062
2,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,1004
3,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0,1001
4,0,-0.9878,0.975749,203.2963,128.382338,0,0,120,54,66,...,106.401998,1.970407,37.167232,17.862301,19.304931,10372.0,92.0,5.550,272.0,1028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,0,-3.6573,13.375843,462.8222,276.940262,0,0,254,120,134,...,237.423604,1.978530,82.566872,37.965469,44.601403,91297.0,211.0,8.405,602.0,2065
248,0,-3.4877,12.164051,371.6893,214.602956,0,0,191,99,92,...,197.073279,1.990639,78.403426,30.481362,38.096680,55739.0,165.0,3.613,498.0,8370
249,0,-2.7880,7.772944,363.1421,205.541784,0,0,182,94,88,...,187.209315,1.991588,73.414277,30.406151,37.961942,49207.0,158.0,1.842,476.0,8075
250,0,-2.8466,8.103132,379.2485,217.916128,0,0,194,98,96,...,194.883906,1.988611,73.535292,30.448936,38.040174,54210.0,168.0,3.592,496.0,8076


In [125]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
df

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.002370,2.434280,4.865870,...,11.590451,162.312312,1663.015310,6.547304,91297,211,602.0,703.0,50.611111,26.777778
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,133.544218,2.399797,4.799594,...,11.363578,152.529489,1571.020120,6.309318,74798,175,538.0,606.0,48.763889,24.777778
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,138.687365,2.431377,4.862754,...,11.482930,153.892806,1550.951647,6.516604,73336,187,562.0,647.0,45.645833,24.736111
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,131.842973,2.484317,4.905149,...,11.534364,163.933604,1535.873430,6.736287,67356,187,554.0,645.0,46.569444,23.486111
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,133.583282,2.474028,4.901032,...,11.515712,164.907694,1529.943702,6.538221,69591,185,556.0,644.0,46.208333,23.763889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.436032,2.302525,4.605049,...,9.885069,65.694305,430.294391,6.236151,2750,44,146.0,161.0,11.138889,7.083333
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.421813,2.317144,4.634288,...,9.891162,65.702291,430.294391,6.236151,2714,44,146.0,161.0,11.138889,7.083333
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.475166,2.317701,4.635401,...,9.885578,65.694814,430.294391,6.236151,2660,44,146.0,161.0,11.138889,7.083333
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.446535,2.317849,4.635698,...,9.885069,65.694305,430.294391,6.236151,2660,44,146.0,161.0,11.138889,7.083333


In [126]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1020,C[C@H]1C(=O)N[C@H](C)C(=O)N[C@H](C)C(=O)N[C@H]...,-5.620000,0,-4.2390,17.969121,111.3714,69.282962,0,0,...,6.882636,61.494697,1.921709,33.025324,14.973326,18.051998,2732.0,56.0,0.188,156.0
1,1062,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-5.980000,0,-2.5853,6.683776,183.7187,117.767994,0,0,...,6.517768,101.771380,1.995517,37.261449,17.936517,19.324931,9053.0,85.0,4.485,262.0
2,1004,CC(C)C[C@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](C...,-7.350000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
3,1001,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-6.370000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950923,1.979430,36.643890,17.922198,18.721692,9274.0,80.0,6.411,254.0
4,1028,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[C@H](...,-4.820000,0,-0.9878,0.975749,203.2963,128.382338,0,0,...,6.287494,106.401998,1.970407,37.167232,17.862301,19.304931,10372.0,92.0,5.550,272.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.220000,0,-3.6573,13.375843,462.8222,276.940262,0,0,...,6.547304,237.423604,1.978530,82.566872,37.965469,44.601403,91297.0,211.0,8.405,602.0
248,8370,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)C[C@@H...,-7.154902,0,-3.4877,12.164051,371.6893,214.602956,0,0,...,7.395970,197.073279,1.990639,78.403426,30.481362,38.096680,55739.0,165.0,3.613,498.0
249,8075,CC(C)[C@H]1C(=O)N[C@H](C(=O)N2CCCC2)CC(=O)N(C)...,-7.000000,0,-2.7880,7.772944,363.1421,205.541784,0,0,...,7.332979,187.209315,1.991588,73.414277,30.406151,37.961942,49207.0,158.0,1.842,476.0
250,8076,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)C[C@@H...,-6.853872,0,-2.8466,8.103132,379.2485,217.916128,0,0,...,7.168375,194.883906,1.988611,73.535292,30.448936,38.040174,54210.0,168.0,3.592,496.0


In [127]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,0,-3.6573,13.375843,462.8222,276.940262,0,0,...,6.547304,237.423604,1.978530,82.566872,37.965469,44.601403,91297.0,211.0,8.405,602.0
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,0,-6.4306,41.352616,403.2109,261.255434,0,0,...,6.309318,217.445869,1.958972,88.222714,50.298796,37.923918,74798.0,175.0,6.768,538.0
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,0,-0.7114,0.506090,433.7132,259.803918,0,0,...,6.516604,222.331854,1.985106,74.211794,35.995049,38.216745,73336.0,187.0,10.339,562.0
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,0,-3.8138,14.545070,395.3981,246.117160,0,0,...,6.736287,213.827378,1.979883,85.394192,35.950728,41.996280,67356.0,187.0,6.203,554.0
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,0,-3.6776,13.524742,398.5483,250.791125,0,0,...,6.538221,215.983270,1.981498,82.730212,35.983356,41.825434,69591.0,185.0,8.843,556.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,0,-1.7749,3.150270,115.8831,74.384134,0,0,...,6.236151,61.708009,1.990581,19.682164,7.618006,12.064158,2750.0,44.0,4.111,146.0
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,0,-2.0636,4.258445,117.3827,74.384134,0,0,...,6.236151,61.707604,1.990568,20.026186,7.618719,12.407467,2714.0,44.0,2.941,146.0
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,0,-1.7749,3.150270,115.8831,74.384134,0,0,...,6.236151,61.707690,1.990571,19.711184,7.642415,12.068770,2660.0,44.0,4.111,146.0
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,0,-1.7749,3.150270,115.8831,74.384134,0,0,...,6.236151,61.709088,1.990616,19.692320,7.625771,12.066549,2660.0,44.0,4.111,146.0


In [128]:
df_ordered.to_csv('features/Descriptors/Test_2d_padel_curated_Caco2.csv', index=False)

In [129]:
#3d Train descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_Caco2.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.258699,2.180929,3.007381,3.721109,4.540169,5.368250,6.042420,6.903012,7.614435,8.370072,...,0.387445,0.447585,0.521827,0.413585,31.447933,285.379271,934.147657,0.350093,1.382997,1008
1,1.260282,2.182566,3.015559,3.738023,4.603886,5.408368,6.092705,6.876287,7.597202,8.105999,...,0.387269,0.400432,0.503676,0.382835,29.575198,272.296284,1043.363799,0.251767,1.286942,1013
2,1.259948,2.182851,3.008583,3.729256,4.581752,5.435160,6.161342,7.007041,7.751367,8.440102,...,0.424211,0.418171,0.500741,0.437093,33.832509,314.273089,845.954108,0.412108,1.356005,1016
3,1.260622,2.181420,3.012163,3.740586,4.576897,5.358042,6.050712,6.994985,7.662369,8.447052,...,0.368047,0.472114,0.463353,0.419514,33.063529,302.312085,920.058358,0.380651,1.354981,1014
4,1.257316,2.180195,3.034966,3.762433,4.535904,5.445854,6.088056,6.941994,7.605202,8.040807,...,0.391565,0.517564,0.450820,0.418180,18.365955,97.429245,238.541256,0.350888,1.386564,1019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,1.277559,2.207668,3.036711,3.835865,4.651275,5.544488,6.257050,6.971570,7.770863,8.426010,...,0.370557,0.508737,0.533217,0.293847,30.764490,293.852523,1157.926980,0.249475,1.335801,998
1003,1.260738,2.194763,3.000624,3.722693,4.663512,5.502301,6.291980,7.054932,7.720227,8.371306,...,0.355440,0.381091,0.527586,0.310474,30.587341,291.669478,1162.926912,0.235066,1.219152,8463
1004,1.260757,2.180387,3.014419,3.745043,4.580792,5.345018,6.125664,6.946868,7.621243,8.306583,...,0.387454,0.471391,0.495983,0.466282,31.752232,294.306346,1003.015248,0.337670,1.433656,8462
1005,1.277692,2.214994,3.037614,3.840098,4.682495,5.521210,6.235786,6.917594,7.539357,8.146669,...,0.312242,0.474062,0.510982,0.342733,28.958671,236.885689,779.417892,0.352981,1.327776,991


In [130]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
df 

/tmp/ipykernel_1796054/4010585191.py:1: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.071798,2.429231,4.858462,...,11.569599,162.262107,1663.015310,6.547304,94190,208,600.0,698.0,50.000000,26.805556
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,148.508426,2.439658,4.878720,...,11.569485,161.240399,1648.999660,6.569720,90388,207,596.0,694.0,49.750000,26.555556
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,130.882818,2.423927,4.847855,...,11.437501,150.677817,1585.815201,6.894849,69754,181,540.0,618.0,48.506944,24.125000
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,140.033288,2.434167,4.866917,...,11.543988,155.034166,1572.968359,6.526840,79884,202,566.0,663.0,49.750000,25.222222
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,132.454775,2.413793,4.827585,...,11.409840,150.603018,1565.869824,6.720471,69806,179,536.0,612.0,47.284722,24.347222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,37.874256,2.316794,4.629331,...,9.934744,65.834243,430.258006,6.619354,2632,45,148.0,164.0,11.750000,7.000000
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,37.883498,2.320955,4.638096,...,9.935228,65.834728,430.258006,6.619354,2620,45,148.0,164.0,11.750000,7.000000
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,35.896500,2.304816,4.609632,...,9.859065,63.474076,402.263091,6.385128,2286,42,138.0,153.0,10.638889,6.583333
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,34.487329,2.307278,4.602399,...,9.845805,62.359842,388.247441,6.470791,2069,41,134.0,149.0,10.388889,6.333333


In [131]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1008,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-4.700,1.258699,2.180929,3.007381,3.721109,4.540169,5.368250,6.042420,...,0.512617,0.387445,0.447585,0.521827,0.413585,31.447933,285.379271,934.147657,0.350093,1.382997
1,1013,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-6.420,1.260282,2.182566,3.015559,3.738023,4.603886,5.408368,6.092705,...,0.447243,0.387269,0.400432,0.503676,0.382835,29.575198,272.296284,1043.363799,0.251767,1.286942
2,1016,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N(C)[C@...,-4.790,1.259948,2.182851,3.008583,3.729256,4.581752,5.435160,6.161342,...,0.517194,0.424211,0.418171,0.500741,0.437093,33.832509,314.273089,845.954108,0.412108,1.356005
3,1014,CC(C)C[C@H]1C(=O)N[C@H](CC(C)C)C(=O)N[C@@H](Cc...,-4.630,1.260622,2.181420,3.012163,3.740586,4.576897,5.358042,6.050712,...,0.552387,0.368047,0.472114,0.463353,0.419514,33.063529,302.312085,920.058358,0.380651,1.354981
4,1019,C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@@H](C)N(C)C(=O)...,-6.820,1.257316,2.180195,3.034966,3.762433,4.535904,5.445854,6.088056,...,0.509027,0.391565,0.517564,0.450820,0.418180,18.365955,97.429245,238.541256,0.350888,1.386564
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,998,CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...,-4.710,1.277559,2.207668,3.036711,3.835865,4.651275,5.544488,6.257050,...,0.462427,0.370557,0.508737,0.533217,0.293847,30.764490,293.852523,1157.926980,0.249475,1.335801
1003,8463,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-5.545,1.260738,2.194763,3.000624,3.722693,4.663512,5.502301,6.291980,...,0.467937,0.355440,0.381091,0.527586,0.310474,30.587341,291.669478,1162.926912,0.235066,1.219152
1004,8462,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-4.890,1.260757,2.180387,3.014419,3.745043,4.580792,5.345018,6.125664,...,0.504326,0.387454,0.471391,0.495983,0.466282,31.752232,294.306346,1003.015248,0.337670,1.433656
1005,991,CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...,-5.130,1.277692,2.214994,3.037614,3.840098,4.682495,5.521210,6.235786,...,0.568654,0.312242,0.474062,0.510982,0.342733,28.958671,236.885689,779.417892,0.352981,1.327776


In [132]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,1.261273,2.190355,3.027664,3.762887,4.540828,5.367823,6.058709,...,0.682480,0.263316,0.443174,0.407327,0.382690,86.049198,1710.236099,8002.655598,0.523721,1.233191
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,1.262431,2.192863,3.031661,3.781496,4.558010,5.339872,5.992600,...,0.649321,0.261240,0.473762,0.435183,0.424702,75.043451,1413.897215,7900.542155,0.473981,1.333648
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,1.271211,2.197716,3.029135,3.777147,4.582303,5.311646,6.049205,...,0.453945,0.442816,0.519102,0.558050,0.307857,61.920501,1125.686310,6114.512304,0.345141,1.385010
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,1.260840,2.185970,3.029271,3.761923,4.514570,5.306246,6.022174,...,0.684172,0.255207,0.455347,0.407037,0.345978,81.472355,1536.980018,7342.617572,0.526258,1.208362
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,1.267464,2.193763,3.015203,3.741535,4.552034,5.392482,6.073215,...,0.602089,0.313243,0.519864,0.540548,0.332260,66.912273,1191.395466,6042.190896,0.403133,1.392672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,1.258979,2.196991,2.997396,3.718550,4.608931,5.418393,6.151181,...,0.472803,0.459062,0.423922,0.370460,0.380497,20.467358,117.521135,264.784723,0.397798,1.174879
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,1.255928,2.193836,2.988750,3.734600,4.602278,5.397440,6.027611,...,0.522270,0.375640,0.414966,0.480757,0.407114,19.019121,104.124280,260.934838,0.346865,1.302838
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,1.255999,2.189732,2.981461,3.735628,4.540402,5.308589,5.927278,...,0.707274,0.194034,0.458709,0.388888,0.412112,20.136274,91.711934,222.430095,0.560911,1.259709
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,1.259567,2.196124,2.980865,3.696733,4.505346,5.222768,5.823652,...,0.727162,0.167350,0.503309,0.433183,0.434002,18.625555,74.950560,176.520805,0.590742,1.370494


In [133]:
df_ordered.to_csv('features/Descriptors/Train_3d_padel_curated_Caco2.csv', index=False)

In [134]:
#3d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_Caco2.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.267902,2.228243,3.035981,3.852027,4.729825,5.542040,6.311603,7.041036,7.805660,8.577321,...,0.431811,0.454910,0.470772,0.394548,39.111968,411.564708,1062.752937,0.432037,1.320230,1076
1,1.261783,2.195594,3.007083,3.743466,4.670346,5.503053,6.240033,6.976042,7.671405,8.510260,...,0.337401,0.461713,0.420158,0.286605,33.813268,337.573927,1306.525925,0.295715,1.168476,1067
2,1.265698,2.197781,3.010333,3.779917,4.652641,5.436324,6.213579,6.952003,7.612897,8.270584,...,0.374928,0.515335,0.447281,0.357422,28.626378,240.121522,784.708635,0.327479,1.320038,1046
3,1.261028,2.179813,3.020771,3.767318,4.626784,5.373797,6.025979,6.882972,7.570218,8.214862,...,0.379444,0.458094,0.519181,0.407998,31.550476,275.945732,804.977414,0.384848,1.385274,1015
4,1.267432,2.213127,3.053467,3.849820,4.616662,5.379100,6.092198,6.905381,7.642099,8.392989,...,0.408240,0.502138,0.430201,0.289164,34.616984,347.871728,1207.484556,0.351757,1.221503,1026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,1.264627,2.194580,3.034653,3.780254,4.536317,5.352328,6.048521,6.880090,7.694998,8.504601,...,0.408020,0.466148,0.552108,0.450563,64.696337,1164.557490,5282.975406,0.394501,1.468819,8497
248,1.260657,2.189287,3.022255,3.749302,4.544307,5.336204,6.088267,6.941841,7.663620,8.307428,...,0.421365,0.470027,0.439001,0.336254,60.837527,1063.591919,5256.172608,0.366613,1.245282,8478
249,1.261422,2.185241,3.023746,3.752996,4.572727,5.345884,6.056876,6.842111,7.505846,8.229327,...,0.406160,0.521125,0.529161,0.467468,59.446712,1031.956951,5427.786206,0.344474,1.517754,8480
250,1.276715,2.212395,3.032477,3.826975,4.600393,5.478488,6.224973,6.963692,7.717242,8.269915,...,0.350230,0.497514,0.578441,0.401524,29.507207,241.522017,719.366631,0.366758,1.477479,989


In [135]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
df

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.002370,2.434280,4.865870,...,11.590451,162.312312,1663.015310,6.547304,91297,211,602.0,703.0,50.611111,26.777778
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,133.544218,2.399797,4.799594,...,11.363578,152.529489,1571.020120,6.309318,74798,175,538.0,606.0,48.763889,24.777778
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,138.687365,2.431377,4.862754,...,11.482930,153.892806,1550.951647,6.516604,73336,187,562.0,647.0,45.645833,24.736111
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,131.842973,2.484317,4.905149,...,11.534364,163.933604,1535.873430,6.736287,67356,187,554.0,645.0,46.569444,23.486111
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,133.583282,2.474028,4.901032,...,11.515712,164.907694,1529.943702,6.538221,69591,185,556.0,644.0,46.208333,23.763889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.436032,2.302525,4.605049,...,9.885069,65.694305,430.294391,6.236151,2750,44,146.0,161.0,11.138889,7.083333
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.421813,2.317144,4.634288,...,9.891162,65.702291,430.294391,6.236151,2714,44,146.0,161.0,11.138889,7.083333
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.475166,2.317701,4.635401,...,9.885578,65.694814,430.294391,6.236151,2660,44,146.0,161.0,11.138889,7.083333
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.446535,2.317849,4.635698,...,9.885069,65.694305,430.294391,6.236151,2660,44,146.0,161.0,11.138889,7.083333


In [136]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1076,C[C@@H](O)[C@@H]1NC(=O)[C@H](CCCCN)NC(=O)[C@@H...,-6.03,1.267902,2.228243,3.035981,3.852027,4.729825,5.542040,6.311603,...,0.522881,0.431811,0.454910,0.470772,0.394548,39.111968,411.564708,1062.752937,0.432037,1.320230
1,1067,CC(C)C[C@@H]1NC(=O)[C@H](Cc2cc3ccccc3[nH]2)NC(...,-5.02,1.261783,2.195594,3.007083,3.743466,4.670346,5.503053,6.240033,...,0.526409,0.337401,0.461713,0.420158,0.286605,33.813268,337.573927,1306.525925,0.295715,1.168476
2,1046,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-7.00,1.265698,2.197781,3.010333,3.779917,4.652641,5.436324,6.213579,...,0.510058,0.374928,0.515335,0.447281,0.357422,28.626378,240.121522,784.708635,0.327479,1.320038
3,1015,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)N(C)C(=O)[C@@...,-5.67,1.261028,2.179813,3.020771,3.767318,4.626784,5.373797,6.025979,...,0.543788,0.379444,0.458094,0.519181,0.407998,31.550476,275.945732,804.977414,0.384848,1.385274
4,1026,C[C@H](O)[C@H]1NC(=O)[C@@H](CCCCN)N(C)C(=O)[C@...,-6.14,1.267432,2.213127,3.053467,3.849820,4.616662,5.379100,6.092198,...,0.492932,0.408240,0.502138,0.430201,0.289164,34.616984,347.871728,1207.484556,0.351757,1.221503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,8497,CC[C@H](C)[C@H]1C(=O)N[C@@H]([C@@H](C)O)C(=O)N...,-5.74,1.264627,2.194580,3.034653,3.780254,4.536317,5.352328,6.048521,...,0.521648,0.408020,0.466148,0.552108,0.450563,64.696337,1164.557490,5282.975406,0.394501,1.468819
248,8478,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.96,1.260657,2.189287,3.022255,3.749302,4.544307,5.336204,6.088267,...,0.489711,0.421365,0.470027,0.439001,0.336254,60.837527,1063.591919,5256.172608,0.366613,1.245282
249,8480,CC(C)C[C@H]1C(=O)N[C@@H]([C@@H](C)O)C(=O)N(C)C...,-5.52,1.261422,2.185241,3.023746,3.752996,4.572727,5.345884,6.056876,...,0.490156,0.406160,0.521125,0.529161,0.467468,59.446712,1031.956951,5427.786206,0.344474,1.517754
250,989,CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...,-6.70,1.276715,2.212395,3.032477,3.826975,4.600393,5.478488,6.224973,...,0.560942,0.350230,0.497514,0.578441,0.401524,29.507207,241.522017,719.366631,0.366758,1.477479


In [137]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,1.263404,2.191690,3.035150,3.789481,4.558684,5.359123,6.043198,...,0.716440,0.242246,0.443935,0.427639,0.435742,83.625039,1490.670916,5767.464867,0.574661,1.307316
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,1.257372,2.169873,2.993434,3.687867,4.520505,5.372375,6.145715,...,0.521776,0.409600,0.550410,0.543744,0.402603,73.734856,1509.449580,7462.654757,0.397064,1.496757
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,1.260817,2.189506,3.019340,3.753634,4.564461,5.397186,6.099380,...,0.507796,0.403815,0.488706,0.481901,0.280013,63.182949,1140.267771,5775.076070,0.367416,1.250620
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,1.268739,2.190621,3.019774,3.745311,4.521554,5.331307,6.040931,...,0.535190,0.393164,0.577035,0.564099,0.424267,65.667269,1194.175198,5528.761689,0.392532,1.565401
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,1.264969,2.187069,3.010662,3.762064,4.588166,5.379708,6.126033,...,0.499362,0.399019,0.557737,0.513407,0.345796,66.494387,1284.657984,7304.227786,0.347570,1.416940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,1.254548,2.185015,2.974378,3.726451,4.616889,5.411257,6.101693,...,0.596354,0.355038,0.532925,0.525713,0.444230,21.235527,116.332557,236.122220,0.427088,1.502868
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,1.256303,2.182635,2.973618,3.704436,4.517821,5.218354,5.861354,...,0.666921,0.279758,0.442211,0.464638,0.469105,21.677207,111.392008,234.404547,0.500382,1.375955
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,1.254770,2.186992,2.968123,3.708552,4.557850,5.340383,5.977843,...,0.585075,0.342986,0.454788,0.501705,0.443123,20.730176,114.928147,264.264648,0.392091,1.399616
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,1.254327,2.187590,2.965219,3.693635,4.558658,5.349671,6.042910,...,0.570406,0.368369,0.477650,0.533995,0.392357,21.516702,123.888733,273.556994,0.408162,1.404002


In [138]:
df_ordered.to_csv('features/Descriptors/Test_3d_padel_curated_Caco2.csv', index=False)

In [139]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 1444)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1444)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019162 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 222063
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1023
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1909,0.3308,0.4369,0.7022,0.8380,0.8245,0.1412,0.2871,0.3758,0.7734,0.8809,0.8670
DecisionTreeRegressor,0.3610,0.4388,0.6008,0.4368,0.7146,0.6909,0.2043,0.3458,0.4520,0.6721,0.8228,0.8081
RandomForestRegressor,0.1926,0.3376,0.4389,0.6994,0.8386,0.8296,0.1544,0.2991,0.3930,0.7522,0.8715,0.8566
GradientBoostingRegressor,0.1965,0.3396,0.4433,0.6934,0.8330,0.8165,0.1480,0.2914,0.3847,0.7625,0.8745,0.8625
AdaBoostRegressor,0.2440,0.4026,0.4939,0.6193,0.7951,0.7638,0.2093,0.3724,0.4575,0.6642,0.8287,0.7919
XGBRegressor,0.2135,0.3462,0.4621,0.6668,0.8179,0.8070,0.1505,0.2911,0.3879,0.7585,0.8713,0.8563
ExtraTreesRegressor,0.1790,0.3158,0.4231,0.7207,0.8494,0.8465,0.1391,0.2846,0.3729,0.7768,0.8828,0.8699
LinearRegression,10.6503,3.0474,3.2635,-15.6182,0.0653,0.0482,3.1195,1.4222,1.7662,-4.0063,0.0150,0.0200
KNeighborsRegressor,0.2305,0.3579,0.4801,0.6403,0.8043,0.7871,0.1791,0.3117,0.4232,0.7126,0.8469,0.8207
SVR,0.2142,0.3513,0.4628,0.6658,0.8169,0.8024,0.1728,0.3216,0.4157,0.7226,0.8513,0.8336


In [140]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.358050576787527, -7.239684117601858, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5515026741745785, -6.412034917185415, -6....","[-6.737679234426139, -6.500143746923191, -6.56...","[0.11407076754980458, 0.11414741091902134, 0.0..."
1,DecisionTreeRegressor,"[-6.96, -7.03, -7.05, -6.89, -6.47, -6.3, -5.7...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.96, -6.24, -5.96, -5.85, -5.72, -7.0, -5....","[-6.530000000000001, -6.306, -6.638, -5.863999...","[0.7230490993010089, 0.25048752463945206, 0.60..."
2,RandomForestRegressor,"[-6.37281344796, -7.230498039199996, -7.065238...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.459352167599995, -6.271213743533332, -6.4...","[-6.557677198028668, -6.439558371389333, -6.45...","[0.08908918316088775, 0.09638775814779336, 0.0..."
3,GradientBoostingRegressor,"[-6.306873805242829, -7.1377080165452345, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.635140630787711, -6.195114417588883, -6.4...","[-6.767571667109991, -6.427146840821348, -6.58...","[0.17503393019196528, 0.13276337817608092, 0.0..."
4,AdaBoostRegressor,"[-6.1453618182725185, -7.284038717993677, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.406747873624999, -6.197758620689661, -6.4...","[-6.53646263180377, -6.27321764977504, -6.2072...","[0.10844798959051137, 0.1166066443284845, 0.11..."
5,XGBRegressor,"[-6.1687126, -7.540371, -7.045729, -6.7731266,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.104009, -6.2956376, -6.2331915, -5.849873...","[-6.6306915, -6.3750443, -6.4651437, -5.923516...","[0.30547088, 0.07410415, 0.24323413, 0.1472878..."
6,ExtraTreesRegressor,"[-6.349836803879999, -7.3017999999999965, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.432315328049999, -6.279000000000001, -6.2...","[-6.732743445288003, -6.458109803920001, -6.31...","[0.17284189895613972, 0.10728557845915476, 0.0..."
7,LinearRegression,"[-10.0, -3.4, -3.4, -10.0, -3.4, -10.0, -10.0,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-3.4, -3.4, -3.4, -5.850000000072768, -7.098...","[-3.4, -6.04, -3.4, -6.679999961802844, -6.779...","[0.0, 3.2333264604737955, 0.0, 1.6600000190985..."
8,KNeighborsRegressor,"[-6.28, -7.36, -6.98, -7.053333333333334, -6.4...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.105, -6.311666666666667, -5.786666...","[-6.544, -6.396000000000001, -6.23333333333333...","[0.22580620993330663, 0.15254726043645192, 0.1..."
9,SVR,"[-6.330826298217516, -7.1506412103693355, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.202990472359993, -6.167570926172878, -6.1...","[-6.4805247217803, -6.275593966124012, -6.2809...","[0.27780927912185815, 0.08268390735640099, 0.0..."


In [141]:
result_df.to_csv('results/Descriptors/Results_2D_padel_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_Caco2.csv')

In [142]:
#2d padel descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 1086)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1086)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017731 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 222063
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1023
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1909,0.3308,0.4369,0.7022,0.8380,0.8245,0.1412,0.2871,0.3758,0.7734,0.8809,0.8670
DecisionTreeRegressor,0.3693,0.4449,0.6077,0.4238,0.7166,0.6994,0.2061,0.3411,0.4540,0.6693,0.8214,0.8021
RandomForestRegressor,0.1930,0.3396,0.4393,0.6988,0.8384,0.8301,0.1542,0.2996,0.3926,0.7526,0.8719,0.8569
GradientBoostingRegressor,0.1965,0.3401,0.4433,0.6934,0.8330,0.8163,0.1477,0.2916,0.3843,0.7629,0.8747,0.8625
AdaBoostRegressor,0.2491,0.4070,0.4991,0.6113,0.7897,0.7624,0.2108,0.3717,0.4591,0.6617,0.8269,0.7881
XGBRegressor,0.2135,0.3462,0.4621,0.6668,0.8179,0.8070,0.1505,0.2911,0.3879,0.7585,0.8713,0.8563
ExtraTreesRegressor,0.1780,0.3161,0.4219,0.7222,0.8503,0.8485,0.1388,0.2832,0.3726,0.7772,0.8831,0.8699
LinearRegression,10.6503,3.0474,3.2635,-15.6182,0.0653,0.0482,3.1195,1.4222,1.7662,-4.0063,0.0150,0.0228
KNeighborsRegressor,0.2305,0.3579,0.4801,0.6403,0.8043,0.7871,0.1791,0.3117,0.4232,0.7126,0.8469,0.8207
SVR,0.2141,0.3514,0.4628,0.6659,0.8169,0.8024,0.1728,0.3216,0.4157,0.7226,0.8513,0.8336


In [143]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.358050576787527, -7.239684117601858, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5515026741745785, -6.412034917185415, -6....","[-6.737679234426139, -6.500143746923191, -6.56...","[0.11407076754980458, 0.11414741091902134, 0.0..."
1,DecisionTreeRegressor,"[-6.96, -7.05, -7.0, -7.0, -5.89, -6.77, -5.68...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.15, -6.24, -5.96, -5.85, -5.72, -7.06, -6...","[-6.380000000000001, -6.236, -6.94, -5.858, -5...","[0.6634154053080169, 0.1988567323476879, 0.685..."
2,RandomForestRegressor,"[-6.362383277659998, -7.2548999999999975, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.439784395019997, -6.297023552753334, -6.5...","[-6.542268765612, -6.458553008292666, -6.44686...","[0.0979487791647036, 0.1054593265284316, 0.068..."
3,GradientBoostingRegressor,"[-6.247685172407995, -7.137708016545235, -7.06...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.60245439562556, -6.195114417588883, -6.42...","[-6.782442203923521, -6.419842869171274, -6.59...","[0.18033168867065744, 0.1312929859358836, 0.10..."
4,AdaBoostRegressor,"[-6.058793692754673, -7.184276399793545, -7.14...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.143500452758068, -6.2054958836550975, -6....","[-6.395026076107295, -6.476754957462804, -6.22...","[0.17886133092299294, 0.2569859732555077, 0.10..."
5,XGBRegressor,"[-6.1687126, -7.540371, -7.045729, -6.7731266,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.104009, -6.2956376, -6.2331915, -5.849873...","[-6.6306915, -6.3750443, -6.4651437, -5.923516...","[0.30547088, 0.07410415, 0.24323413, 0.1472878..."
6,ExtraTreesRegressor,"[-6.308596294539999, -7.335599999999996, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.37693887716, -6.27075, -6.166079400079999...","[-6.779452822778005, -6.5189414457599995, -6.3...","[0.20235964800242226, 0.140990502902913, 0.103..."
7,LinearRegression,"[-10.0, -3.4, -3.4, -10.0, -3.4, -10.0, -10.0,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-3.4, -3.4, -3.4, -5.850000000137584, -7.098...","[-3.4, -6.04, -3.4, -6.680000009564435, -6.779...","[0.0, 3.2333264604737955, 0.0, 1.6599999952177..."
8,KNeighborsRegressor,"[-6.28, -7.36, -6.98, -7.053333333333334, -6.4...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.105, -6.311666666666667, -5.786666...","[-6.544, -6.396000000000001, -6.23333333333333...","[0.22580620993330663, 0.15254726043645192, 0.1..."
9,SVR,"[-6.330610697813137, -7.15060593618611, -7.082...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.20273498897294, -6.167366335801514, -6.11...","[-6.480458544031494, -6.275393998088708, -6.28...","[0.2779324930642154, 0.08255175419584457, 0.08..."


In [144]:
result_df.to_csv('results/Descriptors/Results_2D_padel_const_rem_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_const_rem_Caco2.csv')

In [145]:
#2d padel descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 758)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 758)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014776 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 148455
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 721
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1856,0.3234,0.4308,0.7104,0.8431,0.8338,0.1392,0.2856,0.3731,0.7766,0.8823,0.8698
DecisionTreeRegressor,0.3760,0.4508,0.6132,0.4134,0.7036,0.6767,0.1642,0.3083,0.4052,0.7365,0.8590,0.8517
RandomForestRegressor,0.1957,0.3395,0.4424,0.6946,0.8353,0.8268,0.1494,0.2941,0.3865,0.7602,0.8757,0.8615
GradientBoostingRegressor,0.2008,0.3433,0.4481,0.6868,0.8290,0.8128,0.1484,0.2944,0.3852,0.7619,0.8744,0.8600
AdaBoostRegressor,0.2547,0.4098,0.5047,0.6026,0.7825,0.7587,0.2132,0.3723,0.4617,0.6579,0.8238,0.7825
XGBRegressor,0.2087,0.3381,0.4568,0.6744,0.8221,0.8163,0.1476,0.2856,0.3841,0.7632,0.8740,0.8628
ExtraTreesRegressor,0.1761,0.3122,0.4197,0.7252,0.8521,0.8475,0.1412,0.2825,0.3757,0.7734,0.8805,0.8661
LinearRegression,1.6879,0.8808,1.2992,-1.6338,0.4258,0.5145,0.7042,0.6028,0.8391,-0.1301,0.5429,0.5850
KNeighborsRegressor,0.2241,0.3525,0.4733,0.6504,0.8110,0.7948,0.1693,0.3054,0.4114,0.7283,0.8565,0.8347
SVR,0.2110,0.3486,0.4594,0.6707,0.8198,0.8057,0.1679,0.3149,0.4097,0.7306,0.8557,0.8372


In [146]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.378493423598591, -7.3052660802741425, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.32580291312872, -6.594565162680331, -6.49...","[-6.661746991785033, -6.527593682818957, -6.54...","[0.18105201291595024, 0.07918352005555301, 0.1..."
1,DecisionTreeRegressor,"[-5.77, -8.0, -7.05, -7.698970004, -5.86999999...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.77, -6.54, -5.92, -5.85, -5.77, -7.0, -5....","[-6.562, -6.566, -7.0760000000000005, -5.90498...","[0.5995798528970102, 0.4625408090104053, 0.765..."
2,RandomForestRegressor,"[-6.434292652893333, -7.232786073149997, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.49641961952, -6.2611575749099995, -6.3319...","[-6.628656222287999, -6.40727765396, -6.378270...","[0.12593005672396326, 0.09777132968986614, 0.0..."
3,GradientBoostingRegressor,"[-6.44281443180094, -7.47564683361688, -7.1640...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.887574742245097, -6.274584246066608, -6.3...","[-6.8354914650001275, -6.419732845763869, -6.5...","[0.3125235612280081, 0.13559694162593358, 0.15..."
4,AdaBoostRegressor,"[-6.170644161091715, -7.04839714663321, -7.048...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.546622343538028, -6.546622343538028, -6.3...","[-6.552575987687216, -6.473546970763818, -6.18...","[0.16325425097129276, 0.18425779067845846, 0.0..."
5,XGBRegressor,"[-6.2097063, -7.503507, -7.4476924, -6.648505,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.520713, -6.3228407, -6.3116593, -5.849593...","[-6.68937, -6.3375235, -6.596836, -5.900491, -...","[0.11759273, 0.15857461, 0.14757602, 0.0999652..."
6,ExtraTreesRegressor,"[-6.130128985659997, -7.374899999999994, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.2119057219999965, -6.23433521083, -6.2428...","[-6.756524861794003, -6.5076261304199985, -6.3...","[0.2782729754184148, 0.13820327504772484, 0.07..."
7,LinearRegression,"[-7.392709908505366, -8.147302045611951, -10.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.987863092430529, -3.4, -7.656182176204601...","[-6.090727798805898, -4.886044010178833, -7.66...","[0.12869301724453402, 1.7144220449146104, 0.35..."
8,KNeighborsRegressor,"[-6.28, -7.36, -6.98, -7.053333333333334, -6.3...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31166666666666...","[-6.544, -6.476000000000001, -6.45933333333333...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.478714779147334, -7.169840640169275, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.378137868860178, -6.338170048090965, -6.1...","[-6.664228277664196, -6.434038863752299, -6.28...","[0.2744493415748487, 0.06281853649355863, 0.09..."


In [147]:
result_df.to_csv('results/Descriptors/Results_2D_padel_LVR_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_const_LVR_Caco2.csv')

In [148]:
#2d All descriptors
df_train_padel = pd.read_csv('features/Descriptors/Train_2d_padel_curated_Caco2.csv')
df_train_rdkit = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_Caco2.csv')
df_train_mordred = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')

df_2d_train = df_train_rdkit.merge(df_train_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_train

/tmp/ipykernel_1796054/719508899.py:4: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_mordred = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')


,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,15.976055,15.976055,0.030813,-1.935404,0.058779,22.883333,1664.156,...,6.547304,237.600778,1.980006,82.612830,37.983553,44.629277,94190.0,208.0,8.466,600.0
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,16.083777,16.083777,0.027668,-1.934865,0.053531,23.411765,1650.129,...,6.569720,235.601472,1.979844,82.357901,37.975600,44.382301,90388.0,207.0,8.692,596.0
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,15.668220,15.668220,0.003714,-4.150777,0.085546,24.018349,1587.863,...,6.894849,214.100758,1.964227,89.673466,40.258888,41.162462,69754.0,181.0,6.958,540.0
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,16.048909,16.048909,0.025704,-1.891709,0.101375,24.159292,1574.031,...,6.526840,222.500266,1.969029,82.337179,37.880967,44.456212,79884.0,202.0,6.376,566.0
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,15.511700,15.511700,0.033138,-4.150544,0.085338,23.366972,1567.445,...,6.720471,214.421483,1.967170,87.553124,40.278853,41.547594,69806.0,179.0,7.107,536.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,13.038711,13.038711,0.105820,-0.784997,0.662387,22.258065,430.549,...,6.619354,61.528299,1.984784,22.437229,10.154168,12.283061,2632.0,45.0,2.298,148.0
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,13.057052,13.057052,0.142991,-0.802728,0.571252,24.322581,430.549,...,6.619354,61.528106,1.984778,22.197058,10.153913,12.043145,2620.0,45.0,3.111,148.0
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,12.835172,12.835172,0.168936,-0.728726,0.606745,24.965517,402.539,...,6.385128,57.707840,1.989926,19.684197,7.618398,12.065799,2286.0,42.0,3.395,138.0
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,12.763992,12.763992,0.179945,-0.726122,0.611430,25.000000,388.512,...,6.470791,55.707630,1.989558,19.686908,7.618921,12.067987,2069.0,41.0,3.037,134.0


In [149]:
df_2d_train.to_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv', index=False)

In [150]:
df_test_padel = pd.read_csv('features/Descriptors/Test_2d_padel_curated_Caco2.csv')
df_test_rdkit = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_Caco2.csv')
df_test_mordred = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')

df_2d_test = df_test_rdkit.merge(df_test_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_test

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,16.271931,16.271931,0.025756,-1.930898,0.058792,23.408333,1664.156,...,6.547304,237.423604,1.978530,82.566872,37.965469,44.601403,91297.0,211.0,8.405,602.0
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,15.516152,15.516152,0.000091,-1.679143,0.049536,24.360360,1572.049,...,6.309318,217.445869,1.958972,88.222714,50.298796,37.923918,74798.0,175.0,6.768,538.0
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,15.993973,15.993973,0.020362,-1.750211,0.057933,24.294643,1552.024,...,6.516604,222.331854,1.985106,74.211794,35.995049,38.216745,73336.0,187.0,10.339,562.0
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,15.804049,15.804049,0.002346,-2.970606,0.165351,26.981481,1537.344,...,6.736287,213.827378,1.979883,85.394192,35.950728,41.996280,67356.0,187.0,6.203,554.0
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,15.545371,15.545371,0.013183,-2.948188,0.126607,27.422018,1530.953,...,6.538221,215.983270,1.981498,82.730212,35.983356,41.825434,69591.0,185.0,8.843,556.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,12.958074,12.958074,0.156212,-0.733388,0.586385,24.903226,430.593,...,6.236151,61.708009,1.990581,19.682164,7.618006,12.064158,2750.0,44.0,4.111,146.0
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,13.040403,13.040403,0.016783,-0.701825,0.680196,22.838710,430.593,...,6.236151,61.707604,1.990568,20.026186,7.618719,12.407467,2714.0,44.0,2.941,146.0
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,13.114656,13.114656,0.186152,-0.689890,0.586385,24.903226,430.593,...,6.236151,61.707690,1.990571,19.711184,7.642415,12.068770,2660.0,44.0,4.111,146.0
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,13.101637,13.101637,0.172241,-0.755076,0.586385,24.903226,430.593,...,6.236151,61.709088,1.990616,19.692320,7.625771,12.066549,2660.0,44.0,4.111,146.0


In [151]:
df_2d_test.to_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv', index=False)

In [153]:
#2d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv')
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/252792062.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
/tmp/ipykernel_1796054/252792062.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 3090)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 3090)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.091274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 500189
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2339
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1847,0.3288,0.4298,0.7118,0.8438,0.8352,0.1371,0.2807,0.3703,0.7800,0.8848,0.8730
DecisionTreeRegressor,0.4048,0.4730,0.6362,0.3684,0.6865,0.6612,0.1903,0.3280,0.4363,0.6946,0.8351,0.8254
RandomForestRegressor,0.1918,0.3356,0.4380,0.7007,0.8394,0.8333,0.1487,0.2956,0.3856,0.7614,0.8768,0.8626
GradientBoostingRegressor,0.1898,0.3327,0.4357,0.7038,0.8395,0.8261,0.1420,0.2860,0.3769,0.7721,0.8802,0.8662
AdaBoostRegressor,0.2446,0.3998,0.4946,0.6184,0.7929,0.7617,0.2035,0.3614,0.4511,0.6734,0.8342,0.7958
XGBRegressor,0.2044,0.3398,0.4522,0.6810,0.8257,0.8167,0.1469,0.2911,0.3833,0.7642,0.8745,0.8685
ExtraTreesRegressor,0.1797,0.3132,0.4239,0.7196,0.8486,0.8467,0.1332,0.2776,0.3649,0.7863,0.8884,0.8761
LinearRegression,10.7969,3.0627,3.2859,-15.8469,0.0418,0.0138,2.9768,1.4504,1.7253,-3.7774,-0.0810,-0.0612
KNeighborsRegressor,0.2267,0.3562,0.4761,0.6463,0.8073,0.7918,0.1756,0.3089,0.4190,0.7182,0.8498,0.8231
SVR,0.2110,0.3470,0.4594,0.6707,0.8200,0.8089,0.1675,0.3163,0.4093,0.7312,0.8572,0.8419


In [154]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.406845135140703, -7.257900168141617, -7.15...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.359634404583435, -6.773394457747328, -6.4...","[-6.660697961922115, -6.666293163903013, -6.48...","[0.1588348663332401, 0.12508236082848403, 0.05..."
1,DecisionTreeRegressor,"[-6.59, -7.03, -7.0, -7.03, -5.85, -6.55, -5.6...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.27, -7.57, -5.92, -5.85, -5.89, -7.0, -5....","[-6.448, -6.65, -6.174000000000001, -5.736, -5...","[0.6303776645789414, 0.6529624797796579, 0.274..."
2,RandomForestRegressor,"[-6.377186058469999, -7.268620599909997, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.415550692809998, -6.339138827900001, -6.3...","[-6.563880176459999, -6.4344876699626665, -6.4...","[0.09033932264852147, 0.06130030954081356, 0.1..."
3,GradientBoostingRegressor,"[-6.623266089479321, -7.245672309810108, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6052495479652515, -6.362608644102881, -6....","[-6.778592272041214, -6.4607144039011875, -6.5...","[0.13698378086025734, 0.1533360415776074, 0.13..."
4,AdaBoostRegressor,"[-6.251, -7.275901781678159, -7.20341176470588...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.144788889515147, -6.468714285714288, -6.4...","[-6.337921560227554, -6.612990951131006, -6.25...","[0.16846105593588293, 0.2281364337322519, 0.12..."
5,XGBRegressor,"[-6.736552, -7.326673, -7.2021484, -6.6067653,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.502589, -6.4387417, -6.4571443, -5.849794...","[-6.542516, -6.398233, -6.328965, -5.8606143, ...","[0.1759971, 0.22252488, 0.13664219, 0.02121713..."
6,ExtraTreesRegressor,"[-6.43024224299, -7.341599999999996, -6.969078...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.516087980769999, -6.4904261703, -6.359726...","[-6.768690938556003, -6.475327907723333, -6.36...","[0.14354810971209003, 0.03509080831653249, 0.0..."
7,LinearRegression,"[-10.0, -10.0, -3.4, -3.4, -10.0, -3.4, -10.0,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -3.4, -3.4, -3.4, -3.4, -3.4, -10.0, ...","[-6.039999999999999, -4.72, -4.72, -4.72, -4.7...","[3.2333264604737955, 2.64, 2.64, 2.64, 2.64, 3..."
8,KNeighborsRegressor,"[-6.28, -7.343333333333334, -6.98, -7.03666666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31166666666666...","[-6.544, -6.476000000000001, -6.24933333333333...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.290734429095297, -7.1529091561073574, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.186506633074858, -6.247212257258087, -6.1...","[-6.668328613702235, -6.348138986300372, -6.27...","[0.28540870367165594, 0.07027043737190647, 0.0..."


In [155]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_Caco2.csv')

In [156]:
#2d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/914900448.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')


X_train shape:  (1007, 2469)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_1796054/914900448.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_test shape:  (252, 2469)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.102196 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 500189
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2339
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1847,0.3288,0.4298,0.7118,0.8438,0.8352,0.1371,0.2807,0.3703,0.7800,0.8848,0.8730
DecisionTreeRegressor,0.3808,0.4583,0.6171,0.4058,0.7049,0.6783,0.1930,0.3305,0.4393,0.6903,0.8318,0.8180
RandomForestRegressor,0.1908,0.3356,0.4368,0.7023,0.8403,0.8337,0.1496,0.2965,0.3868,0.7599,0.8760,0.8624
GradientBoostingRegressor,0.1907,0.3330,0.4367,0.7024,0.8386,0.8254,0.1420,0.2860,0.3768,0.7722,0.8804,0.8658
AdaBoostRegressor,0.2395,0.3969,0.4893,0.6264,0.7990,0.7741,0.2097,0.3671,0.4579,0.6635,0.8285,0.7831
XGBRegressor,0.2044,0.3398,0.4522,0.6810,0.8257,0.8167,0.1469,0.2911,0.3833,0.7642,0.8745,0.8685
ExtraTreesRegressor,0.1755,0.3101,0.4189,0.7262,0.8525,0.8505,0.1327,0.2779,0.3643,0.7870,0.8887,0.8774
LinearRegression,10.7969,3.0627,3.2859,-15.8469,0.0418,0.0138,2.9768,1.4504,1.7253,-3.7774,-0.0810,-0.0612
KNeighborsRegressor,0.2267,0.3562,0.4761,0.6463,0.8073,0.7918,0.1756,0.3089,0.4190,0.7182,0.8498,0.8231
SVR,0.2110,0.3470,0.4594,0.6707,0.8200,0.8089,0.1675,0.3163,0.4093,0.7312,0.8572,0.8420


In [157]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.406845135140703, -7.257900168141617, -7.15...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.359634404583435, -6.773394457747328, -6.4...","[-6.660697961922115, -6.666293163903013, -6.48...","[0.1588348663332401, 0.12508236082848403, 0.05..."
1,DecisionTreeRegressor,"[-6.0, -7.0, -7.0, -7.0, -6.11, -6.06, -5.4, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -7.68, -5.92, -5.85, -5.89, -7.18, -5...","[-6.3420000000000005, -6.456, -6.042, -5.789, ...","[0.6640602382314424, 0.7548668756807387, 0.165..."
2,RandomForestRegressor,"[-6.370638420269996, -7.2438490195999945, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.351700641069999, -6.433014492829998, -6.3...","[-6.555848015752001, -6.453497451377999, -6.43...","[0.10441569075431174, 0.063070429971284, 0.123..."
3,GradientBoostingRegressor,"[-6.697118719842148, -7.245672309810109, -7.04...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.605249547965252, -6.355022672014547, -6.4...","[-6.787453368396532, -6.449668209936661, -6.55...","[0.12708521667118447, 0.17948051269431137, 0.1..."
4,AdaBoostRegressor,"[-6.650238095238092, -7.315201078173079, -7.20...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.22814872865104, -6.66507007577049, -6.276...","[-6.324739199785346, -6.49325249823089, -6.217...","[0.07441652823901139, 0.21351017612249976, 0.1..."
5,XGBRegressor,"[-6.736552, -7.326673, -7.2021484, -6.6067653,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.502589, -6.4387417, -6.4571443, -5.849794...","[-6.542516, -6.398233, -6.328965, -5.8606143, ...","[0.1759971, 0.22252488, 0.13664219, 0.02121713..."
6,ExtraTreesRegressor,"[-6.538366110700002, -7.322349019599996, -7.01...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.57938638307, -6.329210299959999, -6.30889...","[-6.799550209706004, -6.5032518639119985, -6.3...","[0.11741523992393672, 0.09107211779920728, 0.0..."
7,LinearRegression,"[-10.0, -10.0, -3.4, -3.4, -10.0, -3.4, -10.0,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -3.4, -3.4, -3.4, -3.4, -3.4, -10.0, ...","[-6.039999999999999, -4.72, -4.72, -4.72, -4.7...","[3.2333264604737955, 2.64, 2.64, 2.64, 2.64, 3..."
8,KNeighborsRegressor,"[-6.28, -7.343333333333334, -6.98, -7.03666666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31166666666666...","[-6.544, -6.476000000000001, -6.24933333333333...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.290905792877216, -7.152986108792946, -7.06...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.1867060494014, -6.247363811272828, -6.134...","[-6.668341260728917, -6.348090469368334, -6.27...","[0.2853076544989516, 0.07021347803514166, 0.08..."


In [158]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_const_rem_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_const_rem_Caco2.csv')

In [159]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/1997117473.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
/tmp/ipykernel_1796054/1997117473.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 1753)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1753)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060927 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 334530
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1665
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1849,0.3248,0.4300,0.7115,0.8437,0.8327,0.1380,0.2850,0.3714,0.7786,0.8835,0.8708
DecisionTreeRegressor,0.3656,0.4457,0.6046,0.4296,0.7138,0.6956,0.1663,0.3139,0.4078,0.7332,0.8575,0.8476
RandomForestRegressor,0.1936,0.3371,0.4400,0.6979,0.8373,0.8302,0.1472,0.2925,0.3837,0.7638,0.8781,0.8649
GradientBoostingRegressor,0.1918,0.3346,0.4379,0.7008,0.8377,0.8236,0.1444,0.2878,0.3800,0.7683,0.8781,0.8649
AdaBoostRegressor,0.2479,0.4062,0.4979,0.6132,0.7903,0.7604,0.2140,0.3720,0.4626,0.6565,0.8222,0.7664
XGBRegressor,0.2069,0.3434,0.4549,0.6771,0.8243,0.8182,0.1413,0.2797,0.3759,0.7733,0.8796,0.8657
ExtraTreesRegressor,0.1755,0.3102,0.4189,0.7262,0.8525,0.8495,0.1354,0.2761,0.3680,0.7826,0.8858,0.8744
LinearRegression,11.0526,3.1087,3.3246,-16.2460,0.0068,-0.0178,2.2889,1.2437,1.5129,-2.6734,0.0148,-0.0199
KNeighborsRegressor,0.2222,0.3539,0.4714,0.6533,0.8116,0.7966,0.1667,0.3029,0.4083,0.7324,0.8579,0.8342
SVR,0.2081,0.3457,0.4562,0.6753,0.8228,0.8132,0.1629,0.3111,0.4035,0.7386,0.8614,0.8459


In [160]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.412603992887125, -7.332383460507817, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.189224616098255, -6.582959703796662, -6.3...","[-6.643954124847977, -6.681642012457959, -6.49...","[0.24702697916379987, 0.11041946362447899, 0.1..."
1,DecisionTreeRegressor,"[-6.21, -7.03, -7.0, -8.0, -6.28, -6.85, -6.54...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -7.85, -6.89, -5.85, -5.85, -7.301029...","[-6.324, -6.901999999999999, -6.764, -6.05, -5...","[0.7166756588583154, 0.6323100505290105, 0.305..."
2,RandomForestRegressor,"[-6.4101360731499994, -7.207199999999998, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.318593409839999, -6.428137416509998, -6.3...","[-6.58684778474, -6.496464356926, -6.427676417...","[0.1574424666496804, 0.09184695552178423, 0.04..."
3,GradientBoostingRegressor,"[-6.8394968922117005, -7.1411170614203066, -6....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.617846340171105, -6.469295570657155, -6.3...","[-6.835892768367873, -6.547582548278794, -6.54...","[0.26397380868434167, 0.26782483687080944, 0.1..."
4,AdaBoostRegressor,"[-6.38431818181818, -7.185953333425287, -7.115...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.434090909090902, -6.221439393939395, -6.2...","[-6.518656777237872, -6.539884987870534, -6.21...","[0.16610272901100606, 0.2736027059625638, 0.13..."
5,XGBRegressor,"[-6.032533, -7.241788, -7.35, -7.009788, -6.17...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.966042, -6.4702015, -6.4911194, -5.849868...","[-6.4025154, -6.50355, -6.407956, -5.838879, -...","[0.40334326, 0.15317671, 0.16591857, 0.0224714..."
6,ExtraTreesRegressor,"[-6.388729829099999, -7.254486073149994, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.459937648869998, -6.338100000000001, -6.2...","[-6.7289992219320025, -6.452770222710001, -6.3...","[0.16605500922343538, 0.0581038991842704, 0.11..."
7,LinearRegression,"[-10.0, -3.4, -3.4, -10.0, -3.4, -3.4, -10.0, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -3.4, -10.0, -10.0, -10.0, -3.4, -10....","[-7.359999999999999, -4.720000000000001, -8.68...","[3.2333264604737946, 2.64, 2.64, 3.23332646047..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31166666666666...","[-6.544, -6.476000000000001, -6.45933333333333...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.4170219438308855, -7.175057586293216, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.349609422193352, -6.345157399710396, -6.1...","[-6.7972756937864505, -6.431108772953171, -6.3...","[0.2498380787559364, 0.05240892690362041, 0.09..."


In [161]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_LVR_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_LVR_Caco2.csv')

In [162]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/1997117473.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
/tmp/ipykernel_1796054/1997117473.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 1753)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1753)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.061498 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 334530
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1665
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1849,0.3248,0.4300,0.7115,0.8437,0.8327,0.1380,0.2850,0.3714,0.7786,0.8835,0.8708
DecisionTreeRegressor,0.3656,0.4457,0.6046,0.4296,0.7138,0.6956,0.1663,0.3139,0.4078,0.7332,0.8575,0.8476
RandomForestRegressor,0.1936,0.3371,0.4400,0.6979,0.8373,0.8302,0.1472,0.2925,0.3837,0.7638,0.8781,0.8649
GradientBoostingRegressor,0.1918,0.3346,0.4379,0.7008,0.8377,0.8236,0.1444,0.2878,0.3800,0.7683,0.8781,0.8649
AdaBoostRegressor,0.2479,0.4062,0.4979,0.6132,0.7903,0.7604,0.2140,0.3720,0.4626,0.6565,0.8222,0.7664
XGBRegressor,0.2069,0.3434,0.4549,0.6771,0.8243,0.8182,0.1413,0.2797,0.3759,0.7733,0.8796,0.8657
ExtraTreesRegressor,0.1755,0.3102,0.4189,0.7262,0.8525,0.8495,0.1354,0.2761,0.3680,0.7826,0.8858,0.8744
LinearRegression,11.0526,3.1087,3.3246,-16.2460,0.0068,-0.0178,2.2889,1.2437,1.5129,-2.6734,0.0148,-0.0199
KNeighborsRegressor,0.2222,0.3539,0.4714,0.6533,0.8116,0.7966,0.1667,0.3029,0.4083,0.7324,0.8579,0.8342
SVR,0.2081,0.3457,0.4562,0.6753,0.8228,0.8132,0.1629,0.3111,0.4035,0.7386,0.8614,0.8459


In [163]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.412603992887125, -7.332383460507817, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.189224616098255, -6.582959703796662, -6.3...","[-6.643954124847977, -6.681642012457959, -6.49...","[0.24702697916379987, 0.11041946362447899, 0.1..."
1,DecisionTreeRegressor,"[-6.21, -7.03, -7.0, -8.0, -6.28, -6.85, -6.54...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -7.85, -6.89, -5.85, -5.85, -7.301029...","[-6.324, -6.901999999999999, -6.764, -6.05, -5...","[0.7166756588583154, 0.6323100505290105, 0.305..."
2,RandomForestRegressor,"[-6.410136073149997, -7.207199999999996, -6.98...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.318593409839998, -6.42813741651, -6.35920...","[-6.58684778474, -6.496464356926, -6.427676417...","[0.15744246664968045, 0.09184695552178293, 0.0..."
3,GradientBoostingRegressor,"[-6.8394968922117005, -7.1411170614203066, -6....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.617846340171105, -6.469295570657155, -6.3...","[-6.835892768367873, -6.547582548278794, -6.54...","[0.26397380868434167, 0.26782483687080944, 0.1..."
4,AdaBoostRegressor,"[-6.38431818181818, -7.185953333425287, -7.115...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.434090909090902, -6.221439393939395, -6.2...","[-6.518656777237872, -6.539884987870534, -6.21...","[0.16610272901100606, 0.2736027059625638, 0.13..."
5,XGBRegressor,"[-6.032533, -7.241788, -7.35, -7.009788, -6.17...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.966042, -6.4702015, -6.4911194, -5.849868...","[-6.4025154, -6.50355, -6.407956, -5.838879, -...","[0.40334326, 0.15317671, 0.16591857, 0.0224714..."
6,ExtraTreesRegressor,"[-6.388729829099999, -7.254486073149994, -7.02...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.459937648869999, -6.338100000000001, -6.2...","[-6.7289992219320025, -6.452770222710001, -6.3...","[0.16605500922343555, 0.058103899184270384, 0...."
7,LinearRegression,"[-10.0, -3.4, -3.4, -10.0, -3.4, -3.4, -10.0, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -3.4, -10.0, -10.0, -10.0, -3.4, -10....","[-7.359999999999999, -4.720000000000001, -8.68...","[3.2333264604737946, 2.64, 2.64, 3.23332646047..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31166666666666...","[-6.544, -6.476000000000001, -6.45933333333333...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.4170219438308855, -7.175057586293216, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.349609422193352, -6.345157399710396, -6.1...","[-6.7972756937864505, -6.431108772953171, -6.3...","[0.2498380787559364, 0.05240892690362041, 0.09..."


In [164]:
def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [165]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [166]:
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
X_train = df_train[selected_features] 
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv')
df_test =df_test.dropna()
X_test =  df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test =  df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/3082500535.py:1: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')


X_train shape:  (1007, 232)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 232)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005866 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 46939
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 222
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1967,0.3347,0.4435,0.6931,0.8327,0.8235,0.1479,0.2919,0.3846,0.7627,0.8749,0.8593
DecisionTreeRegressor,0.3699,0.4410,0.6082,0.4228,0.7081,0.6887,0.1952,0.3229,0.4418,0.6867,0.8311,0.8021
RandomForestRegressor,0.1979,0.3424,0.4448,0.6912,0.8339,0.8237,0.1635,0.3082,0.4043,0.7376,0.8626,0.8435
GradientBoostingRegressor,0.2011,0.3433,0.4484,0.6863,0.8291,0.8169,0.1652,0.3130,0.4064,0.7349,0.8592,0.8388
AdaBoostRegressor,0.2630,0.4200,0.5128,0.5896,0.7747,0.7405,0.2292,0.3876,0.4787,0.6322,0.8073,0.7604
XGBRegressor,0.2164,0.3492,0.4652,0.6623,0.8145,0.8063,0.1488,0.2912,0.3858,0.7612,0.8737,0.8588
ExtraTreesRegressor,0.1804,0.3180,0.4247,0.7186,0.8481,0.8409,0.1449,0.2879,0.3807,0.7674,0.8775,0.8622
LinearRegression,0.3355,0.4097,0.5792,0.4766,0.7388,0.7704,0.3029,0.3700,0.5503,0.5140,0.7593,0.8149
KNeighborsRegressor,0.2277,0.3542,0.4771,0.6448,0.8089,0.7944,0.1663,0.3014,0.4078,0.7331,0.8593,0.8410
SVR,0.1942,0.3344,0.4407,0.6970,0.8355,0.8288,0.1672,0.3121,0.4089,0.7317,0.8565,0.8418


In [167]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.0363561397268395, -7.338029189520495, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.153856192063611, -6.41261822643857, -6.13...","[-6.514183218358809, -6.58649543539247, -6.438...","[0.19698003028636538, 0.11536960087321771, 0.1..."
1,DecisionTreeRegressor,"[-6.03, -8.0, -7.05, -8.0, -5.89, -6.6, -5.03,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.85, -5.835, -5.1, -5.85, -5.68, -8.0, -6....","[-6.8782059992, -6.247, -6.607925890999999, -6...","[0.48264897499889253, 0.9305460762369591, 0.87..."
2,RandomForestRegressor,"[-6.2791989729500015, -7.271449019599999, -6.9...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.263630433649997, -6.479588719639997, -6.3...","[-6.4854280775133315, -6.538473176125999, -6.3...","[0.11308211526376831, 0.09518961807953233, 0.0..."
3,GradientBoostingRegressor,"[-6.304252657527587, -7.4515495750768235, -7.1...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.443805945002247, -6.451373187157688, -6.1...","[-6.637316602279277, -6.67264360858343, -6.474...","[0.12009459304298994, 0.25554756766314396, 0.1..."
4,AdaBoostRegressor,"[-6.377492063492074, -7.210231646139521, -6.82...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.377492063492074, -6.679930034755817, -6.1...","[-6.4126238720529996, -6.669360544546768, -6.2...","[0.2332963964051376, 0.08804228302625139, 0.11..."
5,XGBRegressor,"[-6.4045477, -7.400424, -7.129481, -6.686027, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.3491974, -6.5181456, -6.2067423, -5.84959...","[-6.4696045, -6.372979, -6.4455023, -5.8734713...","[0.1878237, 0.19276375, 0.16895148, 0.04726866..."
6,ExtraTreesRegressor,"[-6.104691759729998, -7.494099999999998, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.240515689569998, -6.530391003480002, -6.0...","[-6.5563915427260016, -6.649600350640003, -6.2...","[0.17401341374837936, 0.12991695878274057, 0.0..."
7,LinearRegression,"[-10.0, -7.4751284622721865, -6.98296362996588...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-10.0, -6.399786653361839, -5.78434876621627...","[-7.2259581002312245, -6.594868358190368, -5.7...","[1.3908573717115078, 0.32108184907430354, 0.02..."
8,KNeighborsRegressor,"[-6.28, -7.3066666666666675, -6.98, -7.0533333...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.4383333333333335, -5.9616666666666...","[-6.544, -6.304, -5.997333333333333, -5.834, -...","[0.22580620993330663, 0.12591708029934245, 0.0..."
9,SVR,"[-6.109107034814453, -7.221288767449276, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.1040160531715575, -6.16599329075138, -6.0...","[-6.756321994087867, -6.31159464690867, -6.202...","[0.3278833672115252, 0.0880273111487787, 0.071..."


In [168]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_LVR_remove_corr_features_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_LVRremove_corr_features_Caco2.csv')

In [169]:
#3d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc_Caco2.csv')
df_train = df_train.fillna(0)
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc_Caco2.csv')
df_test = df_test.fillna(0)
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 11)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 11)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000665 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 11
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.26770455882508315


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5084,0.5973,0.7130,0.2067,0.4751,0.3587,0.4468,0.5552,0.6684,0.2830,0.5420,0.3701
DecisionTreeRegressor,0.8859,0.7357,0.9412,-0.3822,0.3226,0.2901,0.5816,0.6239,0.7626,0.0666,0.4079,0.2401
RandomForestRegressor,0.4878,0.5876,0.6984,0.2389,0.4976,0.3883,0.4370,0.5527,0.6611,0.2987,0.5527,0.3809
GradientBoostingRegressor,0.4844,0.5849,0.6960,0.2442,0.4977,0.3734,0.4378,0.5625,0.6616,0.2975,0.5487,0.3801
AdaBoostRegressor,0.4975,0.6165,0.7054,0.2237,0.4811,0.3029,0.4587,0.5844,0.6772,0.2639,0.5282,0.3462
XGBRegressor,0.5800,0.6306,0.7616,0.0950,0.4238,0.3257,0.4671,0.5692,0.6835,0.2503,0.5262,0.3678
ExtraTreesRegressor,0.5069,0.5971,0.7120,0.2090,0.4822,0.3811,0.4555,0.5628,0.6749,0.2690,0.5375,0.3706
LinearRegression,0.4856,0.5938,0.6968,0.2424,0.4928,0.4061,0.4566,0.5664,0.6758,0.2672,0.5175,0.3766
KNeighborsRegressor,0.6327,0.6461,0.7955,0.0127,0.3620,0.2822,0.4971,0.5769,0.7050,0.2022,0.4846,0.3483
SVR,0.5337,0.6012,0.7306,0.1672,0.4368,0.3372,0.4424,0.5473,0.6651,0.2900,0.5401,0.4172


In [170]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.072478366219209, -6.192846656369473, -6.47...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.035403250471146, -6.776186989699203, -6.4...","[-6.203797359595262, -6.881397362202351, -6.58...","[0.23663696271774307, 0.17573930408520713, 0.1..."
1,DecisionTreeRegressor,"[-5.96, -6.207608311, -7.03, -6.38, -5.74, -5....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.96, -7.05, -6.21, -6.92, -8.0, -5.85, -6....","[-6.145999999999999, -6.552, -6.372, -6.531794...","[0.7956783269638554, 0.5228154550125694, 0.470..."
2,RandomForestRegressor,"[-6.267811431909996, -6.047586993969998, -6.99...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.117720644459995, -7.0238999999999985, -6....","[-6.263880960201996, -6.830519067161999, -6.71...","[0.18201410564154558, 0.22120266087605736, 0.1..."
3,GradientBoostingRegressor,"[-6.542691849261779, -6.392191172350719, -6.96...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.01815215618002, -7.026244067822431, -6.79...","[-6.160679786536041, -6.8718023477825625, -6.6...","[0.16336857458363765, 0.14287793472731938, 0.1..."
4,AdaBoostRegressor,"[-6.44476906173574, -6.44861486791681, -6.4721...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.42005618517398, -6.472157560192303, -6.44...","[-6.421633769350595, -6.508227638874553, -6.45...","[0.06349517111745695, 0.03254261572523169, 0.0..."
5,XGBRegressor,"[-7.11254, -6.0059676, -7.298164, -6.435168, -...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.0788593, -7.3230376, -6.7497454, -6.25737...","[-6.1288195, -6.96568, -6.4672995, -6.3764915,...","[0.34332985, 0.21775055, 0.15314557, 0.1400962..."
6,ExtraTreesRegressor,"[-6.635384465720002, -6.038608664879998, -6.82...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.168135965519999, -6.96795757491, -6.78285...","[-6.374487885839999, -6.938091514982, -6.72460...","[0.2214425191424685, 0.1420208325701068, 0.160..."
7,LinearRegression,"[-5.72547815369699, -5.974102032567178, -6.166...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.771497114166061, -6.104630078554194, -6.0...","[-5.923874666660731, -6.149250400044349, -6.10...","[0.0862938843044838, 0.04997001047776967, 0.04..."
8,KNeighborsRegressor,"[-6.28, -5.896666666666667, -6.269645131, -5.7...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -7.37, -6.170000000000001, -6.1133333...","[-6.458666666666668, -7.299333333333332, -6.20...","[0.2349430190965936, 0.06045751492669135, 0.17..."
9,SVR,"[-5.930296940248346, -5.943346010172674, -6.06...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.963722612259093, -6.149690676097477, -6.2...","[-6.436848973163218, -6.203621494545009, -6.18...","[0.37330398865704567, 0.10116455345280083, 0.0..."


In [171]:
result_df.to_csv('results/Descriptors/Results_3D_RDKit_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_RDKit_desc_Caco2.csv')

In [172]:
#3d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_curated_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_curated_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 431)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 431)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.087655 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 109905
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 431
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2942,0.4307,0.5424,0.5410,0.7355,0.6980,0.2307,0.3767,0.4804,0.6297,0.7955,0.7632
DecisionTreeRegressor,0.6341,0.5980,0.7963,0.0106,0.5070,0.4796,0.3265,0.4317,0.5714,0.4760,0.6967,0.6283
RandomForestRegressor,0.3079,0.4481,0.5549,0.5196,0.7213,0.6778,0.2710,0.4084,0.5206,0.5651,0.7546,0.7170
GradientBoostingRegressor,0.3047,0.4400,0.5520,0.5246,0.7243,0.6789,0.2435,0.3939,0.4934,0.6092,0.7831,0.7265
AdaBoostRegressor,0.3599,0.5045,0.6000,0.4384,0.6692,0.6073,0.3239,0.4765,0.5691,0.4802,0.7080,0.6504
XGBRegressor,0.3364,0.4521,0.5800,0.4751,0.6944,0.6481,0.2466,0.3880,0.4966,0.6043,0.7789,0.7401
ExtraTreesRegressor,0.2875,0.4332,0.5362,0.5514,0.7449,0.7041,0.2496,0.4018,0.4996,0.5995,0.7803,0.7325
LinearRegression,0.5699,0.5803,0.7549,0.1108,0.5951,0.5981,0.3982,0.5058,0.6310,0.3609,0.6759,0.6471
KNeighborsRegressor,0.4209,0.5074,0.6488,0.3432,0.6135,0.5394,0.2930,0.4271,0.5413,0.5297,0.7332,0.6553
SVR,0.3204,0.4519,0.5661,0.5000,0.7109,0.6617,0.2567,0.4007,0.5066,0.5881,0.7686,0.7048


In [173]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.55623280048068, -7.014543135459498, -7.072...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6095226673042475, -6.2713139360659715, -6...","[-6.686151081289141, -6.467575751849682, -6.58...","[0.1179765272482484, 0.12799384703068656, 0.13..."
1,DecisionTreeRegressor,"[-7.24, -7.08, -8.0, -5.94, -6.64, -6.66, -5.8...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -6.0, -6.0, -6.06, -5.96, -5.96, -5.7...","[-7.07, -6.304, -6.248, -6.018000000000001, -6...","[0.782048591840686, 0.5648929102051115, 0.3451..."
2,RandomForestRegressor,"[-6.7480945075199985, -6.606899999999997, -6.8...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.758367982759999, -6.490049999999999, -6.5...","[-6.68640487249, -6.539914396539999, -6.538537...","[0.09296788232893971, 0.07761015546825088, 0.0..."
3,GradientBoostingRegressor,"[-7.036131208799436, -6.711851643685903, -6.75...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.8821649366420115, -6.31302752723645, -6.6...","[-6.894773082743953, -6.72346032480088, -6.808...","[0.21105071073184883, 0.248419172870093, 0.171..."
4,AdaBoostRegressor,"[-6.371293103448278, -6.799738372093021, -6.84...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.371293103448278, -6.169646302250809, -6.5...","[-6.481550947801432, -6.336997066818453, -6.73...","[0.21353959800105285, 0.11275859615927983, 0.2..."
5,XGBRegressor,"[-6.3922195, -7.2661233, -7.271785, -6.12003, ...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.162653, -6.018799, -6.3532248, -6.1190825...","[-6.6930633, -6.3389173, -6.401063, -5.8260508...","[0.40294725, 0.18520848, 0.13183346, 0.2094717..."
6,ExtraTreesRegressor,"[-6.588589700040003, -6.820167982760003, -6.91...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.676159399210004, -6.550150000000001, -6.6...","[-6.7639478730560025, -6.582290000000002, -6.7...","[0.08473659250379535, 0.08637001447261752, 0.0..."
7,LinearRegression,"[-5.191831741405009, -8.267713363291547, -6.07...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.389431953120959, -7.638736623865496, -5.4...","[-6.295231946734479, -7.44145409064785, -5.180...","[0.5745390862469235, 0.5821649603539887, 0.344..."
8,KNeighborsRegressor,"[-6.28, -6.233333333333333, -7.023333333333333...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.6866666666666665, -6.27, -6.176666...","[-6.544, -6.538333333333332, -6.27733333333333...","[0.22580620993330663, 0.32924830211322936, 0.1..."
9,SVR,"[-6.313954654585109, -6.475591349469363, -6.59...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.424938277446356, -6.023858520189363, -6.1...","[-6.566806264479861, -6.159209150627888, -6.17...","[0.18155032729436077, 0.11958649754469526, 0.1..."


In [174]:
result_df.to_csv('results/Descriptors/Results_3D_padel_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_padel_desc_Caco2.csv')

In [175]:
df_train_rdkit = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc_Caco2.csv')
df_train_rdkit = df_train_rdkit.fillna(0)
df_train_padel = pd.read_csv('features/Descriptors/Train_3d_padel_curated_Caco2.csv')

df_3d_descriptors = df_train_rdkit.merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,40359.396070,100703.602332,131887.957261,0.306013,0.763554,9.055872,0.000019,...,0.682480,0.263316,0.443174,0.407327,0.382690,86.049198,1710.236099,8002.655598,0.523721,1.233191
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,46241.303376,91821.486727,125376.707133,0.368819,0.732365,8.934423,0.000016,...,0.649321,0.261240,0.473762,0.435183,0.424702,75.043451,1413.897215,7900.542155,0.473981,1.333648
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,44278.306580,65633.880188,87002.150360,0.508933,0.754394,7.874394,0.000017,...,0.453945,0.442816,0.519102,0.558050,0.307857,61.920501,1125.686310,6114.512304,0.345141,1.385010
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,38096.457056,83986.903799,110868.745062,0.343618,0.757535,8.602243,0.000020,...,0.684172,0.255207,0.455347,0.407037,0.345978,81.472355,1536.980018,7342.617572,0.526258,1.208362
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,48480.358351,60718.389783,99805.913198,0.485746,0.608365,8.165200,0.000013,...,0.602089,0.313243,0.519864,0.540548,0.332260,66.912273,1191.395466,6042.190896,0.403133,1.392672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,2773.420313,5558.268358,7436.905587,0.372927,0.747390,4.279275,0.000269,...,0.472803,0.459062,0.423922,0.370460,0.380497,20.467358,117.521135,264.784723,0.397798,1.174879
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,2769.517696,4707.235680,6284.754058,0.440672,0.748993,3.997668,0.000270,...,0.522270,0.375640,0.414966,0.480757,0.407114,19.019121,104.124280,260.934838,0.346865,1.302838
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,1345.556715,6066.663542,6605.483068,0.203703,0.918428,4.172722,0.000683,...,0.707274,0.194034,0.458709,0.388888,0.412112,20.136274,91.711934,222.430095,0.560911,1.259709
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,1422.443447,5131.034159,5822.340542,0.244308,0.881267,3.990890,0.000620,...,0.727162,0.167350,0.503309,0.433183,0.434002,18.625555,74.950560,176.520805,0.590742,1.370494


In [176]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [177]:
df_3d_descriptors.to_csv('features/Descriptors/Train_3d_all_descriptors_Caco2.csv', index=False)

In [178]:
df_test_rdkit = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc_Caco2.csv')
df_test_rdkit = df_test_rdkit.fillna(0)
df_test_padel = pd.read_csv('features/Descriptors/Test_3d_padel_curated_Caco2.csv')

df_3d_descriptors = df_test_rdkit.merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,38360.811311,94769.013344,121802.797492,0.314942,0.778053,8.751867,0.000020,...,0.716440,0.242246,0.443935,0.427639,0.435742,83.625039,1490.670916,5767.464867,0.574661,1.307316
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,51411.476133,61298.107115,99240.409524,0.518050,0.617673,8.210482,0.000012,...,0.521776,0.409600,0.550410,0.543744,0.402603,73.734856,1509.449580,7462.654757,0.397064,1.496757
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,46406.553549,53110.909967,86105.006418,0.538953,0.616816,7.733054,0.000013,...,0.507796,0.403815,0.488706,0.481901,0.280013,63.182949,1140.267771,5775.076070,0.367416,1.250620
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,35517.050174,67187.132310,88176.599038,0.402795,0.761961,7.879172,0.000021,...,0.535190,0.393164,0.577035,0.564099,0.424267,65.667269,1194.175198,5528.761689,0.392532,1.565401
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,45705.035323,48149.561623,76960.888070,0.593874,0.625637,7.469090,0.000014,...,0.499362,0.399019,0.557737,0.513407,0.345796,66.494387,1284.657984,7304.227786,0.347570,1.416940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,3042.322578,4867.390132,7174.589188,0.424041,0.678421,4.185180,0.000223,...,0.596354,0.355038,0.532925,0.525713,0.444230,21.235527,116.332557,236.122220,0.427088,1.502868
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,2236.627781,5679.477413,6589.801379,0.339407,0.861859,4.104157,0.000385,...,0.666921,0.279758,0.442211,0.464638,0.469105,21.677207,111.392008,234.404547,0.500382,1.375955
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,2990.499642,4863.755105,7259.835582,0.411924,0.669954,4.189310,0.000224,...,0.585075,0.342986,0.454788,0.501705,0.443123,20.730176,114.928147,264.264648,0.392091,1.399616
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,2486.799820,4734.805218,5907.809024,0.420934,0.801449,3.904579,0.000322,...,0.570406,0.368369,0.477650,0.533995,0.392357,21.516702,123.888733,273.556994,0.408162,1.404002


In [179]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [180]:
df_3d_descriptors.to_csv('features/Descriptors/Test_3d_all_descriptors_Caco2.csv', index=False)

In [181]:
#3d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_Caco2.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models_3dall = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_3dall, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 442)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 442)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.077484 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 112710
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 442
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2920,0.4291,0.5403,0.5444,0.7379,0.7024,0.2367,0.3823,0.4865,0.6201,0.7891,0.7558
DecisionTreeRegressor,0.6100,0.5932,0.7810,0.0482,0.5168,0.4753,0.3316,0.4409,0.5758,0.4679,0.6912,0.6206
RandomForestRegressor,0.3124,0.4514,0.5589,0.5125,0.7162,0.6687,0.2715,0.4081,0.5210,0.5643,0.7541,0.7154
GradientBoostingRegressor,0.3063,0.4407,0.5534,0.5221,0.7226,0.6790,0.2449,0.3943,0.4949,0.6069,0.7813,0.7293
AdaBoostRegressor,0.3606,0.5043,0.6005,0.4373,0.6673,0.6070,0.3232,0.4760,0.5685,0.4813,0.7064,0.6510
XGBRegressor,0.3366,0.4555,0.5801,0.4748,0.6936,0.6446,0.2492,0.3849,0.4992,0.6000,0.7753,0.7360
ExtraTreesRegressor,0.2897,0.4348,0.5383,0.5479,0.7422,0.6990,0.2480,0.3996,0.4980,0.6021,0.7822,0.7349
LinearRegression,0.5818,0.5863,0.7628,0.0922,0.5915,0.5976,0.4276,0.5182,0.6539,0.3137,0.6624,0.6334
KNeighborsRegressor,0.4133,0.5054,0.6429,0.3550,0.6193,0.5530,0.3038,0.4314,0.5512,0.5124,0.7222,0.6419
SVR,0.3202,0.4530,0.5659,0.5004,0.7109,0.6620,0.2576,0.4012,0.5075,0.5867,0.7675,0.7102


In [182]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.4898755073959515, -7.00049198041983, -7.00...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.594960990406415, -6.218456681413507, -6.5...","[-6.75267134003664, -6.503516182324367, -6.589...","[0.1482957134947525, 0.19436578825049217, 0.11..."
1,DecisionTreeRegressor,"[-7.6, -7.68, -8.0, -5.96, -6.28, -6.32, -5.92...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -5.92, -5.92, -6.07, -6.0, -6.1249387...","[-7.052, -6.298, -6.324, -5.747, -5.8420000000...","[0.8150926327724966, 0.6312653958518557, 0.494..."
2,RandomForestRegressor,"[-6.724989700040001, -6.550327114339996, -6.84...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.701269529139998, -6.48424938737, -6.59467...","[-6.688069750294001, -6.5860410396220015, -6.5...","[0.08959267006213797, 0.0881558157876231, 0.06..."
3,GradientBoostingRegressor,"[-6.522480269226677, -6.634262144503919, -6.87...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.6235781495092105, -6.186414811290562, -6....","[-6.901668978276364, -6.793463863134581, -6.80...","[0.206865589765932, 0.41196120605409037, 0.225..."
4,AdaBoostRegressor,"[-6.211831931493931, -6.748057142857138, -6.80...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.424279661016951, -6.363416666666667, -6.2...","[-6.397667973138415, -6.3498714841038915, -6.4...","[0.17579065528839585, 0.12247357213890016, 0.1..."
5,XGBRegressor,"[-6.293887, -7.3411617, -7.1846623, -6.217369,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.202994, -6.160659, -6.43266, -5.8772655, ...","[-6.6045084, -6.397026, -6.503208, -5.8633184,...","[0.36873084, 0.26971877, 0.18362507, 0.1193670..."
6,ExtraTreesRegressor,"[-6.592449387369999, -6.81354364805, -6.926589...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.786576083110003, -6.445349999999996, -6.5...","[-6.799926038828005, -6.545102936882, -6.70832...","[0.09930421331660122, 0.06305977378107577, 0.1..."
7,LinearRegression,"[-5.186609784391193, -8.213153468274147, -6.07...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.443682046735834, -7.538844169757512, -5.7...","[-6.3447711132932625, -7.505117346333425, -5.4...","[0.5499321502651475, 0.509974956432313, 0.3104..."
8,KNeighborsRegressor,"[-6.28, -6.233333333333333, -7.023333333333333...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.686666666666667, -5.99333333333333...","[-6.544, -6.540333333333334, -6.05799999999999...","[0.22580620993330663, 0.3252595544757727, 0.09..."
9,SVR,"[-6.239108685457855, -6.423686519190544, -6.61...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.36331786365044, -6.0582774450542285, -6.2...","[-6.516327253622973, -6.176532747253679, -6.20...","[0.1896958842086631, 0.12150753859587465, 0.10..."


In [183]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_Caco2.csv')

In [184]:
#3d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


X_train shape:  (1007, 442)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 442)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


In [185]:
#3d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_Caco2.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 372)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 372)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007215 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 94860
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 372
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2951,0.4299,0.5432,0.5396,0.7347,0.6982,0.2378,0.3885,0.4877,0.6183,0.7882,0.7525
DecisionTreeRegressor,0.6426,0.6190,0.8016,-0.0027,0.4893,0.4562,0.3279,0.4517,0.5726,0.4738,0.6938,0.6341
RandomForestRegressor,0.3137,0.4517,0.5601,0.5105,0.7149,0.6639,0.2732,0.4150,0.5227,0.5615,0.7525,0.7131
GradientBoostingRegressor,0.3104,0.4432,0.5571,0.5157,0.7182,0.6731,0.2499,0.3987,0.4999,0.5990,0.7769,0.7244
AdaBoostRegressor,0.3585,0.5033,0.5988,0.4406,0.6684,0.6081,0.3212,0.4731,0.5668,0.4845,0.7092,0.6577
XGBRegressor,0.3340,0.4535,0.5780,0.4788,0.6970,0.6486,0.2425,0.3919,0.4924,0.6108,0.7826,0.7552
ExtraTreesRegressor,0.2958,0.4375,0.5439,0.5385,0.7355,0.6904,0.2513,0.4030,0.5013,0.5967,0.7785,0.7240
LinearRegression,0.4692,0.5399,0.6850,0.2679,0.6338,0.6168,0.4158,0.5169,0.6449,0.3326,0.6505,0.6121
KNeighborsRegressor,0.3942,0.4869,0.6278,0.3850,0.6401,0.5687,0.2801,0.4152,0.5293,0.5505,0.7449,0.6676
SVR,0.3126,0.4470,0.5591,0.5123,0.7199,0.6724,0.2525,0.3959,0.5025,0.5947,0.7739,0.7119


In [186]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.795632818056128, -6.914334894829284, -6.91...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.629194068816179, -6.31637975187376, -6.59...","[-6.708756229619074, -6.559188143841939, -6.68...","[0.15827214500743808, 0.17081796685643882, 0.1..."
1,DecisionTreeRegressor,"[-8.04, -7.68, -8.0, -5.85, -5.7, -6.51, -5.47...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-7.24, -6.82, -6.82, -5.94, -5.46, -7.15, -5...","[-6.540000000000001, -6.178000000000001, -6.15...","[0.7500399989333904, 0.5956643350075612, 0.518..."
2,RandomForestRegressor,"[-6.704139700039999, -6.610148505619998, -6.72...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.658839700039997, -6.481999999999999, -6.5...","[-6.637345531064, -6.60942934033, -6.652459999...","[0.09667331153056422, 0.09463129472429757, 0.0..."
3,GradientBoostingRegressor,"[-6.724909263002119, -6.6089999178327, -6.7845...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.757446103180893, -6.093692101135014, -6.8...","[-6.887336831304, -6.706062047013778, -6.96760...","[0.2219595823165584, 0.38399281125868573, 0.23..."
4,AdaBoostRegressor,"[-6.09365306122449, -6.653394835254847, -6.828...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.111562035204903, -6.315908107069542, -6.1...","[-6.4484509910341785, -6.440942501791929, -6.6...","[0.27785844608237875, 0.16338319456115402, 0.3..."
5,XGBRegressor,"[-6.6257195, -6.5180564, -6.6453714, -5.932089...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.886726, -6.19323, -6.119976, -6.0416913, ...","[-6.9182906, -6.382556, -6.5956144, -6.18347, ...","[0.3364699, 0.2761756, 0.31115904, 0.25195718,..."
6,ExtraTreesRegressor,"[-6.760489700039998, -6.701807819769997, -6.92...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.737117982760001, -6.563899999999998, -6.6...","[-6.867724941144003, -6.642309999999999, -6.74...","[0.13170506829929637, 0.0913102644832445, 0.07..."
7,LinearRegression,"[-4.992500802920301, -7.970711077584308, -6.92...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-5.309943575428061, -7.220396406380285, -5.7...","[-6.256675611313912, -7.2487468906204855, -5.5...","[0.5100730101870719, 0.4303129911148657, 0.282..."
8,KNeighborsRegressor,"[-6.28, -6.168333333333333, -7.023333333333333...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.243333333333332, -6.686666666666667, -5.9...","[-6.522, -6.523666666666666, -6.01066666666666...","[0.23578332803185645, 0.3210233636357331, 0.11..."
9,SVR,"[-6.419288862777276, -6.4061628500617696, -6.6...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.5371343017635555, -6.1609423074189165, -6...","[-6.6867891629966705, -6.265249077798246, -6.3...","[0.1592403548602838, 0.11331302534273491, 0.09..."


In [187]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_LVR_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_LVR_Caco2.csv')

In [188]:
#2d and 3d descriptors all
df_train_2d = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')
df_train_2d
df_train_3d = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_Caco2.csv')
df_train_3d

df_2d_3d_train = df_train_2d.merge(df_train_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_train.to_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv', index=False)
df_2d_3d_train

/tmp/ipykernel_1796054/1456265609.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_2d = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_Caco2.csv')


,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2064,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.19,15.976055,15.976055,0.030813,-1.935404,0.058779,22.883333,1664.156,...,0.682480,0.263316,0.443174,0.407327,0.382690,86.049198,1710.236099,8002.655598,0.523721,1.233191
1,2067,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.24,16.083777,16.083777,0.027668,-1.934865,0.053531,23.411765,1650.129,...,0.649321,0.261240,0.473762,0.435183,0.424702,75.043451,1413.897215,7900.542155,0.473981,1.333648
2,1914,CCCCN1CC(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@@H](CCC...,-8.00,15.668220,15.668220,0.003714,-4.150777,0.085546,24.018349,1587.863,...,0.453945,0.442816,0.519102,0.558050,0.307857,61.920501,1125.686310,6114.512304,0.345141,1.385010
3,2026,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.64,16.048909,16.048909,0.025704,-1.891709,0.101375,24.159292,1574.031,...,0.684172,0.255207,0.455347,0.407037,0.345978,81.472355,1536.980018,7342.617572,0.526258,1.208362
4,1920,CCCCN1CC(=O)N(C)[C@@H](Cc2cccc(Cl)c2)C(=O)N[C@...,-7.05,15.511700,15.511700,0.033138,-4.150544,0.085338,23.366972,1567.445,...,0.602089,0.313243,0.519864,0.540548,0.332260,66.912273,1191.395466,6042.190896,0.403133,1.392672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,5605,CC(C)C[C@@H]1NC(=O)CN(C)C(=O)CCCCNC(=O)[C@H](C...,-5.32,13.038711,13.038711,0.105820,-0.784997,0.662387,22.258065,430.549,...,0.472803,0.459062,0.423922,0.370460,0.380497,20.467358,117.521135,264.784723,0.397798,1.174879
1003,5602,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)CCCCNC(=O)[C@...,-5.45,13.057052,13.057052,0.142991,-0.802728,0.571252,24.322581,430.549,...,0.522270,0.375640,0.414966,0.480757,0.407114,19.019121,104.124280,260.934838,0.346865,1.302838
1004,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.19,12.835172,12.835172,0.168936,-0.728726,0.606745,24.965517,402.539,...,0.707274,0.194034,0.458709,0.388888,0.412112,20.136274,91.711934,222.430095,0.560911,1.259709
1005,2468,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.11,12.763992,12.763992,0.179945,-0.726122,0.611430,25.000000,388.512,...,0.727162,0.167350,0.503309,0.433183,0.434002,18.625555,74.950560,176.520805,0.590742,1.370494


In [189]:
df_test_2d = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_Caco2.csv')
df_test_2d
df_test_3d = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_Caco2.csv')
df_test_3d

df_2d_3d_test = df_test_2d.merge(df_test_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_test.to_csv('features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv', index=False)
df_2d_3d_test

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,2065,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.22,16.271931,16.271931,0.025756,-1.930898,0.058792,23.408333,1664.156,...,0.716440,0.242246,0.443935,0.427639,0.435742,83.625039,1490.670916,5767.464867,0.574661,1.307316
1,8066,CC[C@H]1C(=O)N[C@@H](COCCC(C)C)C(=O)N(C)[C@@H]...,-6.21,15.516152,15.516152,0.000091,-1.679143,0.049536,24.360360,1572.049,...,0.521776,0.409600,0.550410,0.543744,0.402603,73.734856,1509.449580,7462.654757,0.397064,1.496757
2,2068,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-7.24,15.993973,15.993973,0.020362,-1.750211,0.057933,24.294643,1552.024,...,0.507796,0.403815,0.488706,0.481901,0.280013,63.182949,1140.267771,5775.076070,0.367416,1.250620
3,2231,CC(C)C[C@H]1C(=O)N[C@@H](COC(C)(C)C)C(=O)N(C)[...,-5.92,15.804049,15.804049,0.002346,-2.970606,0.165351,26.981481,1537.344,...,0.535190,0.393164,0.577035,0.564099,0.424267,65.667269,1194.175198,5528.761689,0.392532,1.565401
4,2230,CC(C)C[C@H]1C(=O)N[C@H](C(=O)N2CCCCC2)CC(=O)N[...,-5.96,15.545371,15.545371,0.013183,-2.948188,0.126607,27.422018,1530.953,...,0.499362,0.399019,0.557737,0.513407,0.345796,66.494387,1284.657984,7304.227786,0.347570,1.416940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-6.00,12.958074,12.958074,0.156212,-0.733388,0.586385,24.903226,430.593,...,0.596354,0.355038,0.532925,0.525713,0.444230,21.235527,116.332557,236.122220,0.427088,1.502868
248,2489,CC(C)CN1CC(=O)NCCCCCCN[C@@H](Cc2ccccc2)C(=O)N[...,-3.99,13.040403,13.040403,0.016783,-0.701825,0.680196,22.838710,430.593,...,0.666921,0.279758,0.442211,0.464638,0.469105,21.677207,111.392008,234.404547,0.500382,1.375955
249,2478,CC(C)C[C@@H]1NC(=O)[C@H](C)NCCCCCCNC(=O)[C@H](...,-5.24,13.114656,13.114656,0.186152,-0.689890,0.586385,24.903226,430.593,...,0.585075,0.342986,0.454788,0.501705,0.443123,20.730176,114.928147,264.264648,0.392091,1.399616
250,2479,CC(C)C[C@@H]1NCCCCCCNC(=O)[C@H](C)NC(=O)[C@H](...,-4.77,13.101637,13.101637,0.172241,-0.755076,0.586385,24.903226,430.593,...,0.570406,0.368369,0.477650,0.533995,0.392357,21.516702,123.888733,273.556994,0.408162,1.404002


In [190]:
#All 2d and 3d descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/1325129287.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
/tmp/ipykernel_1796054/1325129287.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 3532)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 3532)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.109282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 612899
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2781
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1928,0.3343,0.4391,0.6991,0.8364,0.8263,0.1615,0.2990,0.4019,0.7408,0.8619,0.8504
DecisionTreeRegressor,0.4145,0.4737,0.6438,0.3532,0.6743,0.6511,0.2391,0.3670,0.4890,0.6163,0.7917,0.7761
RandomForestRegressor,0.2097,0.3525,0.4580,0.6727,0.8218,0.8095,0.1847,0.3269,0.4298,0.7036,0.8423,0.8308
GradientBoostingRegressor,0.1994,0.3418,0.4465,0.6889,0.8304,0.8155,0.1676,0.3076,0.4093,0.7311,0.8565,0.8390
AdaBoostRegressor,0.2481,0.4031,0.4981,0.6129,0.7897,0.7659,0.2135,0.3706,0.4621,0.6574,0.8245,0.7755
XGBRegressor,0.2183,0.3542,0.4673,0.6593,0.8124,0.7991,0.1644,0.3091,0.4054,0.7362,0.8596,0.8494
ExtraTreesRegressor,0.1779,0.3210,0.4217,0.7225,0.8516,0.8385,0.1401,0.2873,0.3743,0.7752,0.8828,0.8758
LinearRegression,1.8556,1.0049,1.3622,-1.8954,0.3994,0.4572,0.8006,0.7062,0.8948,-0.2849,0.5455,0.5915
KNeighborsRegressor,0.2329,0.3608,0.4826,0.6365,0.8024,0.7851,0.1793,0.3094,0.4234,0.7123,0.8460,0.8267
SVR,0.2165,0.3544,0.4653,0.6622,0.8150,0.8026,0.1681,0.3196,0.4100,0.7303,0.8568,0.8409


In [191]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.585503012263708, -7.317529600210938, -7.07...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.473612717987427, -6.714447000100684, -6.3...","[-6.705327251892316, -6.565021413275565, -6.44...","[0.1473797045072883, 0.11873046438092062, 0.08..."
1,DecisionTreeRegressor,"[-5.92, -7.05, -7.0, -7.05, -6.28, -5.85, -6.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.7, -6.68, -6.21, -5.85, -5.81000000000000...","[-6.744, -6.2299999999999995, -6.9479999999999...","[0.580365402139032, 0.29024127893874796, 0.665..."
2,RandomForestRegressor,"[-6.5643035223899995, -7.130199999999995, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.420676083109999, -6.4941999999999975, -6....","[-6.569597020476001, -6.5123077785820005, -6.4...","[0.08902044980910367, 0.0704093663630938, 0.08..."
3,GradientBoostingRegressor,"[-6.591788089855236, -7.134676646618652, -7.05...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.58646699933712, -6.273092041981439, -6.42...","[-6.883900298163114, -6.607817619560485, -6.56...","[0.20144772477982184, 0.35001648422680953, 0.0..."
4,AdaBoostRegressor,"[-6.646103896103894, -7.073304722333333, -6.94...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.144444444444445, -6.585593220338982, -6.1...","[-6.385845631492365, -6.708012509479741, -6.25...","[0.1509345269133586, 0.10278276873491482, 0.09..."
5,XGBRegressor,"[-6.4965215, -7.3233256, -7.1371765, -7.022797...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.67662, -6.2048297, -6.784189, -5.8541527,...","[-6.667491, -6.4330635, -6.452455, -5.90137, -...","[0.21447031, 0.19588032, 0.17812467, 0.0469646..."
6,ExtraTreesRegressor,"[-6.42743871964, -7.348849019599998, -7.064157...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.431357574909998, -6.418536383060001, -6.1...","[-6.756957036386001, -6.4805290366460016, -6.2...","[0.2051757119552671, 0.09394022766150517, 0.09..."
7,LinearRegression,"[-3.4, -6.033759552961481, -6.574653331485132,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-3.4, -8.7638078193181, -5.867417377580365, ...","[-5.319152873847107, -7.4297247924885195, -6.5...","[1.3108023662133386, 2.103219822081671, 0.6735..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31666666666666...","[-6.544, -6.476000000000001, -6.19866666666666...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.278872282564082, -7.118329047162066, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.238752438786845, -6.302756877436265, -6.1...","[-6.68162090469225, -6.394553044243118, -6.223...","[0.280267171742973, 0.07255299899249676, 0.079..."


In [192]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_Caco2.csv')

In [193]:
#All 2d and 3d descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/1180161388.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')


X_train shape:  (1007, 2911)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 2911)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_1796054/1180161388.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.104493 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 612899
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2781
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1928,0.3343,0.4391,0.6991,0.8364,0.8263,0.1615,0.2990,0.4019,0.7408,0.8619,0.8504
DecisionTreeRegressor,0.4170,0.4740,0.6457,0.3494,0.6759,0.6579,0.2393,0.3626,0.4892,0.6160,0.7906,0.7730
RandomForestRegressor,0.2098,0.3528,0.4580,0.6727,0.8217,0.8094,0.1835,0.3251,0.4283,0.7056,0.8434,0.8316
GradientBoostingRegressor,0.2005,0.3425,0.4478,0.6871,0.8292,0.8148,0.1669,0.3068,0.4086,0.7321,0.8570,0.8396
AdaBoostRegressor,0.2435,0.3985,0.4934,0.6201,0.7945,0.7699,0.2136,0.3692,0.4621,0.6572,0.8268,0.7832
XGBRegressor,0.2183,0.3542,0.4673,0.6593,0.8124,0.7991,0.1644,0.3091,0.4054,0.7362,0.8596,0.8494
ExtraTreesRegressor,0.1765,0.3183,0.4202,0.7246,0.8533,0.8434,0.1404,0.2868,0.3747,0.7747,0.8825,0.8767
LinearRegression,1.8556,1.0049,1.3622,-1.8954,0.3994,0.4572,0.8006,0.7062,0.8948,-0.2849,0.5455,0.5915
KNeighborsRegressor,0.2329,0.3608,0.4826,0.6365,0.8024,0.7851,0.1793,0.3094,0.4234,0.7123,0.8460,0.8267
SVR,0.2165,0.3544,0.4653,0.6622,0.8150,0.8026,0.1681,0.3196,0.4100,0.7303,0.8568,0.8410


In [194]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.585503012263708, -7.317529600210938, -7.07...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.473612717987427, -6.714447000100684, -6.3...","[-6.705327251892316, -6.565021413275565, -6.44...","[0.1473797045072883, 0.11873046438092062, 0.08..."
1,DecisionTreeRegressor,"[-5.92, -7.03, -7.03, -6.89, -6.68, -5.85, -5....",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-8.06, -6.68, -6.21, -5.85, -5.68, -7.0, -5....","[-6.7540000000000004, -6.435, -6.544, -5.94, -...","[0.8803090366456546, 0.3971649531366027, 0.672..."
2,RandomForestRegressor,"[-6.524188141619996, -7.114474792789996, -7.12...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.463460998589997, -6.457501183629997, -6.3...","[-6.574425741730001, -6.479680660077999, -6.42...","[0.08897689098916352, 0.06513917768623674, 0.1..."
3,GradientBoostingRegressor,"[-6.5461759511873066, -7.134676646618653, -7.0...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.501135217188521, -6.30749450052114, -6.38...","[-6.856076769035495, -6.605974806670996, -6.55...","[0.2199138841231681, 0.32412970358606735, 0.09..."
4,AdaBoostRegressor,"[-6.484800000000003, -7.185055192444443, -6.94...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.115926966292148, -6.682777777777781, -6.2...","[-6.362803681788235, -6.631836644684276, -6.18...","[0.1523604714986441, 0.12204926864723514, 0.07..."
5,XGBRegressor,"[-6.4965215, -7.3233256, -7.1371765, -7.022797...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.67662, -6.2048297, -6.784189, -5.8541527,...","[-6.667491, -6.4330635, -6.452455, -5.90137, -...","[0.21447031, 0.19588032, 0.17812467, 0.0469646..."
6,ExtraTreesRegressor,"[-6.345221982699999, -7.277143648059998, -7.09...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.458711104730001, -6.470600000000003, -6.2...","[-6.756615201426003, -6.469703820026001, -6.35...","[0.1761115098469282, 0.04443231857089962, 0.08..."
7,LinearRegression,"[-3.4, -6.033759552961253, -6.574653331484896,...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-3.4, -8.763807819318668, -5.867417377580491...","[-5.319152873847122, -7.4297247924887415, -6.5...","[1.3108023662132733, 2.1032198220818086, 0.673..."
8,KNeighborsRegressor,"[-6.28, -7.3500000000000005, -6.98, -7.0366666...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.28, -6.501666666666666, -6.31666666666666...","[-6.544, -6.476000000000001, -6.19866666666666...","[0.22580620993330663, 0.13166286914354805, 0.1..."
9,SVR,"[-6.278872355830644, -7.118328703484984, -7.10...",0 -7.22 1 -6.21 2 -7.24 3 -5.9...,"[[-6.238752579890877, -6.302756795697351, -6.1...","[-6.681558362343031, -6.394516867043597, -6.22...","[0.2802203411649314, 0.07253622788944354, 0.07..."


In [195]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_const_rem_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_const_rem_Caco2.csv')

In [196]:
#All 2d and 3d descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/2496381194.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')


X_train shape:  (1007, 2125)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_1796054/2496381194.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_test shape:  (252, 2125)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.078642 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 429390
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 2037
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1891,0.3319,0.4349,0.7049,0.8399,0.8286,0.1635,0.3026,0.4043,0.7377,0.8597,0.8461
DecisionTreeRegressor,0.4089,0.4741,0.6394,0.3620,0.6774,0.6582,0.2073,0.3407,0.4553,0.6673,0.8191,0.7976
RandomForestRegressor,0.2064,0.3481,0.4543,0.6779,0.8250,0.8111,0.1751,0.3178,0.4185,0.7189,0.8511,0.8393
GradientBoostingRegressor,0.2050,0.3446,0.4528,0.6801,0.8251,0.8086,0.1661,0.3068,0.4076,0.7334,0.8581,0.8396
AdaBoostRegressor,0.2532,0.4076,0.5032,0.6049,0.7835,0.7635,0.2166,0.3711,0.4654,0.6524,0.8218,0.7806
XGBRegressor,0.2146,0.3502,0.4633,0.6651,0.8159,0.8067,0.1643,0.3051,0.4054,0.7363,0.8594,0.8413
ExtraTreesRegressor,0.1755,0.3176,0.4189,0.7262,0.8539,0.8422,0.1384,0.2846,0.3721,0.7778,0.8850,0.8781
LinearRegression,2.5578,1.2419,1.5993,-2.9911,0.3116,0.3451,1.7278,1.1085,1.3145,-1.7729,0.4264,0.4852
KNeighborsRegressor,0.2274,0.3569,0.4769,0.6452,0.8076,0.7923,0.1619,0.3011,0.4024,0.7402,0.8616,0.8379
SVR,0.2138,0.3515,0.4624,0.6664,0.8177,0.8079,0.1635,0.3124,0.4043,0.7376,0.8609,0.8439


In [197]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_LVR_Caco2.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_LVR_Caco2.csv')

In [215]:
#Stacked architecture model
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
from tqdm import tqdm
import joblib

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [216]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [217]:
from tqdm import tqdm
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Train/All_fingerprints_train_Caco2.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Test/All_fingerprints_test_Caco2.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Train_all_atomic_desc_Caco2.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Test_all_atomic_desc_Caco2.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
target_column = 'Permeability'
def scale_features(df_train, df_test):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test)
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test)
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test)
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test)
print(df_desc_train)
models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(), 
    MLPRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),

]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]


/tmp/ipykernel_1796054/1099517201.py:3: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(1007, 264)
(252, 264)
(1007, 868)
(252, 868)
(1007, 759)
(252, 759)
(1007, 12)
(252, 12)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
       ID                                             SMILES  Permeability  \
872    33  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...        -5.480   
839    41  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...        -7.050   
878   982  CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...        -5.890   
880   983  CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...        -5.890   
877   984  CC[C@@H](C)[C@@H]1NC

In [218]:
df_desc_train

,ID,SMILES,Permeability,qed,SPS,FpDensityMorgan1,BCUT2D_MRHI,AvgIpc,BalabanJ_x,Ipc,...,RPCS,RNCS,LOBMAX,LOBMIN,MOMI-XY,MOMI-YZ,geomShape,Dm,Dv,L3p
872,33,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-5.480,0.703485,0.366038,0.775151,-0.371621,1.477512,0.585759,-0.031663,...,-0.126965,1.179074,1.089993,1.505852,-1.104190,0.300321,0.858344,-1.741270,-1.708913,-0.588000
839,41,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-7.050,0.238136,-0.169850,0.925592,-0.372297,1.518377,0.630085,-0.031663,...,0.215494,2.194042,0.131484,-0.409972,-1.170630,0.069549,-0.088750,-1.748783,-1.816749,0.088517
878,982,CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...,-5.890,1.155947,1.640163,1.658459,3.688229,1.786100,-1.004366,-0.031663,...,1.109265,0.762380,0.089037,0.082431,-0.069238,0.032177,0.323230,-0.078535,-0.409299,-1.163107
880,983,CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...,-5.890,1.155947,1.275162,1.970921,3.688092,1.764191,-1.020744,-0.031663,...,0.783911,0.658786,0.760195,0.436051,-1.594363,1.026646,-0.015715,-0.702074,-1.074581,-0.792240
877,984,CC[C@@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H]...,-6.700,1.155947,1.275162,1.970921,3.688228,1.763976,-1.021415,-0.031663,...,-0.291715,0.493467,0.625830,1.068602,-1.411417,0.628122,0.558279,-0.640380,-0.615901,-0.550880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,8496,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-5.600,-0.609973,-0.813457,-0.685012,-0.312132,-0.892268,-0.626917,-0.031663,...,-0.322630,0.033653,-0.351810,-1.108537,-0.855009,-0.362026,0.954129,0.277858,0.446473,2.018426
60,8498,CC[C@H](C)[C@H]1C(=O)N[C@@H]([C@@H](C)O)C(=O)N...,-5.960,-0.679912,-0.781956,-0.225161,-0.300506,-0.907553,-0.506843,-0.031659,...,-0.188056,-1.061812,-0.091701,-0.043618,0.423329,-0.007579,-1.086349,0.390494,0.483826,-0.496830
309,8499,CC[C@H](C)[C@H]1C(=O)N[C@@H]([C@@H](C)O)C(=O)N...,-5.595,-0.541665,-0.233781,-0.211631,-0.304607,-0.903611,0.060191,-0.031663,...,2.213894,0.520635,-0.563098,-0.051392,1.011592,-0.674931,-0.758391,0.772861,0.667632,-0.150518
286,8500,CC[C@H](C)[C@H]1C(=O)N[C@@H]([C@@H](C)O)C(=O)N...,-5.890,-0.428992,-0.371670,-0.052275,0.800964,-0.895949,-0.651215,-0.031663,...,-0.131164,0.133707,-0.158394,-0.351689,0.832959,-0.475312,-0.203797,0.559273,0.263524,-0.129120


In [219]:
dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]
target_column = 'Permeability'


meta_features_train = []
meta_features_test = []

# Stage 1: Train weak learners with 5-fold cross-validation
for df_train, df_test in tqdm(dataframes, desc="Processing dataframe pairs"):
    X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
    X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
    y_weak = df_train[target_column]
    y_eval = df_test[target_column]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    # Storing predictions for the current dataframe
    fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
    fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

    for i, model in tqdm(enumerate(models_weak), desc="Training models"):
        fold_predictions = np.zeros(X_weak.shape[0])
        test_predictions_folds = []

        for train_index, val_index in kf.split(X_weak):
            X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
            y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
            
            model.fit(X_train, y_train)

            # Predictions for validation set
            fold_predictions[val_index] =  np.clip( model.predict(X_val), -10, -3.4)

            # Predictions for test set
            test_predictions_fold =  np.clip( model.predict(X_eval), -10, -3.4)
            test_predictions_folds.append(test_predictions_fold)

        # Store predictions for the meta-learner
        fold_meta_features_train[:, i] = fold_predictions
        fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)

    meta_features_train.append(fold_meta_features_train)
    meta_features_test.append(fold_meta_features_test)

# Convert lists to arrays for the meta-learner
meta_features_train = np.hstack(meta_features_train)
meta_features_test = np.hstack(meta_features_test)

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print("Dimensions of meta_features_train:", meta_features_train.shape)
print("Dimensions of meta_features_test:", meta_features_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Stage 1 completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Stage 2: Train the meta-learner using predictions from weak learners
kf = KFold(n_splits=5, shuffle=True, random_state=101)
results = {}
predictions = []
for model in models_meta:
    model_name = model.__class__.__name__
    predictions_train = []
    actual_y_train = []
    
    test_predictions_folds = []

    for train_index, val_index in kf.split(meta_features_train):
        X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
        y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
        
        model.fit(X_fold_train, y_fold_train)

        y_pred_fold = model.predict(X_fold_val)
        y_pred_fold = np.clip(y_pred_fold, -10, -3.4)
        predictions_train.extend(y_pred_fold)
        actual_y_train.extend(y_fold_val)

        # Predictions for test set
        test_predictions_fold = model.predict(meta_features_test)
        test_predictions_fold = np.clip(test_predictions_fold, -10, -3.4)
        test_predictions_folds.append(test_predictions_fold)

    # Metrics
    predictions_test_mean = np.mean(test_predictions_folds, axis=0)
    predictions_test_std = np.std(test_predictions_folds, axis=0)

    mse_train = mean_squared_error(actual_y_train, predictions_train)
    mae_train = mean_absolute_error(actual_y_train, predictions_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(actual_y_train, predictions_train)
    pearson_train, _ = pearsonr(actual_y_train, predictions_train)
    spearman_train, _ = spearmanr(actual_y_train, predictions_train)

    mse_test = mean_squared_error(y_eval, predictions_test_mean)
    mae_test = mean_absolute_error(y_eval, predictions_test_mean)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_eval, predictions_test_mean)
    pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
    spearman_test, _ = spearmanr(y_eval, predictions_test_mean)
    print(f'{model_name} Evaluation completed: Test R2 score: {r2_test}')

    predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_eval,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

    results[model_name] = {
        'Train MSE (5 fold CV)': mse_train,
        'Train MAE (5 fold CV)': mae_train,
        'Train RMSE (5 fold CV)': rmse_train,
        'Train R2 (5 fold CV)': r2_train,
        'Train PCC (5 fold CV)': pearson_train,
        'Train SCC (5 fold CV)': spearman_train,
        'Test MSE': mse_test,
        'Test MAE': mae_test,
        'Test RMSE': rmse_test,
        'Test R2': r2_test,
        'Test PCC': pearson_test,
        'Test SCC': spearman_test,
    }

results_df = pd.DataFrame(results).T
prediction_df = pd.DataFrame(predictions)
results_df

Processing dataframe pairs:   0%|          | 0/4 [00:00<?, ?it/s]
Training models: 0it [00:00, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.164949 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 54374
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 251
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain


Training models: 1it [00:10, 10.89s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf



Training models: 2it [00:20,  9.89s/it]
Training models: 3it [00:46, 17.64s/it]
Training models: 4it [00:55, 13.95s/it]
Training models: 5it [01:34, 23.20s/it]
Training models: 6it [01:37, 16.27s/it]
Training models: 7it [01:38, 11.22s/it]
Training models: 8it [01:38,  7.81s/it]/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(

Training models: 9it [01:43,  6.87s/it]
Training models: 10it [01:44, 10.47s/it]
Processing dataframe pairs:  25%|██▌       | 1/4 [01:46<05:18, 106.31s/it]
Training models: 0it [00:00, ?it/s]

...

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMRegressor Evaluation completed: Test R2 score: 0.7631706725931604
DecisionTreeRegressor Evaluation completed: Test R2 score: 0.7417188935878385
RandomForestRegressor Evaluation completed: Test R2 score: 0.7825975438017221
GradientBoostingRegressor Evaluation completed: Test R2 score: 0.7670785447899214
AdaBoostRegressor Evaluation completed: Test R2 score: 0.7454696252874482
XGBRegressor Evaluation completed: Test R2 score: 0.7721404365677171
ExtraTreesRegressor Evaluation completed: Test R2 score: 0.7744821573657891
LinearRegression Evaluation completed: Test R2 score: 0.7639775824249695
KNeighborsRegressor Evaluation completed: Test R2 score: 0.7600589375779152
SVR Evaluation completed: Test R2 score: 0.7609430586256124


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor Evaluation completed: Test R2 score: 0.727407377770521


,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.135592,0.275783,0.368228,0.788429,0.888067,0.880549,0.147571,0.290286,0.384149,0.763171,0.873950,0.851635
DecisionTreeRegressor,0.271498,0.394193,0.521055,0.576368,0.789846,0.779183,0.160937,0.306412,0.401170,0.741719,0.861376,0.839971
RandomForestRegressor,0.129780,0.272027,0.360250,0.797497,0.893034,0.883796,0.135466,0.276802,0.368056,0.782598,0.884772,0.857066
GradientBoostingRegressor,0.136258,0.281183,0.369132,0.787390,0.887447,0.878967,0.145136,0.286530,0.380967,0.767079,0.875910,0.851862
AdaBoostRegressor,0.143428,0.291582,0.378719,0.776202,0.881088,0.873015,0.158600,0.303428,0.398247,0.745470,0.863450,0.845891
XGBRegressor,0.146234,0.290064,0.382406,0.771823,0.878933,0.866721,0.141981,0.283628,0.376804,0.772140,0.878863,0.843367
ExtraTreesRegressor,0.127145,0.269157,0.356573,0.801610,0.895383,0.886717,0.140522,0.279454,0.374863,0.774482,0.880102,0.848733
LinearRegression,0.140103,0.289717,0.374303,0.781391,0.884131,0.877992,0.147068,0.289595,0.383494,0.763978,0.874092,0.847731
KNeighborsRegressor,0.145448,0.286482,0.381377,0.773050,0.879764,0.867034,0.149510,0.290693,0.386665,0.760059,0.872143,0.848088
SVR,0.133912,0.277260,0.365940,0.791051,0.889436,0.878023,0.148959,0.281841,0.385952,0.760943,0.873632,0.848207


In [220]:
results_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/Stacked_architecture/Results_5_folds_stacked_archi_Caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/Stacked_architecture/Prediction_data_5_folds_stacked_archi_Caco2.csv')

In [221]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
from tqdm import tqdm
import joblib

# Ensure the models directory exists
os.makedirs('/home/users/akshay/PCPpred/Caco2/models_Caco2/', exist_ok=True)

# Assuming remove_low_variance_columns and features functions are defined elsewhere
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_desc = features(train, "Permeability")
joblib.dump(selected_features_desc, '/home/users/akshay/PCPpred/Caco2/models_Caco2/selected_features_descriptors.joblib')
df_desc_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_desc]], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test = df_desc_test.dropna()
df_desc_test = df_desc_test[df_desc_train.columns]

# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Train/All_fingerprints_train_Caco2.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_fp = features(train, "Permeability")
joblib.dump(selected_features_fp, '/home/users/akshay/PCPpred/Caco2/models_Caco2/selected_features_fingerprints.joblib')
df_fp_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_fp]], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Test/All_fingerprints_test_Caco2.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test = df_fp_test[df_fp_train.columns]

# Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_emb = features(train, "Permeability")
joblib.dump(selected_features_emb, '/home/users/akshay/PCPpred/Caco2/models_Caco2/selected_features_embeddings.joblib')
df_emb_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_emb]], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test = df_emb_test[df_emb_train.columns]

# Atomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Train_all_atomic_desc_Caco2.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_atomic = features(train, "Permeability")
joblib.dump(selected_features_atomic, '/home/users/akshay/PCPpred/Caco2/models_Caco2/selected_features_atomic.joblib')
df_atomic_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_atomic]], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Test_all_atomic_desc_Caco2.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test = df_atomic_test[df_atomic_train.columns]

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Filter dataframes to have consistent IDs
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]
df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]
df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

target_column = 'Permeability'

def scale_features(df_train, df_test, feature_type):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    # Save the scaler
    joblib.dump(scaler, f'/home/users/akshay/PCPpred/Caco2/models_Caco2/scaler_{feature_type}.joblib')
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test, 'Descriptor')
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test, 'Fingerprints')
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test, 'Embeddings')
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test , 'Atomic')

models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),
]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]

dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]
data_names = ['descriptors', 'fingerprints', 'embeddings', 'atomic']

meta_features_train = []
meta_features_test = []

# Stage 1: Train weak learners with 5-fold cross-validation
for df_idx, (df_train, df_test) in enumerate(tqdm(dataframes, desc="Processing dataframe pairs")):
    X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
    X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
    y_weak = df_train[target_column]
    y_eval = df_test[target_column]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
    fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

    for i, model in tqdm(enumerate(models_weak), desc="Training models", total=len(models_weak)):
        fold_predictions = np.zeros(X_weak.shape[0])
        test_predictions_folds = []

        for fold_idx, (train_index, val_index) in enumerate(kf.split(X_weak)):
            X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
            y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
            
            model.fit(X_train, y_train)
            
            model_name = model.__class__.__name__
            joblib.dump(model, f'/home/users/akshay/PCPpred/Caco2/models_Caco2/weak_{data_names[df_idx]}_{model_name}_fold_{fold_idx}.joblib')

            fold_predictions[val_index] = np.clip(model.predict(X_val), -10, -3.4)

            test_predictions_fold = np.clip(model.predict(X_eval), -10, -3.4)
            test_predictions_folds.append(test_predictions_fold)

        fold_meta_features_train[:, i] = fold_predictions
        fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)

    meta_features_train.append(fold_meta_features_train)
    meta_features_test.append(fold_meta_features_test)
    
    joblib.dump(fold_meta_features_train, f'/home/users/akshay/PCPpred/Caco2/models_Caco2/meta_features_train_{data_names[df_idx]}.joblib')
    joblib.dump(fold_meta_features_test, f'/home/users/akshay/PCPpred/Caco2/models_Caco2/meta_features_test_{data_names[df_idx]}.joblib')

meta_features_train = np.hstack(meta_features_train)
meta_features_test = np.hstack(meta_features_test)

joblib.dump(meta_features_train, '/home/users/akshay/PCPpred/Caco2/models_Caco2/meta_features_train_combined.joblib')
joblib.dump(meta_features_test, '/home/users/akshay/PCPpred/Caco2/models_Caco2/meta_features_test_combined.joblib')

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print("Dimensions of meta_features_train:", meta_features_train.shape)
print("Dimensions of meta_features_test:", meta_features_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Stage 1 completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Stage 2: Train the meta-learner using predictions from weak learners
kf = KFold(n_splits=5, shuffle=True, random_state=101)
results = {}
predictions = []
for model in models_meta:
    model_name = model.__class__.__name__
    predictions_train = []
    actual_y_train = []
    
    test_predictions_folds = []

    for fold_idx, (train_index, val_index) in enumerate(kf.split(meta_features_train)):
        X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
        y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
        
        model.fit(X_fold_train, y_fold_train)
        
        joblib.dump(model, f'/home/users/akshay/PCPpred/Caco2/models_Caco2/meta_{model_name}_fold_{fold_idx}.joblib')

        y_pred_fold = model.predict(X_fold_val)
        y_pred_fold = np.clip(y_pred_fold, -10, -3.4)
        predictions_train.extend(y_pred_fold)
        actual_y_train.extend(y_fold_val)

        test_predictions_fold = model.predict(meta_features_test)
        test_predictions_fold = np.clip(test_predictions_fold, -10, -3.4)
        test_predictions_folds.append(test_predictions_fold)

    # Metrics
    predictions_test_mean = np.mean(test_predictions_folds, axis=0)
    predictions_test_std = np.std(test_predictions_folds, axis=0)

    mse_train = mean_squared_error(actual_y_train, predictions_train)
    mae_train = mean_absolute_error(actual_y_train, predictions_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(actual_y_train, predictions_train)
    pearson_train, _ = pearsonr(actual_y_train, predictions_train)
    spearman_train, _ = spearmanr(actual_y_train, predictions_train)

    mse_test = mean_squared_error(y_eval, predictions_test_mean)
    mae_test = mean_absolute_error(y_eval, predictions_test_mean)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_eval, predictions_test_mean)
    pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
    spearman_test, _ = spearmanr(y_eval, predictions_test_mean)
    

    predictions.append({
        'Model': model_name,
        'Y Train pred': predictions_train,
        'Y Test actual': y_eval,
        'Test prediction folds': test_predictions_folds,
        'Test Predictions Mean': predictions_test_mean,
        'Test Predictions Std': predictions_test_mean,
    })

    results[model_name] = {
        'Train MSE (5 fold CV)': mse_train,
        'Train MAE (5 fold CV)': mae_train,
        'Train RMSE (5 fold CV)': rmse_train,
        'Train R2 (5 fold CV)': r2_train,
        'Train PCC (5 fold CV)': pearson_train,
        'Train SCC (5 fold CV)': spearman_train,
        'Test MSE': mse_test,
        'Test MAE': mae_test,
        'Test RMSE': rmse_test,
        'Test R2': r2_test,
        'Test PCC': pearson_test,
        'Test SCC': spearman_test,
    }

results_df = pd.DataFrame(results).T

/tmp/ipykernel_1796054/1388518722.py:24: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(1007, 264)
(252, 264)
(1007, 868)
(252, 868)
(1007, 759)
(252, 759)
(1007, 12)
(252, 12)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
       ID                                             SMILES  Permeability  \
872    33  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...        -5.480   
839    41  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...        -7.050   
878   982  CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...        -5.890   
880   983  CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...        -5.890   
877   984  CC[C@@H](C)[C@@H]1NC

Training models:   0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006999 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 54374
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 251
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain


Training models:  10%|█         | 1/10 [00:05<00:46,  5.15s/it]

...

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of data points in the train set: 806, number of used features: 40
[LightGBM] [Info] Start training from score -6.189756
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001094 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9235
[LightGBM] [Info] Number of data points in the train set: 8

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [230]:
#Saving best model
#2d Mordred descriptors const removal
import os
import joblib 


def train_and_test_predict(models, X_train, y_train, X_test, y_test, save_dir='models_Caco2_2d'):
   
    os.makedirs(save_dir, exist_ok=True)

    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        fold_no = 1
        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            fold_model_path = os.path.join(save_dir, f"{model_name}_fold{fold_no}_Caco2.joblib")
            joblib.dump(model, fold_model_path)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.4)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.4)
            test_predictions_folds.append(predictions_test_fold)

            fold_no += 1

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,
        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df

df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
X_test = df_test[X_train.columns]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Saving the scaler and features
joblib.dump(scaler, '/home/users/akshay/PCPpred/Caco2/models_Caco2_2d/scaler_caco2.joblib')
joblib.dump(X_train.columns, '/home/users/akshay/PCPpred/Caco2/models_Caco2_2d/features.joblib')

models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
]
result_df, prediction_df = train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_1796054/245504947.py:88: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_Caco2.csv')
/tmp/ipykernel_1796054/245504947.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


X_train shape:  (1007, 1216)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1216)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024299 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 260021
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1170
[LightGBM] [Info] Start training from score -6.193048
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1932,0.3320,0.4395,0.6986,0.8360,0.8280,0.1291,0.2722,0.3593,0.7928,0.8920,0.8800


In [231]:
models_dir = '/home/users/akshay/PCPpred/Caco2/models_Caco2_2d' 
scaler_path = '/home/users/akshay/PCPpred/Caco2/models_Caco2_2d/scaler_caco2.joblib' 
features_path =   'models_Caco2_2d/features.joblib'                         
model_base_name = 'LGBMRegressor'                   
n_folds = 5                                    
features = joblib.load('/home/users/akshay/PCPpred/Caco2/models_Caco2_2d/features.joblib')
print(features)

df_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Test_2d_Mordred_desc_Caco2.csv')
X_test = df_test[features]
for col in X_test.select_dtypes(include=['object']).columns:
    X_test[col] = 0
y_test = df_test['Permeability']

scaler = joblib.load(scaler_path)
X_new_scaled = scaler.transform(X_test)
X_new_scaled = pd.DataFrame(X_new_scaled, columns=X_test.columns,index= X_test.index)

all_fold_preds = []

for fold in range(1, n_folds + 1):
    fold_model_path = os.path.join(models_dir, f"{model_base_name}_fold{fold}_Caco2.joblib")
    fold_model = joblib.load(fold_model_path)
    preds = fold_model.predict(X_new_scaled)
    preds = np.clip(preds, -10, -3.4)  
    all_fold_preds.append(preds)


all_fold_preds = np.array(all_fold_preds)
mean_prediction = np.mean(all_fold_preds, axis=0)

mse_test = mean_squared_error(y_test, mean_prediction)
print(f"{mse_test:.4f}")
mae_test = mean_absolute_error(y_test, mean_prediction)
print(f"{mae_test:.4f}")
rmse_test = np.sqrt(mse_test)
print(f"{rmse_test:.4f}")
r2_test = r2_score(y_test, mean_prediction)
print(f"{r2_test:.4f}")
pearson_test, _ = pearsonr(y_test, mean_prediction)
print(f"{pearson_test:.4f}")
spearman_test, _ = spearmanr(y_test, mean_prediction)
print(f"{spearman_test:.4f}")

print("Prediction on new data complete.")


Index(['AcidicGroupCount', 'BasicGroupCount', 'AdjacencyMatrix',
       'AdjacencyMatrix.1', 'AdjacencyMatrix.2', 'AdjacencyMatrix.3',
       'AdjacencyMatrix.4', 'AdjacencyMatrix.5', 'AdjacencyMatrix.6',
       'AdjacencyMatrix.7',
       ...
       'WalkCount.19', 'WalkCount.20', 'Weight', 'Weight.1', 'WienerIndex',
       'WienerIndex.1', 'ZagrebIndex', 'ZagrebIndex.1', 'ZagrebIndex.2',
       'ZagrebIndex.3'],
      dtype='object', length=1216)
0.1291
0.2722
0.3593
0.7928
0.8920
0.8800
Prediction on new data complete.


/tmp/ipykernel_1796054/3754404327.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = 0


In [232]:
#Ablation study
import os
import joblib
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler 
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

In [233]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [234]:
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Train/All_fingerprints_train_Caco2.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Test/All_fingerprints_test_Caco2.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Train_all_atomic_desc_Caco2.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Test_all_atomic_desc_Caco2.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
def scale_features(df_train, df_test):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    return df_train_scaled, df_test_scaled

target_column = 'Permeability'
df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test)
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test)
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test)
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test)

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')


/tmp/ipykernel_1796054/2426944403.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(1007, 264)
(252, 264)
(1007, 868)
(252, 868)
(1007, 759)
(252, 759)
(1007, 12)
(252, 12)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(1007, 264)
(252, 264)
(1007, 868)
(252, 868)
(1007, 759)
(252, 759)
(1007, 12)
(252, 12)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
       ID                                             SMILES  Permeability  \
872    33  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...        -5.480   

In [235]:
models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101, eval_metric='rmse'),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),

]
models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101, eval_metric='rmse'),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101, max_iter=500)
]

dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]

In [236]:
ablation_results = {}

for ablation_idx in range(len(dataframes)):
    print(f"========== Ablation: Excluding feature at index {ablation_idx} ==========")
    feature_names = ['Descriptor', 'Fingerprints', 'Embeddings', 'Atomic']
    print(f"========== Ablation: Excluding feature :-- {feature_names[ablation_idx]} ==========")

    ablated_dataframes = [pair for i, pair in enumerate(dataframes) if i != ablation_idx]

    meta_features_train = []
    meta_features_test = []

    # Stage 1
    for df_train, df_test in tqdm(ablated_dataframes, desc="Processing ablated dataframes"):
        X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
        y_weak = df_train[target_column]
        X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
        y_eval = df_test[target_column]

        kf = KFold(n_splits=5, shuffle=True, random_state=101)

        fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
        fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

        for i, model in tqdm(enumerate(models_weak), desc="Training weak models", total=len(models_weak)):
            fold_predictions = np.zeros(X_weak.shape[0])
            test_predictions_folds = []

            for train_index, val_index in kf.split(X_weak):
                X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
                y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]

                model.fit(X_train, y_train)

                fold_predictions[val_index] = np.clip(model.predict(X_val), -10, -3.4)
                test_predictions_fold = np.clip(model.predict(X_eval), -10, -3.4)
                test_predictions_folds.append(test_predictions_fold)

            fold_meta_features_train[:, i] = fold_predictions
            fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)
            print(f'Model training done {i}: {model.__class__.__name__}')

        meta_features_train.append(fold_meta_features_train)
        meta_features_test.append(fold_meta_features_test)
        print('Dataframe training completed')

    # Stack all meta-features
    meta_features_train = np.hstack(meta_features_train)
    meta_features_test = np.hstack(meta_features_test)

    print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
    print('Stage 1 completed (Weak Learners)')
    print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

    # Stage 2
    results = {}
    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    for model in models_meta:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        for train_index, val_index in kf.split(meta_features_train):
            X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
            y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]

            model.fit(X_fold_train, y_fold_train)
            y_pred_fold = np.clip(model.predict(X_fold_val), -10, -3.4)

            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_fold_val)

            test_predictions_fold = model.predict(meta_features_test)
            test_predictions_fold = np.clip(test_predictions_fold, -10, -3.4)
            test_predictions_folds.append(test_predictions_fold)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        mse_test = mean_squared_error(y_eval, predictions_test_mean)
        mae_test = mean_absolute_error(y_eval, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_eval, predictions_test_mean)
        pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
        spearman_test, _ = spearmanr(y_eval, predictions_test_mean)

        results[model_name] = {
            'Train MSE (5 fold CV)': mse_train,
            'Train MAE (5 fold CV)': mae_train,
            'Train RMSE (5 fold CV)': rmse_train,
            'Train R2 (5 fold CV)': r2_train,
            'Train PCC (5 fold CV)': pearson_train,
            'Train SCC (5 fold CV)': spearman_train,
            'Test MSE': mse_test,
            'Test MAE': mae_test,
            'Test RMSE': rmse_test,
            'Test R2': r2_test,
            'Test PCC': pearson_test,
            'Test SCC': spearman_test,
        }

    ablation_results[f"Ablation_{feature_names[ablation_idx]}"] = pd.DataFrame(results).T

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Ablation Study Completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# To view the results
ablation_results_df = {key: value for key, value in ablation_results.items()}


========== Ablation: Excluding feature at index 0 ==========
========== Ablation: Excluding feature :-- Descriptor ==========


Training weak models:   0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2399
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 649
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,


Training weak models:  10%|█         | 1/10 [00:05<00:49,  5.45s/it]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Model training done 0: LGBMRegressor


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000817 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7295
[LightGBM] [Info] Number of data points in the train set: 806, number of used features: 30
[LightGBM] [Info] Start training from score -6.173113
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Ablation Study Completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


In [237]:
ablation_results

{'Ablation_Descriptor':                            Train MSE (5 fold CV)  Train MAE (5 fold CV)  \
 LGBMRegressor                           0.139787               0.282037   
 DecisionTreeRegressor                   0.273349               0.402000   
 RandomForestRegressor                   0.133021               0.274761   
 GradientBoostingRegressor               0.135888               0.280679   
 AdaBoostRegressor                       0.144500               0.296127   
 XGBRegressor                            0.154141               0.299476   
 ExtraTreesRegressor                     0.131427               0.272303   
 LinearRegression                        0.142635               0.290935   
 KNeighborsRegressor                     0.148939               0.293499   
 SVR                                     0.133759               0.278530   
 MLPRegressor                            0.173988               0.323234   
 
                            Train RMSE (5 fold CV)  Train R2 (5

In [238]:
import os
import pickle

ablation_result_dir = '/home/users/akshay/PCPpred/Caco2/results/Ablation/'
os.makedirs(ablation_result_dir, exist_ok=True)

pickle_path = os.path.join(ablation_result_dir, 'ablation_results.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(ablation_results, f)


with open(pickle_path, 'rb') as f:
    ablation_results = pickle.load(f)


ablation_results

{'Ablation_Descriptor':                            Train MSE (5 fold CV)  Train MAE (5 fold CV)  \
 LGBMRegressor                           0.139787               0.282037   
 DecisionTreeRegressor                   0.273349               0.402000   
 RandomForestRegressor                   0.133021               0.274761   
 GradientBoostingRegressor               0.135888               0.280679   
 AdaBoostRegressor                       0.144500               0.296127   
 XGBRegressor                            0.154141               0.299476   
 ExtraTreesRegressor                     0.131427               0.272303   
 LinearRegression                        0.142635               0.290935   
 KNeighborsRegressor                     0.148939               0.293499   
 SVR                                     0.133759               0.278530   
 MLPRegressor                            0.173988               0.323234   
 
                            Train RMSE (5 fold CV)  Train R2 (5

In [239]:
ablation_result_dir = '/home/users/akshay/PCPpred/Caco2/results/Ablation'
os.makedirs(ablation_result_dir, exist_ok=True)

for ablation_label, df in ablation_results.items():
    print(f"Results for {ablation_label}: \n")
    safe_label = ablation_label.replace(" ", "_").replace("/", "_")
    file_path = os.path.join(ablation_result_dir, f"{safe_label}.csv")
    df.to_csv(file_path)

Results for Ablation_Descriptor: 

Results for Ablation_Fingerprints: 

Results for Ablation_Embeddings: 

Results for Ablation_Atomic: 



In [240]:
from IPython.display import display
for ablation_label, df in ablation_results.items():
    print(f"Results for {ablation_label}: \n")
    display(df)

Results for Ablation_Descriptor: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.139787,0.282037,0.373881,0.781883,0.884445,0.873364,0.151706,0.296114,0.389494,0.756534,0.870355,0.846365
DecisionTreeRegressor,0.273349,0.402000,0.522828,0.573480,0.783268,0.775736,0.159394,0.306940,0.399242,0.744196,0.863375,0.837910
RandomForestRegressor,0.133021,0.274761,0.364720,0.792441,0.890196,0.880727,0.141390,0.285255,0.376019,0.773089,0.879632,0.850764
GradientBoostingRegressor,0.135888,0.280679,0.368631,0.787966,0.887826,0.876628,0.149336,0.292368,0.386441,0.760337,0.872571,0.847942
AdaBoostRegressor,0.144500,0.296127,0.380132,0.774529,0.880205,0.873915,0.161811,0.311955,0.402258,0.740316,0.860515,0.847309
XGBRegressor,0.154141,0.299476,0.392608,0.759486,0.871698,0.863445,0.147365,0.295082,0.383882,0.763501,0.873850,0.841615
ExtraTreesRegressor,0.131427,0.272303,0.362528,0.794929,0.891616,0.883716,0.145831,0.287005,0.381878,0.765963,0.875328,0.843742
LinearRegression,0.142635,0.290935,0.377670,0.777440,0.881855,0.877694,0.152248,0.296053,0.390189,0.755665,0.869382,0.841228
KNeighborsRegressor,0.148939,0.293499,0.385927,0.767602,0.876523,0.862124,0.155050,0.299612,0.393764,0.751167,0.866786,0.839282
SVR,0.133759,0.278530,0.365731,0.791289,0.889586,0.878495,0.156459,0.294118,0.395550,0.748905,0.866890,0.836634


Results for Ablation_Fingerprints: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.141564,0.281569,0.376250,0.779111,0.882962,0.874687,0.155754,0.297741,0.394656,0.750038,0.866693,0.842489
DecisionTreeRegressor,0.271756,0.391882,0.521303,0.575965,0.785439,0.779461,0.153920,0.296005,0.392327,0.752981,0.868232,0.842570
RandomForestRegressor,0.135262,0.278107,0.367780,0.788944,0.888240,0.879825,0.145350,0.284545,0.381248,0.766735,0.875636,0.846240
GradientBoostingRegressor,0.138635,0.283505,0.372338,0.783680,0.885349,0.876590,0.149872,0.291511,0.387133,0.759477,0.871560,0.841337
AdaBoostRegressor,0.148034,0.296178,0.384752,0.769015,0.876977,0.866742,0.163703,0.306937,0.404602,0.737281,0.859079,0.835785
XGBRegressor,0.154649,0.298804,0.393255,0.758693,0.872116,0.864811,0.148440,0.293542,0.385279,0.761775,0.873225,0.840391
ExtraTreesRegressor,0.129795,0.272502,0.360270,0.797475,0.893022,0.884719,0.147544,0.287039,0.384114,0.763214,0.873643,0.839236
LinearRegression,0.138927,0.287939,0.372730,0.783225,0.885118,0.877757,0.151823,0.293240,0.389645,0.756346,0.869748,0.841075
KNeighborsRegressor,0.142484,0.288525,0.377471,0.777674,0.882579,0.871396,0.166114,0.305432,0.407571,0.733411,0.857365,0.826280
SVR,0.136864,0.279978,0.369951,0.786444,0.886876,0.875307,0.147660,0.280860,0.384266,0.763027,0.874616,0.846440


Results for Ablation_Embeddings: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.169315,0.304658,0.411479,0.735810,0.858716,0.844781,0.139066,0.281027,0.372915,0.776820,0.881548,0.867309
DecisionTreeRegressor,0.313774,0.417364,0.560155,0.510404,0.763434,0.753852,0.154548,0.288984,0.393126,0.751973,0.868180,0.854666
RandomForestRegressor,0.158509,0.293612,0.398132,0.752671,0.867649,0.855100,0.137063,0.280380,0.370220,0.780034,0.883280,0.864745
GradientBoostingRegressor,0.163287,0.297670,0.404088,0.745215,0.863817,0.850578,0.139441,0.280856,0.373419,0.776217,0.881456,0.858083
AdaBoostRegressor,0.175461,0.317814,0.418881,0.726219,0.853928,0.839729,0.147207,0.298262,0.383676,0.763754,0.876225,0.856506
XGBRegressor,0.182983,0.316342,0.427765,0.714482,0.847490,0.834242,0.148765,0.288820,0.385700,0.761255,0.873406,0.855161
ExtraTreesRegressor,0.153509,0.290732,0.391802,0.760473,0.872055,0.857443,0.135408,0.280568,0.367979,0.782689,0.884758,0.867865
LinearRegression,0.165942,0.308003,0.407360,0.741072,0.861039,0.856363,0.131171,0.278050,0.362176,0.789489,0.888597,0.872522
KNeighborsRegressor,0.168843,0.307897,0.410905,0.736546,0.859048,0.845155,0.148639,0.298462,0.385538,0.761455,0.873090,0.857593
SVR,0.165358,0.303216,0.406643,0.741983,0.861809,0.850662,0.143308,0.282652,0.378560,0.770012,0.879307,0.860057


Results for Ablation_Atomic: 



,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.139154,0.280758,0.373033,0.782872,0.884968,0.878914,0.148696,0.291675,0.385611,0.761365,0.873037,0.848474
DecisionTreeRegressor,0.255063,0.381988,0.505038,0.602013,0.799979,0.791446,0.147022,0.295113,0.383434,0.764052,0.874326,0.848316
RandomForestRegressor,0.129963,0.272409,0.360504,0.797212,0.892877,0.884524,0.136739,0.278761,0.369783,0.780554,0.883653,0.855402
GradientBoostingRegressor,0.139274,0.282019,0.373195,0.782683,0.884899,0.876031,0.145464,0.287624,0.381397,0.766552,0.875619,0.850887
AdaBoostRegressor,0.145596,0.294454,0.381570,0.772819,0.879212,0.870184,0.159348,0.303055,0.399184,0.744270,0.862749,0.844414
XGBRegressor,0.142406,0.286842,0.377367,0.777797,0.882136,0.872726,0.144831,0.291363,0.380567,0.767567,0.876301,0.844970
ExtraTreesRegressor,0.127802,0.270198,0.357494,0.800585,0.894828,0.886538,0.139967,0.279921,0.374121,0.775374,0.880644,0.852898
LinearRegression,0.138872,0.286391,0.372655,0.783311,0.885155,0.878365,0.147385,0.287402,0.383907,0.763469,0.873814,0.845384
KNeighborsRegressor,0.142015,0.284542,0.376849,0.778407,0.882631,0.869313,0.157772,0.303954,0.397205,0.746799,0.864631,0.832876
SVR,0.135945,0.279880,0.368707,0.787879,0.887662,0.875478,0.146184,0.284756,0.382340,0.765396,0.875942,0.853355
